# Cohort-04 Photometry: Preprocessing, Quality Control and Outcome-Aligned Responses

This notebook takes the fibre-photometry recordings of cohort 04 (nLightG2 norepinephrine sensor in
prelimbic cortex, freely moving mice, 3x3 bandit task) from raw pyPhotometry files to
outcome-aligned responses, and checks the quality of every step.

**What it does, in order**

1. Load the recordings and the trial outcome times; count trials.
2. Check the synchronisation of photometry, video and behaviour onto **pyControl time (seconds)**.
3. Pipeline A: the order recommended in the "Lights, fiber, action!" primer (Simpson et al. 2024).
4. Pipeline B: the procedure of the nLightG2 paper (Rohner et al. 2026) and of the authors' MATLAB code.
5. Compare the two in writing.
6. Quality control for every step, plus signal-to-noise (SNR) measures and a per-mouse ranking.
7. Responses to rewarded and unrewarded outcomes: per mouse, for the signal and the 405-nm control channel, their sensitivity
   to the pre-outcome window and to smoothing, and a comparison of two groups of mice.
8. Evoked-response quality control and the per-mouse ranking.
9. Summary and open questions.

**Where the code lives.** Loading, synchronisation, outcome times and the cutting of event windows are
imported from `src/` (they are tested there). The photometry preprocessing and quality-control functions
are defined in this notebook, on purpose: the preprocessing is preliminary and will change, and
it will move into `src/` once it settles. Each function is still small, pure (arrays in, arrays out) and
kept apart from loading and plotting.

**Conventions.** Every time is in seconds on **pyControl's clock**. Signals are volts until
they are normalised. All parameters sit in the configuration cell below; nothing else needs editing.
Figures are also written to `results/figures/photometry/cohort-04/` and tables to
`results/tables/photometry/cohort-04/` (git-ignored).

In [ ]:
import time
from collections.abc import Callable
from dataclasses import dataclass
from functools import partial
from itertools import combinations
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import ndimage
from scipy import signal as sps
from scipy.integrate import trapezoid
from scipy.optimize import curve_fit, least_squares
from scipy.stats import linregress

from src.behavior_analysis.outcome_events import load_outcome_events, outcome_settings
from src.behavior_import.import_data import read_tsv_file
from src.behavior_visualization.plot_rsync_alignment import plot_rsync_alignment
from src.photometry_analysis.event_windows import (
    extract_event_windows,
    relative_time_grid,
    sync_coverage,
)
from src.photometry_analysis.pulse_groups import (
    categories_with_min_trials,
    pulse_categories,
    pulse_response_correlation,
    split_by_pulses,
)
from src.photometry_import.pyphotometry import import_ppd
from src.photometry_import.recordings import (
    MAX_MEDIAN_DISCREPANCY_MS,
    MIN_FRACTION_MATCHED,
    load_photometry_recordings,
    read_tsv_sync_pulses,
)
from src.photometry_import.sessions import find_photometry_sessions
from src.settings import REPO_ROOT, get_settings
from src.sync.pycontrol_time import photometry_sample_times, sync_quality, video_frame_times
from src.sync.rsync import RsyncError, min_pulses_required

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams.update(
    {
        "figure.dpi": 90,
        "savefig.dpi": 130,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

## 1. Setup and inputs

### Configuration

Everything you may want to change is here. The values in `PREPROCESSING` are part of the cache key (see
below), so changing them recomputes the affected recordings and nothing else.

- **`PHASES`**: which trial phases enter the analysis. `task` is the bandit task proper; `port_finding` is the
  phase in which the mouse finds the ports. Both are included by default; counts per phase are reported below.
- **`EVENT_WINDOW`**: −5 to +10 s around the outcome, as in the earlier notebooks.
- **Pipeline B**: `B_PREPROCESSING` (bleaching model, fit window, optional post-outcome mask of the motion fit) and
  `B_EVENT_LEVEL` (what the trial z-score is taken from, whether ΔF/F has a baseline subtracted) default to what the nLightG2
  paper and the authors' code do; section 4 says where and why a default differs.
- **`PRE_WINDOWS_S`, `SMOOTHING_POINTS`**: the pre-outcome windows and moving-mean lengths of the sensitivity analysis (7c),
  with seconds at 130 Hz printed below. `DEFAULT_PRE_WINDOW`, `DEFAULT_SMOOTHING` (pipeline B) and `DEFAULT_SMOOTHING_A`
  drive the main figures; the analysis functions receive them as arguments.
- **`GROUP_PREFIXES`, `SINGLE_MOUSE`**: the two groups of mice (by subject prefix) and the single mouse of section 7d.
- **QC thresholds** only *flag* a recording; nothing is excluded. They are starting points, not validated limits.

In [ ]:
# --- Data ---
COHORT = "cohort-04"
EXPERIMENT = "3x3_field_magnitude_bandit"
PHASES = ("task", "port_finding")  # trial phases to include

# --- Event windows ---
EVENT_WINDOW = (-5.0, 10.0)  # s relative to the outcome

# --- Preprocessing (shared by pipelines A and B up to dF/F). Part of the cache key. ---
PREPROCESSING = {
    "median_filter_s": None,  # window of the optional spike filter (s); None = off
    "lowpass_hz": 10.0,  # zero-phase Butterworth (order 2) cutoff
    "lowpass_order": 2,
    "fit_decimation": 13,  # the bleaching fit uses every 13th sample (10 Hz); see section 3
    "fit_initial": (0.5, 0.25, 0.25, 3600.0, 0.1),  # fractions of max for the first three; s; ratio
    "fit_lower": (
        0.0,
        0.0,
        0.0,
        600.0,
        0.0,
    ),  # const, amp_fast, amp_slow (x max); tau_slow (s); tau ratio
    "fit_upper": (1.0, 1.0, 1.0, 36000.0, 1.0),
    "code_version": 2,  # bump when a helper function used inside the cached step changes
}
HIGHPASS_ALTERNATIVE_HZ = 0.001  # comparison only, never used downstream

# --- Pipeline B, whole-trace steps (Rohner et al. 2026 and the authors' MATLAB code; see section 4) ---
# Every switch defaults to the paper/code choice, except where that choice cannot apply to this task
# (motion_mask_s, explained in section 4). All values here are part of the pipeline-B cache key.
B_PREPROCESSING = {
    # "exp2": a*exp(b*t) + c*exp(d*t), no constant, no bounds (MATLAB fit(..., 'exp2'));
    # "bounded": pipeline A's bounded double exponential with a constant.
    "bleaching_model": "exp2",
    # Bleaching and motion fits start at MATLAB's 1-based sample 100 (l.1142), i.e. Python index 99: the first 99
    # samples are left out ...
    "fit_skip_samples": 99,
    "fit_skip_end_s": 30.0,  # ... and stop this many seconds before the end of the recording
    # Seconds after each outcome left out of the motion regression. The code leaves out 20 s after each
    # TTL; our outcomes come about every 10 s, so that would remove most of the session. 0 = off.
    "motion_mask_s": 0.0,
    # Settings of this notebook's exp2 optimiser (MATLAB's own start values are not documented). They change the
    # fit, so they are part of the cache key with the rest of this dict.
    "exp2_tau_grid_s": (5.0, 1e5),  # time constants (s) spanned by the grid of starting rates
    "exp2_grid_size": 21,  # rates in that grid (plus the rate 0 = a constant)
    # This notebook's own rule, not MATLAB's: amplitudes more than this many times the largest fitted value mean two
    # nearly equal rates cancel each other, and the fit is rejected.
    "exp2_max_amplitude_ratio": 1e3,
    "code_version": 2,  # bump when a helper used inside the cached pipeline-B step changes
}
# --- Pipeline B, event level ---
B_EVENT_LEVEL = {
    # "corrected": trial z-score of the motion-corrected signal (V), baseline from the same signal (code);
    # "dff": trial z-score of the ΔF/F windows.
    "z_from": "corrected",
    # False (code): per-trial ΔF/F with no baseline subtraction.
    # True (paper, tail-lift trials): subtract the pre-window mean from each trial's ΔF/F.
    "subtract_dff_baseline": False,
}

# --- Pre-outcome windows and smoothing, for the sensitivity analysis (section 7c) and the main figures ---
PRE_WINDOWS_S = [
    (-2.0, 0.0),
    (-5.0, 0.0),
    (-10.0, 0.0),
    (-15.0, 0.0),
]  # s; 15 s = the paper's z baseline
DEFAULT_PRE_WINDOW = (-5.0, 0.0)  # baseline of pipeline B in the main figures
SMOOTHING_POINTS = [1, 10, 40, 80]  # centred moving mean, in samples; 1 = none; 80 = the paper's
NOMINAL_FS_HZ = 130.0  # recording rate; only used to print samples as seconds
DEFAULT_SMOOTHING = (
    80  # pipeline B in the main figures, and both pipelines in the group figures (7d)
)
DEFAULT_SMOOTHING_A = 1  # pipeline A in sections 7 and 8: none beyond the 10 Hz low-pass (primer)
METRIC_WINDOW = (0.0, 10.0)  # post-event window for peak and area under the curve

# --- Responses by number of reward pulses (section 7e) ---
# Phases whose trials are split by pulse count. The port-finding phase gives 2 pulses at every arm, so its trials
# carry no information about reward size; add "port_finding" to include them (it must also be in PHASES above, so
# that section 7e stays a subset of the trials of section 7).
PULSE_PHASES = ("task",)
# A mouse's pulse category is drawn, enters group means and the correlation only with at least this many usable trials.
MIN_TRIALS_PER_PULSE = 5

# --- Groups of mice for section 7d: subject-name prefixes ---
GROUP_PREFIXES = {
    "Group 1": ("RT_176_", "RT_64_"),
    "Group 2": ("RT_177_", "RT_65_"),
}
SINGLE_MOUSE = "RT_176_L"  # compared with SINGLE_MOUSE_VS in the second group figure
SINGLE_MOUSE_VS = "Group 2"

# --- Quality control ---
QC_NOISE_BAND_HZ = (
    15.0,
    50.0,
)  # noise floor band, also the high-pass cutoff for the noise estimate
QC_SIGNAL_BAND_HZ = (0.1, 3.0)  # band of slow fluctuations
QC_WELCH_SEGMENT_S = 30.0
QC_FLATLINE_S = 0.5  # identical samples for longer than this = dropout
QC_JUMP_WINDOW_S = 0.5
QC_JUMP_MAD_THRESHOLD = 10.0  # robust SD (1.4826 x MAD) units of the 0.5-s mean-difference step
QC_BASELINE_SPIKE_SD = (
    6.0  # a baseline (LED-off) sample this many robust SD above its median is a spike
)
QC_TRANSIENT_PROMINENCE_SD = 5.0  # k, in robust SD of the low-passed trace; 3 lets noise ripples in
EVOKED_POST = (0.0, 2.0)
EVOKED_PRE = (-2.0, 0.0)
EVOKED_MIN_TRIALS = 5
QC_THRESHOLDS = {
    "min_duration_min": 5.0,
    "max_baseline_v": 0.1,  # LED-off level; larger suggests ambient light leaking in
    "max_clipping_fraction": 0.001,
    "max_flatline_runs": 0,
    "min_sync_fraction": 0.95,
    "max_sync_discrepancy_samples": 2.0,
    "max_outside_sync_fraction": 0.05,
    "max_hf_noise_pct": 1.0,  # high-frequency noise SD as % of the median signal, before the low-pass
    "max_bleaching_drop_pct": 50.0,
    "max_params_at_bound": 2,
    "min_movement_slope": 0.0,
    "max_jumps_per_min": 1.0,
    "min_snr_db": 0.0,
    "min_specificity_db": 1.0,  # uncorrected sensor SNR minus control SNR
    "min_evoked_trials": EVOKED_MIN_TRIALS,
}
MIN_RANK_DISAGREEMENT = 3  # report mice whose two SNR ranks differ by at least this much

# --- Outputs ---
FIGURE_DIR = REPO_ROOT / "results" / "figures" / "photometry" / COHORT
TABLE_DIR = REPO_ROOT / "results" / "tables" / "photometry" / COHORT


def widest_window(*windows: tuple[float, float]) -> tuple[float, float]:
    """The smallest window that contains every given ``(start, stop)`` window (s)."""
    return min(w[0] for w in windows), max(w[1] for w in windows)


if not set(PULSE_PHASES) <= set(PHASES) or MIN_TRIALS_PER_PULSE < 1:
    raise ValueError(
        f"PULSE_PHASES {PULSE_PHASES} must be a subset of PHASES {PHASES}, and MIN_TRIALS_PER_PULSE "
        "must be at least 1."
    )
if DEFAULT_PRE_WINDOW not in PRE_WINDOWS_S or DEFAULT_SMOOTHING not in SMOOTHING_POINTS:
    raise ValueError("DEFAULT_PRE_WINDOW and DEFAULT_SMOOTHING must be entries of the lists above.")
print(
    "Smoothing (moving-mean length): "
    + "; ".join(f"{p} points = {p / NOMINAL_FS_HZ:.3f} s" for p in SMOOTHING_POINTS)
    + f" at {NOMINAL_FS_HZ:g} Hz."
)
print("Pre-outcome windows: " + "; ".join(f"{-a:g} s" for a, _ in PRE_WINDOWS_S) + ".")

# Expanded once: the window that must be extractable so the default baseline and every metric window fit.
EXTRACTION_WINDOW = widest_window(
    EVENT_WINDOW, DEFAULT_PRE_WINDOW, EVOKED_PRE, EVOKED_POST, METRIC_WINDOW
)

### Saving figures and tables

These two small functions are the only places that write files, so analysis and plotting code never
touches the disk.

In [ ]:
def save_figure(fig: plt.Figure, name: str) -> Path:
    """Write *fig* to the cohort's figure folder as a PNG and return the path."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    return path


def save_table(table: pd.DataFrame, name: str) -> Path:
    """Write *table* to the cohort's table folder as a CSV and return the path."""
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    path = TABLE_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    return path


def show_table(table: pd.DataFrame, name: str) -> None:
    """Print *table*, save it as CSV, and say where."""
    display(table)
    print(f"saved: {save_table(table, name).relative_to(REPO_ROOT)}")

### Find and load the recordings

`find_photometry_sessions` lists the numbered sessions that hold a `.ppd` file (`ses-test` folders are
left out). `load_photometry_recordings` pairs each `.ppd` with the pyControl `.tsv` that shares its sync
pulses and gives every photometry sample its pyControl time. A `.ppd` that cannot be paired (for example
a false start with almost no sync pulses) is returned as *skipped* with a reason, and is listed here.

In [ ]:
cohort_dir = get_settings().data_root / EXPERIMENT / "rawdata" / COHORT
sessions = find_photometry_sessions(cohort_dir)
loaded = [load_photometry_recordings(session) for session in sessions]
recordings = [rec for recs, _ in loaded for rec in recs]
skipped = [skip for _, skips in loaded for skip in skips]
MICE = list(dict.fromkeys(rec.subject for rec in recordings))


def recording_id(recording) -> str:
    """Unique label of a recording: its .ppd file name (a session can hold two recordings)."""
    return recording.ppd_path.stem


def label_outcomes(events: pd.DataFrame) -> pd.DataFrame:
    """*events* with a ``condition`` column: ``rewarded`` or ``unrewarded``."""
    events = events.copy()
    events["condition"] = np.where(events["rewarded"], "rewarded", "unrewarded")
    return events


def load_recording_behaviour(recordings: dict) -> tuple[dict, dict, list[dict]]:
    """Outcome events and logged outcome durations of each recording, with per-recording failures.

    One recording whose ``.tsv`` cannot be parsed (a ``ValueError``, which includes the parser's
    own ``OutcomeParseError``) must not abort the others, so it is reported instead of raised.

    Returns:
        tuple[dict, dict, list[dict]]: Events by recording id, logged settings by recording id,
        and one ``{"recording", "tsv", "reason"}`` row per recording that failed.
    """
    events, settings, failures = {}, {}, []
    for rid, rec in recordings.items():
        try:
            events[rid] = label_outcomes(load_outcome_events(rec.tsv_path))
            settings[rid] = outcome_settings(read_tsv_file(rec.tsv_path))
        except ValueError as error:
            failures.append({"recording": rid, "tsv": rec.tsv_path.name, "reason": str(error)})
    return events, settings, failures


RECORDINGS = {recording_id(rec): rec for rec in recordings}
assert len(RECORDINGS) == len(recordings), "recording ids must be unique"
ALL_EVENTS, EXPECTED, behaviour_failures = load_recording_behaviour(RECORDINGS)
if behaviour_failures:
    print(
        f"\nBehaviour could not be loaded for {len(behaviour_failures)} recording(s); they are left out:"
    )
    display(pd.DataFrame(behaviour_failures))
RECORDINGS = {rid: rec for rid, rec in RECORDINGS.items() if rid in ALL_EVENTS}
recordings = list(RECORDINGS.values())
MICE = list(dict.fromkeys(rec.subject for rec in recordings))
EVENTS = {
    rid: ev[ev["phase"].isin(PHASES)].reset_index(drop=True) for rid, ev in ALL_EVENTS.items()
}

print(
    f"{len(sessions)} sessions with a .ppd; {len(recordings)} recordings aligned; {len(MICE)} mice."
)
print(f"\nSkipped ({len(skipped)}):")
for skip in skipped:
    print(f"  {skip.ppd_path.relative_to(cohort_dir)}: {skip.reason}")

### Channel wiring (to be confirmed)

The recordings are pyPhotometry `2EX_1EM_pulsed`: two excitation LEDs, one emission detector, 130 Hz.
**Assumed wiring: `analog_1` is the nLightG2 signal (470 nm excitation) and `analog_2` is the isosbestic control
(405 nm).** This is taken from the earlier notebooks and has not been confirmed against the physical LED
connections. Everything below that calls a channel "signal" or "control" depends on it; the
movement-correction and specificity QC would be meaningless if the channels were swapped. The LED currents
(printed below) differ between the two channels and between sessions, which is worth knowing when
comparing absolute voltages.

### Recordings and trial counts

In [ ]:
def recordings_table(recordings) -> pd.DataFrame:
    """One row per aligned recording: identity, files, duration and LED currents."""
    return pd.DataFrame(
        [
            {
                "recording": recording_id(rec),
                "mouse": rec.subject,
                "session": rec.session,
                "ppd": rec.ppd_path.name,
                "tsv": rec.tsv_path.name,
                "fs_hz": rec.fs,
                "duration_min": len(rec.signal) / rec.fs / 60,
                "led_current_ma_1": rec.led_current[0],
                "led_current_ma_2": rec.led_current[1],
            }
            for rec in recordings
        ]
    )


COUNT_COLUMNS = [f"{p}_{c}" for p in ("task", "port_finding") for c in ("rewarded", "unrewarded")]


def trial_counts_table(events_by_recording: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Trials per recording by phase and outcome (columns ``<phase>_<condition>``)."""
    rows = []
    for rid, events in events_by_recording.items():
        rec = RECORDINGS[rid]
        counts = events.groupby(["phase", "condition"]).size()
        row = {"recording": rid, "mouse": rec.subject, "session": rec.session}
        for column in COUNT_COLUMNS:
            phase, condition = column.rsplit("_", 1)
            row[column] = int(counts.get((phase, condition), 0))
        rows.append(row)
    table = pd.DataFrame(rows)
    table["total"] = table[COUNT_COLUMNS].sum(axis=1)
    return table


recordings_summary = recordings_table(recordings)
show_table(recordings_summary, "recordings")

trial_counts = trial_counts_table(ALL_EVENTS)
show_table(trial_counts, "trial_counts")
phase_totals = trial_counts[COUNT_COLUMNS].sum()
print("Trials over all recordings, by phase and outcome (all phases counted here):")
print(phase_totals.to_string())
print(f"\nIncluded by PHASES={PHASES}: {sum(len(e) for e in EVENTS.values())} trials.")

## 2. Synchronisation checks

Photometry, video and behaviour each run on their own clock. pyControl sends random-interval `rsync` pulses to
the other systems; `RsyncAligner` finds which pulse is which by matching the *pattern* of intervals, and
the time between pulses is interpolated. Photometry samples before the first and after the last matched pulse get
no time (NaN), and no window is allowed to reach into them.

**What to look for.** `fraction_matched` close to 1; the largest interval disagreement between the two clocks
within about one photometry sample (7.7 ms at 130 Hz); a few percent of samples outside the sync coverage
(the recording starts before, and ends after, the first and last pulse).

In [ ]:
def sync_table(recordings) -> pd.DataFrame:
    """One row per recording: how well its sync pulses matched pyControl's."""
    rows = []
    for rec in recordings:
        quality = rec.sync_quality
        sample_ms = 1000 / rec.fs
        rows.append(
            {
                "recording": recording_id(rec),
                "mouse": rec.subject,
                "session": rec.session,
                "n_pulses_ppd": quality["n_pulses_other"],
                "n_pulses_pycontrol": quality["n_pulses_pycontrol"],
                "n_matched": quality["n_matched"],
                "fraction_matched": quality["fraction_matched"],
                "max_discrepancy_ms": quality["max_abs_interval_discrepancy_ms"],
                "max_discrepancy_samples": quality["max_abs_interval_discrepancy_ms"] / sample_ms,
                "median_discrepancy_ms": quality["median_abs_interval_discrepancy_ms"],
                "first_matched_s": quality["first_matched_time_s"],
                "last_matched_s": quality["last_matched_time_s"],
                "n_samples_outside_sync": rec.n_samples_outside_sync,
                "fraction_outside_sync": rec.n_samples_outside_sync / len(rec.signal),
            }
        )
    return pd.DataFrame(rows)


sync_summary = sync_table(recordings)
show_table(sync_summary, "sync_quality")

### Alignment diagnostic for one example recording

The example is the recording with the most trials (used again for the step-by-step plots below). The left two
panels should show two cleanly separated groups of chunk errors and interval disagreements of a few ms at most;
the wide panel should be a straight line.

In [ ]:
EXAMPLE = max(EVENTS, key=lambda rid: len(EVENTS[rid]))
print(f"Example recording: {EXAMPLE} ({len(EVENTS[EXAMPLE])} trials)")


def recording_aligner(recording):
    """The rsync aligner of a recording (reloads the .ppd pulses and the .tsv rsync times)."""
    data = import_ppd(recording.ppd_path, low_pass=None, high_pass=None)
    pulses = read_tsv_sync_pulses(recording.tsv_path)
    return photometry_sample_times(data, pulses)[1]


example_aligner = recording_aligner(RECORDINGS[EXAMPLE])
axes = plot_rsync_alignment(example_aligner)
axes[0].figure.suptitle(f"Photometry-To-pyControl Sync: {EXAMPLE}", fontsize=11)
save_figure(axes[0].figure, "sync_alignment_example")
plt.show()

### Video on the same clock

The camera writes a `*_GPIO_data.csv` with one row per frame and the rsync pulses on `GPIO1`. The same aligner
puts video frame times on pyControl time, which is what makes frame-accurate behaviour-photometry comparisons
possible. A session can hold several `.tsv` files, so each video is matched to the `.tsv` whose pulse train it shares.

In [ ]:
def match_video_to_tsv(gpio_df: pd.DataFrame, tsv_pulses: dict[Path, np.ndarray]):
    """The one candidate ``.tsv`` whose rsync train aligns with the video's, or None.

    Returns:
        tuple | None: ``(tsv_path, frame_times_s, aligner)``. The acceptance rule is the one used to
        pair a ``.ppd`` with a ``.tsv``: most pulses paired, and a small median interval error.

    Raises:
        ValueError: If more than one ``.tsv`` matches (the pairing would be ambiguous), as in
            ``match_ppd_to_tsv``.
    """
    matches = []
    for path, pulses in tsv_pulses.items():
        if len(pulses) < min_pulses_required():
            continue
        try:
            frame_times, aligner = video_frame_times(gpio_df, pulses)
        except RsyncError:  # not the right .tsv for this video
            continue
        quality = sync_quality(aligner)
        if (
            quality["fraction_matched"] >= MIN_FRACTION_MATCHED
            and quality["median_abs_interval_discrepancy_ms"] <= MAX_MEDIAN_DISCREPANCY_MS
        ):
            matches.append((path, frame_times, aligner))
    if len(matches) > 1:
        raise ValueError(f"the video matches more than one .tsv: {[m[0].name for m in matches]}.")
    return matches[0] if matches else None


def video_sync_row(session, gpio_path: Path, match) -> dict:
    """One row of the video sync table; *match* is the result of :func:`match_video_to_tsv`."""
    row = {"mouse": session.subject, "session": session.session, "gpio": gpio_path.name}
    if match is None:
        return {**row, "tsv": "no matching .tsv"}
    tsv_path, frame_times, aligner = match
    quality = sync_quality(aligner)
    covered = np.isfinite(frame_times)
    return {
        **row,
        "tsv": tsv_path.name,
        "n_frames": len(frame_times),
        "frame_rate_hz": 1 / np.median(np.diff(frame_times[covered])),
        "fraction_matched": quality["fraction_matched"],
        "max_discrepancy_ms": quality["max_abs_interval_discrepancy_ms"],
        "median_discrepancy_ms": quality["median_abs_interval_discrepancy_ms"],
        "fraction_outside_sync": 1 - covered.mean(),
    }


video_rows, video_aligners = [], {}
for session in sessions:
    if not session.gpio_paths:
        continue
    tsv_pulses = {path: read_tsv_sync_pulses(path) for path in session.tsv_paths}
    for gpio_path in session.gpio_paths:
        match = match_video_to_tsv(pd.read_csv(gpio_path), tsv_pulses)
        video_rows.append(video_sync_row(session, gpio_path, match))
        if match is not None:
            video_aligners[(session.subject, session.session, gpio_path.name)] = match[2]
video_sync = pd.DataFrame(video_rows)
show_table(video_sync, "video_sync_quality")
print(f"{len(sessions) - sum(1 for s in sessions if s.gpio_paths)} sessions have no video.")

In [ ]:
if video_aligners:
    video_example = next(iter(video_aligners.items()))
    axes = plot_rsync_alignment(video_example[1])
    axes[0].figure.suptitle(
        "Video-To-pyControl Sync: " + " ".join(video_example[0][:2]), fontsize=11
    )
    save_figure(axes[0].figure, "sync_alignment_video_example")
    plt.show()

### Outcome timing check

Event times come from the pyControl log (`load_outcome_events`):

- **Rewarded trial**: zero is entry to the `choice` state, which is the start of reward pulse 1. The first
  `pulse_off_timer` after it ends the pulse, so `check_time - outcome_time` should be about `pulse_isi`.
- **Unrewarded trial**: zero is entry to `ITI`, which is the onset of the no-reward cue. `no_rew_cue_timer` fires
  at cue *offset*, so `check_time - outcome_time` should be about `no_rew_cue_dur`.

The expected values are read from each session's `run_start` row. **What to look for:** narrow peaks at the
expected values (the dashed lines).

The measured delays are about 2 ms longer than `pulse_isi` (302 vs 300 ms) and about 6 ms shorter than
`no_rew_cue_dur` (494 vs 500 ms). The cause of the 6 ms has not been established here (it needs a look at the
task code); what matters for the photometry is its size. One photometry sample lasts 7.7 ms at 130 Hz, so a 6 ms
offset in the cue onset is below the sampling resolution and cannot move a response by more than one sample.
Whether to correct it anyway is an open question in the summary.

In [ ]:
def outcome_delays(events: pd.DataFrame) -> pd.DataFrame:
    """Delay (ms) from outcome to check time, with the outcome kind."""
    return pd.DataFrame(
        {
            "kind": events["outcome_kind"],
            "delay_ms": 1000 * (events["check_time"] - events["outcome_time"]),
        }
    )


def timing_summary(delays: dict[str, pd.DataFrame], expected: dict[str, dict]) -> pd.DataFrame:
    """Per recording and outcome kind: median delay against the logged duration."""
    rows = []
    for rid, table in delays.items():
        for kind, setting in (
            ("reward_onset", "pulse_isi"),
            ("no_reward_cue_onset", "no_rew_cue_dur"),
        ):
            values = table.loc[table["kind"] == kind, "delay_ms"]
            if values.empty:
                continue
            rows.append(
                {
                    "recording": rid,
                    "kind": kind,
                    "n": len(values),
                    "expected_ms": expected[rid][setting],
                    "median_ms": values.median(),
                    "p5_ms": values.quantile(0.05),
                    "p95_ms": values.quantile(0.95),
                    "median_minus_expected_ms": values.median() - expected[rid][setting],
                }
            )
    return pd.DataFrame(rows)


def plot_delay_histogram(ax, delays_ms: np.ndarray, expected_ms: float, title: str) -> None:
    """Histogram of check-time delays with the logged duration marked."""
    ax.hist(delays_ms, bins=np.arange(expected_ms - 20, expected_ms + 20.5, 1.0), color="0.5")
    ax.axvline(expected_ms, color="k", ls="--", label=f"Logged Duration ({expected_ms:g} ms)")
    ax.axvline(
        np.median(delays_ms), color="tab:red", label=f"Median ({np.median(delays_ms):.1f} ms)"
    )
    ax.set(title=title, xlabel="Check Time Minus Outcome Time (ms)", ylabel="Trials")
    ax.legend()


DELAYS = {rid: outcome_delays(ev) for rid, ev in ALL_EVENTS.items()}
timing = timing_summary(DELAYS, EXPECTED)
show_table(timing, "outcome_timing")

all_delays = pd.concat(DELAYS.values())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
for ax, (kind, setting, title) in zip(
    axes,
    [
        ("reward_onset", "pulse_isi", "Rewarded: Reward Pulse 1"),
        ("no_reward_cue_onset", "no_rew_cue_dur", "Unrewarded: No-Reward Cue"),
    ],
    strict=True,
):
    logged = np.unique([e[setting] for e in EXPECTED.values()])
    assert len(logged) == 1, f"{setting} differs between sessions: {logged}"
    expected_ms = float(logged[0])
    plot_delay_histogram(
        ax, all_delays.loc[all_delays["kind"] == kind, "delay_ms"].to_numpy(), expected_ms, title
    )
save_figure(fig, "outcome_timing")
plt.show()

## 3. Pipeline A: the "Lights, fiber, action!" primer

Simpson, Akam, Patriarchi et al. (2024), *Neuron*, "Lights, fiber, action! A primer on in vivo fiber photometry"
([PMC10939905](https://pmc.ncbi.nlm.nih.gov/articles/PMC10939905)). The primer's order is
**filtering, then bleaching correction, then movement correction, then normalisation**, and the bleaching
correction is done on each channel *before and independently of* the movement correction. Each step below is one
small function:

| Step | What it does | Default | Why |
|---|---|---|---|
| A1 | LED-off baseline subtracted (done by the importer) | on | removes ambient light and electronic offset from each sample |
| A2 | Median filter for electrical spikes | **off** | needed only if there are brief spikes; window about 5x the spike duration |
| A3 | Zero-phase Butterworth low-pass (order 2, `filtfilt`) | 10 Hz | removes high-frequency noise without shifting events in time. The primer gives 2-10 Hz as typical for GCaMP6f/dLight; nLightG2 is not known to be faster than these, so the top of that range is used |
| A4 | Bleaching: double-exponential fit per channel, subtracted | on | the primer's compromise between a single exponential (too rigid) and a high-pass filter (removes real slow signal) |
| A5 | Movement: least-squares fit of the detrended control onto the detrended signal, subtracted | on | what is shared between the channels (movement, light-path changes) is not sensor activity |
| A6 | Normalisation: ΔF/F (%) = 100 x corrected / signal bleaching fit; session z-score of the corrected signal | both | makes sessions and mice comparable |

**Subtract or divide for bleaching?** Subtracting the fit assumes the bleaching comes from a constant
background (autofluorescence of the fibre and tissue) that is not modulated by the indicator; dividing assumes the
bleaching is of the indicator itself, so the signal scales with it. Autofluorescence-dominated bleaching
favours subtraction, indicator-dominated bleaching favours division. Here the fit is **subtracted** (as in the
earlier notebooks) and the fit is then used as the denominator of ΔF/F, which is the same choice
the earlier notebook made. Which assumption holds for these fibres is not tested here.

**A high-pass filter** (0.001 Hz) is also computed for one example below, for comparison only: it needs no
model, but also removes genuine slow changes in the signal.

**The bleaching fit** keeps the parameterisation and bounds of the earlier notebook (constant, fast and slow
amplitudes, slow time constant 600-36000 s, fast time constant as a fraction of the slow one). It is fitted to every
13th sample (10 Hz) and the curve is evaluated at every sample: the fit has only slow timescales, a full-rate fit
costs about 5 s per channel, and the two agree to well under 1% (checked on synthetic data below).

In [ ]:
PARAMETER_NAMES = ("const", "amp_fast", "amp_slow", "tau_slow", "tau_multiplier")


def median_despike(x: np.ndarray, fs: float, window_s: float | None) -> np.ndarray:
    """Step A2: running median over *window_s* seconds; returns *x* unchanged if None.

    The window is rounded up to an odd number of samples and the edges are padded with the nearest
    value, so no artificial step appears at the ends.
    """
    if window_s is None:
        return x
    size = int(round(window_s * fs))
    size += 1 - size % 2
    return ndimage.median_filter(x, size=size, mode="nearest")


def lowpass_zero_phase(x: np.ndarray, fs: float, cutoff_hz: float, order: int = 2) -> np.ndarray:
    """Step A3: Butterworth low-pass applied forwards and backwards, so events are not shifted in time."""
    if not 0 < cutoff_hz < fs / 2:
        raise ValueError(
            f"cutoff {cutoff_hz} Hz must lie between 0 and the Nyquist frequency {fs / 2} Hz."
        )
    b, a = sps.butter(order, cutoff_hz, btype="low", fs=fs)
    return sps.filtfilt(b, a, x)


def highpass_zero_phase(x: np.ndarray, fs: float, cutoff_hz: float, order: int = 2) -> np.ndarray:
    """Zero-phase Butterworth high-pass (second-order sections, stable at very low cutoffs)."""
    sos = sps.butter(order, cutoff_hz, btype="high", fs=fs, output="sos")
    return sps.sosfiltfilt(sos, x, padtype="even")


def double_exponential(t, const, amp_fast, amp_slow, tau_slow, tau_multiplier):
    """Constant plus slow and fast decaying exponentials; the fast time constant is ``tau_slow * tau_multiplier``."""
    return (
        const
        + amp_slow * np.exp(-t / tau_slow)
        + amp_fast * np.exp(-t / (tau_slow * tau_multiplier))
    )


def fit_bleaching(
    t: np.ndarray, y: np.ndarray, settings: dict, fit_mask: np.ndarray | None = None
) -> tuple[np.ndarray, np.ndarray, tuple]:
    """Step A4 (fit): double-exponential bleaching curve for one channel.

    Args:
        t (np.ndarray): Time from the start of the recording (s).
        y (np.ndarray): The channel, low-passed (V).
        settings (dict): ``PREPROCESSING``: uses ``fit_decimation``, ``fit_initial``, ``fit_lower``,
            ``fit_upper``. The first three entries of each are fractions of ``max(y)``.
        fit_mask (np.ndarray | None): Samples the fit may use; None = all.

    Returns:
        tuple[np.ndarray, np.ndarray, tuple]: The curve at every sample (V), the fitted parameters
        (see ``PARAMETER_NAMES``), and the (lower, upper) bounds used.

    Raises:
        RuntimeError: If the fit does not converge in 1000 evaluations.
        ValueError: If the curve is not positive everywhere, which would break ΔF/F.
    """
    scale = np.array([y.max()] * 3 + [1.0, 1.0])
    bounds = (np.array(settings["fit_lower"]) * scale, np.array(settings["fit_upper"]) * scale)
    step = settings["fit_decimation"]
    used = slice(None) if fit_mask is None else fit_mask
    params, _ = curve_fit(
        double_exponential,
        t[used][::step],
        y[used][::step],
        p0=np.array(settings["fit_initial"]) * scale,
        bounds=bounds,
        maxfev=1000,
    )
    curve = double_exponential(t, *params)
    if not np.all(curve > 0):
        raise ValueError("the bleaching fit is not positive everywhere.")
    return curve, params, bounds


def regress_out_control(
    signal_detrended: np.ndarray, control_detrended: np.ndarray, fit_mask: np.ndarray | None = None
):
    """Step A5: subtract the least-squares fit of the control from the signal.

    The line is fitted on the samples of *fit_mask* (None = all) and subtracted from every sample.

    Returns:
        tuple[np.ndarray, np.ndarray, dict]: The corrected signal, the estimated movement component, and
        ``slope``, ``intercept``, ``r`` (Pearson) and ``r2`` of the regression of signal on control.
    """
    used = slice(None) if fit_mask is None else fit_mask
    fit = linregress(control_detrended[used], signal_detrended[used])
    movement = fit.intercept + fit.slope * control_detrended
    stats = {"slope": fit.slope, "intercept": fit.intercept, "r": fit.rvalue, "r2": fit.rvalue**2}
    return signal_detrended - movement, movement, stats


def delta_f_over_f(corrected: np.ndarray, baseline_fit: np.ndarray) -> np.ndarray:
    """Step A6: percent change relative to the (time-varying) bleaching fit."""
    return 100 * corrected / baseline_fit


def session_zscore(x: np.ndarray) -> np.ndarray:
    """Step A6: subtract the session mean and divide by the session standard deviation."""
    sd = x.std()
    if not sd > 0:
        raise ValueError("cannot z-score a constant trace.")
    return (x - x.mean()) / sd


def preprocess_channels(
    signal: np.ndarray, control: np.ndarray, fs: float, settings: dict, lowpass: bool = True
) -> dict:
    """Steps A2-A6 for one recording; every intermediate is returned so any stage can be inspected.

    Args:
        signal, control (np.ndarray): Baseline-subtracted channels (V), same length.
        fs (float): Sampling rate (Hz).
        settings (dict): ``PREPROCESSING``.
        lowpass (bool): Apply step A3. False gives the unfiltered movement-corrected ΔF/F that the
            spectral SNR needs (a 10 Hz low-pass would hide the noise floor above 15 Hz).

    Returns:
        dict: ``denoised_signal/control`` (V), ``fit_signal/control`` (V), ``params_signal/control``,
        ``bounds_signal/control``, ``detrended_signal/control`` (V), ``movement`` (V), ``corrected`` (V),
        ``motion_stats`` (dict), ``dff`` (%), ``uncorrected_dff`` (%, before step A5), ``control_dff`` (%, the
        control's own ΔF/F) and ``zscore``.
    """
    if signal.shape != control.shape or not (
        np.isfinite(signal).all() and np.isfinite(control).all()
    ):
        raise ValueError("signal and control must have the same length and be finite.")
    t = np.arange(len(signal)) / fs
    channels = {}
    for name, raw in (("signal", signal), ("control", control)):
        x = median_despike(raw, fs, settings["median_filter_s"])
        if lowpass:
            x = lowpass_zero_phase(x, fs, settings["lowpass_hz"], settings["lowpass_order"])
        fit, params, bounds = fit_bleaching(t, x, settings)
        channels[name] = {
            "denoised": x,
            "fit": fit,
            "params": params,
            "bounds": bounds,
            "detrended": x - fit,
        }
    corrected, movement, stats = regress_out_control(
        channels["signal"]["detrended"], channels["control"]["detrended"]
    )
    out = {"movement": movement, "corrected": corrected, "motion_stats": stats}
    for name, parts in channels.items():
        for key, value in parts.items():
            out[f"{key}_{name}"] = value
    out["dff"] = delta_f_over_f(corrected, channels["signal"]["fit"])
    out["control_dff"] = delta_f_over_f(
        channels["control"]["detrended"], channels["control"]["fit"]
    )
    out["uncorrected_dff"] = delta_f_over_f(
        channels["signal"]["detrended"], channels["signal"]["fit"]
    )
    out["zscore"] = session_zscore(corrected)
    return out

### Synthetic check of pipeline A

Before touching real data, the pipeline is run on a recording with a **known** answer: a double-exponential
bleaching curve per channel, a movement artefact that enters the signal with gain 1.0 and the control with
gain 0.8 (so the true regression slope of signal on control is 1.25), a few transients in the signal only,
and white noise. The pipeline should recover the bleaching curve, the slope, and the transients.

In [ ]:
def band_limited_noise(rng, n: int, fs: float, band: tuple[float, float], sd: float) -> np.ndarray:
    """Gaussian noise confined to *band* (Hz), scaled to standard deviation *sd*."""
    spectrum = np.fft.rfft(rng.standard_normal(n))
    freqs = np.fft.rfftfreq(n, 1 / fs)
    spectrum[(freqs < band[0]) | (freqs > band[1])] = 0
    x = np.fft.irfft(spectrum, n)
    return sd * x / x.std()


def transient_shape(
    t: np.ndarray, onset: float, rise_s: float = 0.15, decay_s: float = 1.0
) -> np.ndarray:
    """Unit-peak transient: fast rise, slower exponential decay, starting at *onset*."""
    dt = np.clip(t - onset, 0, None)
    shape = (1 - np.exp(-dt / rise_s)) * np.exp(-dt / decay_s)
    return shape / shape.max()


def synthetic_recording(
    rng, fs=130.0, minutes=15.0, noise_sd=0.002, n_transients=20, amplitude=0.03
):
    """Signal and control with a known bleaching curve, shared movement and signal-only transients.

    Returns:
        tuple[np.ndarray, np.ndarray, dict]: signal (V), control (V), and the truth: ``fit_signal``,
        ``fit_control``, ``slope`` (true signal-on-control slope), ``transients`` (signal-only part) and
        ``onsets`` (s).
    """
    n = int(minutes * 60 * fs)
    t = np.arange(n) / fs
    fit_signal = 1.2 + 0.9 * np.exp(-t / 3000) + 0.4 * np.exp(-t / 300)
    fit_control = 1.0 + 0.7 * np.exp(-t / 3000) + 0.3 * np.exp(-t / 300)
    movement = band_limited_noise(rng, n, fs, (0.1, 4.0), sd=0.02)
    onsets = np.sort(rng.uniform(20, minutes * 60 - 20, n_transients))
    transients = amplitude * sum(transient_shape(t, onset) for onset in onsets)
    signal = fit_signal + movement + transients + noise_sd * rng.standard_normal(n)
    control = fit_control + 0.8 * movement + noise_sd * rng.standard_normal(n)
    truth = {
        "fit_signal": fit_signal,
        "fit_control": fit_control,
        "slope": 1 / 0.8,
        "transients": transients,
        "onsets": onsets,
    }
    return signal, control, truth


def check_pipeline_a_on_synthetic_data(settings: dict) -> pd.DataFrame:
    """Run pipeline A on a synthetic recording and assert that it recovers the truth."""
    fs = 130.0
    signal, control, truth = synthetic_recording(np.random.default_rng(1), fs)
    out = preprocess_channels(signal, control, fs, settings)
    bleaching_error = np.abs(out["fit_signal"] / truth["fit_signal"] - 1).max()
    slope_error = abs(out["motion_stats"]["slope"] / truth["slope"] - 1)
    expected = lowpass_zero_phase(
        truth["transients"], fs, settings["lowpass_hz"], settings["lowpass_order"]
    )
    transient_r = np.corrcoef(out["corrected"], expected)[0, 1]
    # full-rate fit for comparison with the decimated one
    full_rate = {**settings, "fit_decimation": 1}
    full_fit, _, _ = fit_bleaching(np.arange(len(signal)) / fs, out["denoised_signal"], full_rate)
    decimation_error = np.abs(out["fit_signal"] / full_fit - 1).max()
    assert bleaching_error < 0.02, bleaching_error
    assert slope_error < 0.10, slope_error
    assert transient_r > 0.95, transient_r
    assert decimation_error < 0.01, decimation_error
    assert np.isclose(out["zscore"].mean(), 0, atol=1e-9) and np.isclose(out["zscore"].std(), 1)
    return pd.DataFrame(
        {
            "check": [
                "Bleaching fit: largest relative error",
                "Movement slope: relative error",
                "Corrected signal vs true transients: r",
                "Decimated vs full-rate fit: largest relative difference",
            ],
            "value": [bleaching_error, slope_error, transient_r, decimation_error],
            "limit": ["< 0.02", "< 0.10", "> 0.95", "< 0.01"],
        }
    )


display(check_pipeline_a_on_synthetic_data(PREPROCESSING))


def check_median_despike() -> None:
    """The optional spike filter removes a single-sample spike and leaves a slow ramp alone."""
    ramp = np.linspace(0, 1, 500)
    spiked = ramp.copy()
    spiked[250] += 10
    cleaned = median_despike(spiked, fs=100.0, window_s=0.05)  # 5 samples
    assert np.abs(cleaned - ramp).max() < 0.01
    assert median_despike(spiked, 100.0, None) is spiked


check_median_despike()
print(
    "Median filter check passed (it removes a one-sample spike and is a no-op when switched off)."
)

### Cache the per-recording preprocessing

Fitting four exponentials and filtering per recording is the slow part of this notebook. Results are cached to
disk with `joblib.Memory`, **keyed on the file path, file size, modification time and the `PREPROCESSING`
dictionary**, so editing a parameter or replacing a file recomputes exactly that recording, and deleting
the cache folder only costs time. The loaded recording is passed along but excluded from the key (it
is fully determined by the file). The cache lives in `get_settings().cache_dir / "photometry"` and is git-ignored.

In [ ]:
memory = joblib.Memory(get_settings().cache_dir / "photometry", verbose=0)


def file_key(path: Path) -> tuple[str, int, int]:
    """Identity of a file for caching: path, size in bytes and modification time in ns."""
    info = Path(path).stat()
    return str(path), info.st_size, info.st_mtime_ns


@memory.cache(ignore=["recording"])
def preprocess_recording_cached(key: tuple, settings: dict, recording) -> dict:
    """Pipeline A (steps A2-A6) for one recording, with and without the low-pass.

    *key* and *settings* identify the computation for the cache; *recording* supplies the data.
    The unfiltered run keeps only the traces the spectral SNR needs.
    """
    filtered = preprocess_channels(recording.signal, recording.control, recording.fs, settings)
    unfiltered = preprocess_channels(
        recording.signal, recording.control, recording.fs, settings, lowpass=False
    )
    return {
        "filtered": filtered,
        "unfiltered": {
            k: unfiltered[k] for k in ("dff", "uncorrected_dff", "control_dff", "motion_stats")
        },
    }


def preprocess_recording(recording) -> dict:
    """Cached pipeline A for *recording*."""
    return preprocess_recording_cached(file_key(recording.ppd_path), PREPROCESSING, recording)

### Run pipeline A on every recording

This is the cached step; the first run takes a few minutes, later runs seconds.

In [ ]:
_start = time.time()
PROCESSED = {}
for rid, rec in RECORDINGS.items():
    PROCESSED[rid] = preprocess_recording(rec)
print(f"Pipeline A on {len(PROCESSED)} recordings: {time.time() - _start:.0f} s.")

### Every stage for one example recording

The example is the same recording as in section 2. Each figure has the whole session on the left and a 60-second
zoom on the right (the zoom is centred on the rewarded outcome closest to the middle of the session; ticks
on the last figure mark outcomes). Panels in a column share their time axis, which is pyControl time. Colours:
green is the nLightG2 signal (470 nm, assumed), purple the isosbestic control (405 nm, assumed).

In [ ]:
SIGNAL_COLOR, CONTROL_COLOR = "tab:green", "tab:purple"
Line = tuple[str, np.ndarray, dict]  # label, y, matplotlib line keywords


def zoom_window(recording, events: pd.DataFrame, length_s: float = 60.0) -> tuple[float, float]:
    """A *length_s* window centred on the rewarded outcome nearest the middle of the sync coverage."""
    first, last = sync_coverage(recording.time)
    rewarded = events.loc[events["condition"] == "rewarded", "outcome_time"].to_numpy()
    centre = rewarded[np.argmin(np.abs(rewarded - (first + last) / 2))]
    start = float(np.clip(centre - length_s / 2, first, last - length_s))
    return start, start + length_s


def plot_lines(ax, time: np.ndarray, lines: list[Line], ylabel: str, legend: bool = True) -> None:
    """Draw labelled traces against *time* on *ax*."""
    for label, y, style in lines:
        ax.plot(time, y, label=label, lw=0.6, **style)
    ax.set_ylabel(ylabel)
    if legend:
        ax.legend(loc="upper right", ncol=len(lines), framealpha=0.85)


def stage_figure(time, rows, zoom, title, outcome_times=None, axes=None):
    """One figure per processing stage: each row is ``(ylabel, lines)``; whole session | zoom.

    Panels in a column share their x-axis. *outcome_times* (optional, dict of colour -> times) adds ticks
    for outcomes along the bottom of the zoom column. *axes* is an optional ``(len(rows), 2)`` array of
    axes to draw on; if None a new figure is created. Returns the figure drawn on.
    """
    if axes is None:
        _, axes = plt.subplots(
            len(rows), 2, figsize=(13, 1.9 * len(rows) + 0.8), sharex="col", constrained_layout=True
        )
    axes = np.asarray(axes).reshape(len(rows), 2)
    fig = axes[0, 0].figure
    first, last = sync_coverage(time)
    for (ylabel, lines), (ax_full, ax_zoom) in zip(rows, axes, strict=True):
        plot_lines(ax_full, time, lines, ylabel, legend=True)
        plot_lines(ax_zoom, time, lines, ylabel, legend=False)
    axes[0, 0].set_xlim(first, last)
    axes[0, 1].set_xlim(*zoom)
    axes[0, 0].set_title("Whole Session")
    axes[0, 1].set_title("60-s Zoom")
    for ax in axes[-1]:
        ax.set_xlabel("pyControl Time (s)")
    if outcome_times:
        for colour, times in outcome_times.items():
            inside = times[(times >= zoom[0]) & (times <= zoom[1])]
            axes[-1, 1].plot(
                inside, np.full(len(inside), axes[-1, 1].get_ylim()[0]), "|", color=colour, ms=12
            )
    fig.suptitle(title, fontsize=12)
    return fig


EXAMPLE_REC, EXAMPLE_PROC = RECORDINGS[EXAMPLE], PROCESSED[EXAMPLE]["filtered"]
EXAMPLE_TIME = EXAMPLE_REC.time
EXAMPLE_ZOOM = zoom_window(EXAMPLE_REC, EVENTS[EXAMPLE])
EXAMPLE_OUTCOMES = {
    "tab:green": EVENTS[EXAMPLE].query("condition == 'rewarded'")["outcome_time"].to_numpy(),
    "tab:red": EVENTS[EXAMPLE].query("condition == 'unrewarded'")["outcome_time"].to_numpy(),
}
print(f"Zoom window: {EXAMPLE_ZOOM[0]:.0f} to {EXAMPLE_ZOOM[1]:.0f} s")

#### A1: LED-off baseline subtraction

pyPhotometry in pulsed mode records each channel twice per cycle: with the LED on, and with it off. The
importer subtracts the LED-off sample from the LED-on sample, which removes ambient light and electronic offset.
**Look for:** a LED-off baseline that is small (tens of mV) and flat compared with the LED-on level; the difference
should follow the LED-on trace closely, shifted down by the baseline. In the example the baselines also show brief upward
spikes of a few mV in both channels; they subtract as small downward blips. Their origin is unknown (they are not at the
rsync pulses) and they are counted per recording in the acquisition QC (section 6.1).

In [ ]:
fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                ("LED on", EXAMPLE_REC.signal_led_on, {"color": "0.6"}),
                ("On minus off", EXAMPLE_REC.signal, {"color": SIGNAL_COLOR}),
            ],
        ),
        ("Signal LED off (V)", [("LED-off baseline", EXAMPLE_REC.signal_baseline, {"color": "k"})]),
        (
            "Control (V)",
            [
                ("LED on", EXAMPLE_REC.control_led_on, {"color": "0.6"}),
                ("On minus off", EXAMPLE_REC.control, {"color": CONTROL_COLOR}),
            ],
        ),
        (
            "Control LED off (V)",
            [("LED-off baseline", EXAMPLE_REC.control_baseline, {"color": "k"})],
        ),
    ],
    EXAMPLE_ZOOM,
    f"A1: LED-Off Baseline Subtraction ({EXAMPLE})",
)
save_figure(fig, "example_A1_baseline_subtraction")
plt.show()

#### A2: optional median filter

Off by default (`median_filter_s=None`). The raw traces do contain a few isolated single-sample dips that hit both channels at the same sample (for example near 745 s in the example); the regression in A5 removes shared artefacts of that kind, so the filter is left off. A running median is a
non-linear filter that removes brief single-sample spikes without blurring edges the way a low-pass does. If you
need it, set `median_filter_s` to about five times the spike duration (for example 0.04 s for 8-ms spikes). It is
applied before the low-pass, because a low-pass would smear a spike into a bump that a median can no longer
remove. The synthetic check above shows it removing a one-sample spike.

#### A3: low-pass filter (10 Hz)

**Look for:** the high-frequency fuzz of the raw trace is gone while the slow structure is unchanged.

In [ ]:
fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                ("Before", EXAMPLE_REC.signal, {"color": "0.6"}),
                ("Low-passed 10 Hz", EXAMPLE_PROC["denoised_signal"], {"color": SIGNAL_COLOR}),
            ],
        ),
        (
            "Control (V)",
            [
                ("Before", EXAMPLE_REC.control, {"color": "0.6"}),
                ("Low-passed 10 Hz", EXAMPLE_PROC["denoised_control"], {"color": CONTROL_COLOR}),
            ],
        ),
    ],
    EXAMPLE_ZOOM,
    f"A3: Zero-Phase Low-Pass Filter ({EXAMPLE})",
)
save_figure(fig, "example_A3_lowpass")
plt.show()

#### A4: bleaching correction

A double exponential is fitted to each low-passed channel and subtracted. The high-pass alternative (0.001 Hz,
dashed) is shown on the detrended rows for comparison only; it does not feed anything downstream.
**Look for:** a smooth fit through the middle of the data without following fast fluctuations, and a
detrended trace that fluctuates around zero without a remaining slope. The two methods should agree except
for very slow components.

In [ ]:
t_example = np.arange(len(EXAMPLE_REC.signal)) / EXAMPLE_REC.fs
highpass = {
    name: highpass_zero_phase(
        EXAMPLE_PROC[f"denoised_{name}"], EXAMPLE_REC.fs, HIGHPASS_ALTERNATIVE_HZ
    )
    for name in ("signal", "control")
}
fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                ("Low-passed", EXAMPLE_PROC["denoised_signal"], {"color": SIGNAL_COLOR}),
                ("Double-exponential fit", EXAMPLE_PROC["fit_signal"], {"color": "k"}),
            ],
        ),
        (
            "Signal Detrended (V)",
            [
                ("Fit subtracted", EXAMPLE_PROC["detrended_signal"], {"color": SIGNAL_COLOR}),
                ("High-pass alternative", highpass["signal"], {"color": "k", "ls": "--"}),
            ],
        ),
        (
            "Control (V)",
            [
                ("Low-passed", EXAMPLE_PROC["denoised_control"], {"color": CONTROL_COLOR}),
                ("Double-exponential fit", EXAMPLE_PROC["fit_control"], {"color": "k"}),
            ],
        ),
        (
            "Control Detrended (V)",
            [
                ("Fit subtracted", EXAMPLE_PROC["detrended_control"], {"color": CONTROL_COLOR}),
                ("High-pass alternative", highpass["control"], {"color": "k", "ls": "--"}),
            ],
        ),
    ],
    EXAMPLE_ZOOM,
    f"A4: Bleaching Correction ({EXAMPLE})",
)
save_figure(fig, "example_A4_bleaching")
plt.show()

#### A5: movement correction

The detrended control is regressed onto the detrended signal (ordinary least squares with an intercept), and
the fitted line is subtracted from the signal. The slope says how much of a control fluctuation appears in the
signal; R² says how much of the signal's variance the control explains. **Look for:** a tight
straight-line scatter if the shared fluctuations are movement-like. A high R² means most of the detrended signal
was shared with the control and has been removed; it does not, by itself, show that what remains is sensor
activity. A second cloud of points off the main line means part of the signal's variance is not shared with the
control (for example a slow dip in the signal alone); the subtraction leaves that part in the corrected trace.

In [ ]:
def plot_regression(ax, control_detrended, signal_detrended, stats, step: int = 20) -> None:
    """Scatter (every *step*-th sample) of detrended signal against detrended control with the fitted line."""
    ax.plot(control_detrended[::step], signal_detrended[::step], ".", ms=2, alpha=0.3, color="0.4")
    x = np.array([control_detrended.min(), control_detrended.max()])
    ax.plot(
        x,
        stats["intercept"] + stats["slope"] * x,
        color="tab:red",
        label=f"slope {stats['slope']:.2f}, R² {stats['r2']:.2f}",
    )
    ax.set(
        xlabel="Detrended Control (V)",
        ylabel="Detrended Signal (V)",
        title="Control-Signal Regression",
    )
    ax.legend()


fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                (
                    "Detrended",
                    EXAMPLE_PROC["detrended_signal"],
                    {"color": SIGNAL_COLOR, "alpha": 0.6},
                ),
                ("Estimated movement", EXAMPLE_PROC["movement"], {"color": "tab:orange"}),
            ],
        ),
        ("Corrected (V)", [("Signal minus movement", EXAMPLE_PROC["corrected"], {"color": "k"})]),
    ],
    EXAMPLE_ZOOM,
    f"A5: Movement Correction ({EXAMPLE})",
)
save_figure(fig, "example_A5_movement_time")
plt.show()

fig, ax = plt.subplots(figsize=(4.5, 4), constrained_layout=True)
plot_regression(
    ax,
    EXAMPLE_PROC["detrended_control"],
    EXAMPLE_PROC["detrended_signal"],
    EXAMPLE_PROC["motion_stats"],
)
save_figure(fig, "example_A5_movement_scatter")
plt.show()

#### A6: normalisation

ΔF/F (%) divides the corrected signal by the signal's bleaching fit, so it is a percent change from the
time-varying baseline fluorescence. The session z-score subtracts the session mean and divides by the session SD
of the corrected signal, so it is in units of that session's variability. They have the same shape here
(the baseline changes slowly) and differ in units: z-scores are comparable across sessions only if sessions
have similar variability. Green and red ticks in the zoom mark rewarded and unrewarded outcomes.

In [ ]:
fig = stage_figure(
    EXAMPLE_TIME,
    [
        ("Corrected (V)", [("Signal minus movement", EXAMPLE_PROC["corrected"], {"color": "k"})]),
        ("ΔF/F (%)", [("Session ΔF/F", EXAMPLE_PROC["dff"], {"color": SIGNAL_COLOR})]),
        ("Session z-score", [("Session z-score", EXAMPLE_PROC["zscore"], {"color": "tab:blue"})]),
    ],
    EXAMPLE_ZOOM,
    f"A6: Normalisation ({EXAMPLE})",
    outcome_times=EXAMPLE_OUTCOMES,
)
save_figure(fig, "example_A6_normalisation")
plt.show()

#### Units check: is ΔF/F (%) the right size?

The responses in sections 7-8 are of the order of 0.05-0.1 % ΔF/F, which is small for a norepinephrine sensor. Before reading anything into that,
the arithmetic is checked on the example recording: ΔF/F (%) = 100 x corrected signal (V) / bleaching fit of the signal (V). The fit is the *total*
baseline fluorescence after LED-off subtraction (about 2.6 V here), so 1 % ΔF/F is about 26 mV. The check below prints the
numbers and asserts that ΔF/F equals 100 x corrected / fit sample by sample, and that its SD equals 100 x the SD of the corrected signal over the median fit
(to within the slow variation of the fit).

The table after it gives, **for every recording**, the SD of the session ΔF/F (%) *before* and *after* movement correction, so the size of what
the regression removes is visible. Both are of the low-passed traces used downstream.

In [ ]:
def units_check(processed: dict, raw_signal: np.ndarray) -> dict:
    """Numbers behind the ΔF/F scale of one recording; asserts the x100 and the denominator."""
    fit, corrected, dff = processed["fit_signal"], processed["corrected"], processed["dff"]
    expected_sd_pct = 100 * corrected.std() / np.median(fit)
    assert np.allclose(dff, 100 * corrected / fit)
    assert np.isclose(dff.std(), expected_sd_pct, rtol=0.1), (dff.std(), expected_sd_pct)
    return {
        "median raw signal, LED on minus off (V)": np.median(raw_signal),
        "median bleaching fit F (V)": np.median(fit),
        "SD of corrected signal (V)": corrected.std(),
        "SD of corrected signal as ΔF/F (%)": dff.std(),
        "expected: 100 x SD (V) / median F (%)": expected_sd_pct,
        "1 % ΔF/F corresponds to (mV)": 10 * np.median(fit),
    }


for key, value in units_check(EXAMPLE_PROC, EXAMPLE_REC.signal).items():
    print(f"{key:42s} {value:10.4f}")


def correction_size_row(processed: dict) -> dict:
    """SD of the session ΔF/F (%) before and after movement correction, and the share of the SD removed."""
    before, after = processed["uncorrected_dff"].std(), processed["dff"].std()
    return {
        "sd_dff_before_pct": before,
        "sd_dff_after_pct": after,
        "sd_removed_pct": 100 * (1 - after / before),
        "median_F_v": np.median(processed["fit_signal"]),
    }


correction_size = pd.DataFrame(
    [
        {"recording": rid, "mouse": RECORDINGS[rid].subject, **correction_size_row(p["filtered"])}
        for rid, p in PROCESSED.items()
    ]
)
show_table(correction_size, "qc_correction_size")

## 4. Pipeline B: the nLightG2 paper and the authors' code

Rohner, Curreli, Lamothe-Molina, ..., Patriarchi (2026), "Next-generation multicolor indicators for in vivo imaging of
norepinephrine", *Nature Methods* 23:636-652, <https://doi.org/10.1038/s41592-026-03006-z>.

**Sources.**

- **Paper**: Methods, "Data analysis of photometry recordings during fear conditioning experiments" (page 21 of the 43-page
  PDF supplied with this project, the Methods after the main text). Cited as *paper*.
- **Code**: the authors' MATLAB scripts (repository `patriarchilab/nlightg2`): `aPhotometryData_IndividualMice_APL_v20240526.m`
  (*individual*: one mouse, whole trace and per event) and `aPhotometryData_Cohorts_APL_v20240526.m` (*cohort*: averaging across
  mice). Cited by line number. The code processes a fear-conditioning task (tone, footshock); the line numbers are for the
  tone-aligned analysis, which is the one the paper's text describes.

Where the paper is silent the code is followed. Where they differ, each row below says which is followed and why: the
low-pass follows the paper (a causal filter would shift event times), and the ΔF/F baseline follows the code (the paper's 20-s baseline
is stated for tail-lift trials only, and the task here is not a tail-lift). Each choice is a switch in the configuration cell
(`B_PREPROCESSING`, `B_EVENT_LEVEL`).

| Step | Paper | Code | Here (default; switch) |
|---|---|---|---|
| B1 low-pass | Butterworth, green channel 10 Hz, order 2 | 40 Hz, order 2, MATLAB `filter`, i.e. causal (individual l.926-941) | 10 Hz, order 2 (**the paper is followed**: the paper states the cut-off, and the code's causal filter would delay the trace relative to the outcome times), applied forwards and backwards so event times are not shifted: the same function as A3 |
| B2 bleaching | "double exponential detrending" | `fit(..., 'exp2')`, i.e. a·e^(bt) + c·e^(dt) with no constant and no bounds, fitted from sample 100 (1-based, i.e. the first 99 samples are skipped) to 30 s before the end (l.1142-1166). If the fit throws, it is replaced by an all-zero fit without stopping (l.1153-1166) | `bleaching_model = "exp2"`, `fit_skip_samples = 99` (MATLAB's sample 100), `fit_skip_end_s = 30`. If the exp2 fit fails here, that channel falls back to the bounded double exponential with a constant (pipeline A's), and the recording is listed below. Failure means non-convergence, or a curve that is not positive and finite (it is the denominator of ΔF/F), or **this notebook's own amplitude-ratio rule** (amplitudes over `exp2_max_amplitude_ratio` times the signal, i.e. two nearly equal rates cancelling). **MATLAB's `fit` has no such rule**, so a fit rejected by it would have been used by the authors' code; the table below reports how well the rejected curve fits inside the fit window against the bounded one |
| B3 motion | 405-nm fitted linearly to the functional signal and subtracted | `fitlm` of detrended 405 onto detrended 465 inside the fit window, **leaving out the 20 s after every TTL**; the line is then subtracted from the whole trace (l.1198-1225) | Same regression. `motion_mask_s = 0` (off): our outcomes come about every 10 s, so a 20-s mask would remove nearly the whole session (measured below). The fraction masked at 2, 5, 10 and 20 s is reported, with the slope each would give |
| B4 ΔF/F0 | normalised by the bleaching fit, ×100 | l.1467 | same function as A6 |
| B5 event ΔF/F | "For TL trials, the baseline was defined as 20 s before the event": stated for tail-lift (TL) trials only | `movmean(100·corrected/fit)`, **no baseline subtraction** (l.1964) | `subtract_dff_baseline = False`: **the code is followed**, because the paper's baseline is stated for a different trial type and does not say that outcome-aligned ΔF/F is baseline-subtracted. True subtracts the mean of the pre-window (the paper's tail-lift baseline) |
| B6 trial z | subtract the mean of 15 s before the cue, divide by its SD | `(corrected - mean_pre) / SD_pre` on the **motion-corrected signal in volts**, baseline from 16 s before the event to one sample before it (`SecBeforeTone_start = 16`, l.2149; l.2181-2197), then `movmean`. The other per-event z in the same script (l.1967-1969) uses the mean and SD of the whole event window and is not the paper's | `z_from = "corrected"` (code); `"dff"` takes the z-score of the ΔF/F windows instead. SD is the sample SD (MATLAB `std`) |
| B7 smoothing | 80-point moving mean | `movmean(..., Window_movmean)` after normalisation; `Window_movmean` is typed in per dataset ("10 for ok signal, 50-100 for noisy slow signal", l.85; the dialog default is 80, l.119) | `DEFAULT_SMOOTHING = 80`; MATLAB's `movmean` handling of even windows and of the ends |
| B8 averaging | trial averages per mouse, then across the cohort | each mouse's trial average is one column; mean across mice and SEM = SD/√(mice) (cohort l.2245-2249 and l.2333-2334) | the group figures (7d) do the same |
| B9 metrics | peak z in the stimulus period; AUC (`trapz`) of the mean trial z | | as before (section 7) |

**Sampling rate of the "80 points" is not stated in the paper.** Its Methods say the Doric console demodulated the signal and
low-pass filtered it at 12 Hz ("Fiber photometry recordings"), but not the rate at which the trace was saved. The code takes the
rate from the file (`FpS = 1/dT`, l.931). The cohort script down-samples traces for plotting only when `FpS > 200` Hz, to 122 Hz
(l.2357-2360), which suggests the traces can be recorded faster than 200 Hz; if so, 80 points are at most 0.4 s there, against
0.615 s at our 130 Hz. This is an inference, not a statement in the source. The sensitivity analysis (7c) varies the length.

**Event-level normalisation needs a pre-event window of its own, so trials whose baseline contains another outcome are kept**
(the next trial can start within seconds); this is counted in section 7c rather than excluded. With outcomes about 10 s apart, a
15-s baseline almost always contains the previous outcome's response.

**Smoothing at the ends of the window.** The moving mean (B7) is applied to windows padded on both sides by its half-width and the result is
cropped to the displayed window, so no displayed sample is smoothed with a shortened (edge) window. The padded part of a window has
to lie inside the sync coverage too, so a few trials at the very start or end of a recording are lost compared with an unpadded
window.

**Choices made here.** The pre-window default is 5 s (`DEFAULT_PRE_WINDOW`) rather than the paper's 15 s, because of the overlap just
described and because longer windows discard more trials (7c). The post-outcome mask of the motion fit is off (`motion_mask_s`), for the
reason in the table.

In [ ]:
def interval_mask(rel_time: np.ndarray, interval: tuple[float, float]) -> np.ndarray:
    """Samples with ``start <= t < stop``; the end is excluded so adjacent windows do not share a sample."""
    mask = (rel_time >= interval[0]) & (rel_time < interval[1])
    if mask.sum() < 2:
        raise ValueError(f"interval {interval} holds fewer than two samples of the window.")
    return mask


def matlab_moving_mean(values: np.ndarray, points: int) -> np.ndarray:
    """Centred moving mean along the last axis, as MATLAB's ``movmean``; 1 point returns the values.

    For an even window the samples are ``points/2`` before and ``points/2 - 1`` after the current one;
    near the ends the window shrinks rather than being padded.
    """
    if points < 1 or int(points) != points:
        raise ValueError("the moving-mean window must be a positive integer.")
    values = np.asarray(values, dtype=float)
    if points == 1:
        return values.copy()
    n = values.shape[-1]
    index = np.arange(n)
    starts = np.maximum(0, index - points // 2)
    stops = np.minimum(n, index + (points - 1) // 2 + 1)
    sums = np.concatenate([np.zeros(values.shape[:-1] + (1,)), np.cumsum(values, axis=-1)], axis=-1)
    return (sums[..., stops] - sums[..., starts]) / (stops - starts)


def smoothing_pad_s(points: int, fs: float) -> float:
    """Seconds a moving mean of *points* samples reaches to either side of a sample (plus one sample of rounding)."""
    return (points // 2 + 1) / fs


def padded_window(window: tuple[float, float], points: int, fs: float) -> tuple[float, float]:
    """*window* widened on both sides so that smoothing with *points* samples never meets the edge of the extraction."""
    pad = smoothing_pad_s(points, fs)
    return window[0] - pad, window[1] + pad


def crop_mask(rel_time: np.ndarray, window: tuple[float, float], fs: float) -> np.ndarray:
    """Samples of the grid *rel_time* inside *window*, both ends included, compared as whole samples."""
    index = np.rint(rel_time * fs)
    return (index >= round(window[0] * fs)) & (index <= round(window[1] * fs))


@dataclass(frozen=True)
class ViewSettings:
    """How event windows become the four displayed views (frozen, so a figure's settings cannot drift).

    Attributes:
        pre_window (tuple[float, float]): Baseline interval before the outcome (s, ``start < stop <= 0``) of
            pipeline B's trial normalisation.
        smoothing_a (int): Moving-mean length in samples for pipeline A's views (1 = none).
        smoothing_b (int): The same for pipeline B's views.
        subtract_dff_baseline (bool): Pipeline B: subtract the pre-window mean from each trial's ΔF/F.
    """

    pre_window: tuple[float, float]
    smoothing_a: int
    smoothing_b: int
    subtract_dff_baseline: bool

    def __post_init__(self):
        if not self.pre_window[0] < self.pre_window[1] <= 0:
            raise ValueError(f"pre_window {self.pre_window} must satisfy start < stop <= 0.")
        for points in (self.smoothing_a, self.smoothing_b):
            if points < 1 or int(points) != points:
                raise ValueError(
                    f"smoothing must be a positive integer number of samples, not {points}."
                )


def subtract_baseline(
    trials: np.ndarray, rel_time: np.ndarray, baseline: tuple[float, float]
) -> np.ndarray:
    """Each trial minus its own mean over *baseline*."""
    mask = interval_mask(rel_time, baseline)
    return trials - trials[:, mask].mean(axis=1, keepdims=True)


def trial_zscore(trials: np.ndarray, rel_time: np.ndarray, baseline: tuple[float, float]):
    """Step B6: each trial minus its own baseline mean, divided by its own baseline sample SD.

    Returns:
        tuple[np.ndarray, np.ndarray]: The z-scored trials, and a boolean mask that is False for trials
        whose baseline SD is zero or not finite (those rows are NaN).
    """
    mask = interval_mask(rel_time, baseline)
    mean = trials[:, mask].mean(axis=1, keepdims=True)
    sd = trials[:, mask].std(axis=1, ddof=1, keepdims=True)
    valid = np.isfinite(sd[:, 0]) & (sd[:, 0] > 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.where(valid[:, None], (trials - mean) / sd, np.nan)
    return z, valid


def normalise_trials_b(
    dff_trials: np.ndarray, z_trials: np.ndarray, rel_time: np.ndarray, settings: ViewSettings
):
    """Steps B5-B7 for the windows of one recording and outcome.

    Args:
        dff_trials (np.ndarray): ``(n_trials, n_time)`` ΔF/F (%), unsmoothed.
        z_trials (np.ndarray): The same trials of the signal the z-score is taken from (the
            motion-corrected signal or the ΔF/F; the z-score is unchanged by scale).
        rel_time (np.ndarray): Time relative to the outcome (s).
        settings (ViewSettings): Pre-window, pipeline-B smoothing and whether to subtract the baseline from ΔF/F.

    Returns:
        tuple[dict, np.ndarray]: ``{"dff": ΔF/F, "z": trial z-score}``, each smoothed, for the valid trials only,
        and the validity mask over the input trials (False where the baseline SD is zero).
    """
    z, valid = trial_zscore(z_trials, rel_time, settings.pre_window)
    dff = (
        subtract_baseline(dff_trials, rel_time, settings.pre_window)
        if settings.subtract_dff_baseline
        else dff_trials
    )
    return {
        "dff": matlab_moving_mean(dff[valid], settings.smoothing_b),
        "z": matlab_moving_mean(z[valid], settings.smoothing_b),
    }, valid


def trace_windows(rec, traces: dict[str, np.ndarray], event_times: np.ndarray, window):
    """Windows (complete ones only) of every trace in *traces* around *event_times*.

    Returns:
        tuple: The relative-time grid (s), ``{trace name: (n_kept, n_time)}`` and the mask of events kept.
    """
    windows = {}
    for name, trace in traces.items():
        rel_time, windows[name], keep = extract_event_windows(
            rec.time, trace, event_times, window, rec.fs
        )
    return rel_time, windows, keep


def mean_and_sem(trials: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Mean over the rows and the standard error of the mean (NaN with a single row)."""
    n = len(trials)
    if n == 0:
        raise ValueError("cannot average zero traces.")
    sem = trials.std(axis=0, ddof=1) / np.sqrt(n) if n > 1 else np.full(trials.shape[1], np.nan)
    return trials.mean(axis=0), sem


def peak_and_auc(rel_time: np.ndarray, trace: np.ndarray, window: tuple[float, float]) -> dict:
    """Largest value, its time, and the signed trapezoidal area of *trace* over *window* (ends included)."""
    mask = (rel_time >= window[0]) & (rel_time <= window[1])
    if mask.sum() < 2:
        raise ValueError(f"window {window} holds fewer than two samples.")
    segment, times = trace[mask], rel_time[mask]
    peak = int(np.argmax(segment))
    return {
        "peak": float(segment[peak]),
        "peak_time_s": float(times[peak]),
        "auc": float(trapezoid(segment, x=times)),
    }

### Pipeline B, steps B1-B4: the whole trace

Small functions, each checked on synthetic data below. `fit_exp2` fits a·e^(bt) + c·e^(dt) by searching a grid of rate
pairs (a constant is the rate 0) for the best least-squares fit, then refining the two rates; for given rates the two amplitudes
are linear and are solved exactly. MATLAB's `fit` does not document its start values, so this is the same model, not the same
optimiser. Like the importer-independent pipeline-A fit it uses every 13th sample (`fit_decimation`).

In [ ]:
class Exp2Rejected(ValueError):
    """The exp2 refinement converged but the curve is not acceptable; *curve* is that curve (V) for inspection."""

    def __init__(self, message: str, curve: np.ndarray):
        super().__init__(message)
        self.curve = curve


def fit_window_mask(n_samples: int, fs: float, skip_samples: int, skip_end_s: float) -> np.ndarray:
    """Samples used for the bleaching and motion fits: all but the first *skip_samples* and last *skip_end_s* s."""
    if skip_samples < 0 or skip_end_s < 0:
        raise ValueError("the samples and seconds to skip must not be negative.")
    stop = n_samples - int(round(skip_end_s * fs))
    if stop - skip_samples < 10 * fs:
        raise ValueError(
            f"skipping {skip_samples} samples and {skip_end_s} s leaves under 10 s of {n_samples} samples to fit."
        )
    mask = np.zeros(n_samples, dtype=bool)
    mask[skip_samples:stop] = True
    return mask


def post_event_mask(time: np.ndarray, event_times: np.ndarray, duration_s: float) -> np.ndarray:
    """Samples whose pyControl time lies within *duration_s* after any event (False where the time is NaN)."""
    if duration_s < 0:
        raise ValueError("the mask duration must not be negative.")
    mask = np.zeros(len(time), dtype=bool)
    events = np.sort(np.asarray(event_times, dtype=float))
    if duration_s == 0 or len(events) == 0:
        return mask
    finite = np.isfinite(time)
    last = (
        np.searchsorted(events, time[finite], side="right") - 1
    )  # latest event at or before each sample
    since = np.where(last >= 0, time[finite] - events[np.maximum(last, 0)], np.inf)
    mask[finite] = since < duration_s
    return mask


def exp2_curve(t: np.ndarray, a: float, b: float, c: float, d: float) -> np.ndarray:
    """MATLAB's ``exp2``: ``a*exp(b*t) + c*exp(d*t)``."""
    with np.errstate(over="ignore"):
        return a * np.exp(b * t) + c * np.exp(d * t)


def exp2_amplitudes(t: np.ndarray, y: np.ndarray, b: float, d: float):
    """Least-squares amplitudes ``(a, c)`` of the two exponentials with rates *b* and *d*, and the fitted values."""
    basis = np.column_stack([np.exp(b * t), np.exp(d * t)])
    amplitudes = np.linalg.lstsq(basis, y, rcond=None)[0]
    return amplitudes, basis @ amplitudes


def fit_exp2(t: np.ndarray, y: np.ndarray, fit_mask: np.ndarray, decimation: int, b_settings: dict):
    """Step B2 (fit): the ``exp2`` bleaching curve of one channel.

    Args:
        t (np.ndarray): Time from the start of the recording (s).
        y (np.ndarray): The channel, low-passed (V).
        fit_mask (np.ndarray): Samples the fit may use.
        decimation (int): Every *decimation*-th sample of the mask is used.
        b_settings (dict): ``B_PREPROCESSING``; the ``exp2_*`` entries set the grid and the amplitude rule.

    Returns:
        tuple[np.ndarray, np.ndarray]: The curve at every sample (V) and ``[a, b, c, d]``.

    Raises:
        RuntimeError: If the refinement does not converge.
        Exp2Rejected: If the amplitudes cancel each other (this notebook's rule; MATLAB's ``fit`` has none), or the
            curve is not finite and positive everywhere (it is the denominator of ΔF/F). Carries the curve.
    """
    used = np.flatnonzero(fit_mask)[::decimation]
    t_fit, y_fit = t[used], y[used]
    span = t_fit[-1] - t_fit[0]  # rates are refined in units of 1/span so both are of order one
    taus = np.geomspace(*b_settings["exp2_tau_grid_s"], b_settings["exp2_grid_size"])
    rates = np.concatenate([[0.0], -1 / taus])
    sse = {
        pair: float(np.sum((exp2_amplitudes(t_fit, y_fit, *pair)[1] - y_fit) ** 2))
        for pair in combinations(rates, 2)
    }
    start = np.array(min(sse, key=sse.get)) * span

    def residuals(scaled_rates):
        return exp2_amplitudes(t_fit, y_fit, *(scaled_rates / span))[1] - y_fit

    result = least_squares(residuals, start)
    if not result.success:
        raise RuntimeError(f"the exp2 refinement did not converge: {result.message}")
    b, d = result.x / span
    (a, c), _ = exp2_amplitudes(t_fit, y_fit, b, d)
    params = np.array([a, b, c, d])
    curve = exp2_curve(t, *params)
    if np.abs([a, c]).max() > b_settings["exp2_max_amplitude_ratio"] * np.abs(y_fit).max():
        raise Exp2Rejected(
            "amplitude-ratio rule of this notebook (MATLAB's fit has no such rule): the two exponentials "
            "cancel each other (rates almost equal).",
            curve,
        )
    if not (np.isfinite(curve).all() and np.all(curve > 0)):
        raise Exp2Rejected("the exp2 fit is not finite and positive everywhere.", curve)
    return curve, params


def fit_channel_b(
    t: np.ndarray,
    y: np.ndarray,
    b_settings: dict,
    preprocessing: dict,
    fit_mask: np.ndarray,
    exp2_fit: Callable = fit_exp2,
):
    """Step B2: the bleaching curve of one channel by the chosen model, falling back to the bounded fit if exp2 fails.

    The fallback is deliberate and never silent: the model actually used and the reason are returned, and
    the notebook lists every recording that fell back. *exp2_fit* is injectable so the fallback can be tested.

    Returns:
        tuple: The curve, its parameters (``[a, b, c, d]`` for exp2, the five bounded-fit parameters otherwise),
        the model used (``"exp2"`` or ``"bounded"``), the reason for the fallback (empty if there was none), and
        the rejected exp2 curve (None unless the fit converged and was then rejected).
    """
    model = b_settings["bleaching_model"]
    if model not in ("exp2", "bounded"):
        raise ValueError(f"bleaching_model must be 'exp2' or 'bounded', not {model!r}.")
    reason, rejected = "", None
    if model == "exp2":
        try:
            curve, params = exp2_fit(t, y, fit_mask, preprocessing["fit_decimation"], b_settings)
            return curve, params, "exp2", "", None
        except (ValueError, RuntimeError, np.linalg.LinAlgError) as error:
            reason = f"{type(error).__name__}: {error}"
            rejected = getattr(error, "curve", None)
    curve, params, _ = fit_bleaching(t, y, preprocessing, fit_mask)
    return curve, params, "bounded", reason, rejected


def window_rmse(y: np.ndarray, curve: np.ndarray, window: np.ndarray) -> float:
    """Root-mean-square difference between *y* and *curve* over the samples of the boolean *window* (V)."""
    return float(np.sqrt(np.mean((y[window] - curve[window]) ** 2)))


def preprocess_channels_b(
    signal: np.ndarray,
    control: np.ndarray,
    time: np.ndarray,
    fs: float,
    preprocessing: dict,
    b_settings: dict,
    mask_times: np.ndarray,
) -> dict:
    """Steps B1-B4 for one recording.

    Args:
        signal, control (np.ndarray): Baseline-subtracted channels (V), same length, finite.
        time (np.ndarray): pyControl time of each sample (s), NaN outside the sync coverage.
        fs (float): Sampling rate (Hz).
        preprocessing (dict): ``PREPROCESSING`` (low-pass, optional median filter, fallback-fit settings).
        b_settings (dict): ``B_PREPROCESSING``.
        mask_times (np.ndarray): Outcome times (s); the motion regression leaves out ``motion_mask_s`` after each.

    Returns:
        dict: ``dff`` (%), ``corrected`` (V), ``fit_signal``/``fit_control`` (V), ``detrended_signal``/
        ``detrended_control`` (V), ``params_*``, ``model_*``, ``fallback_reason_*`` (per channel),
        ``rmse_fit_*`` (RMSE of the curve used against the low-passed channel inside the fit window, V),
        ``rmse_rejected_exp2_*`` (the same for a rejected exp2 curve, NaN if there was none),
        ``motion_stats`` (slope, intercept, r, r2), ``fraction_masked`` (of the fit window) and
        ``n_regression_samples``.
    """
    if signal.shape != control.shape or not (
        np.isfinite(signal).all() and np.isfinite(control).all()
    ):
        raise ValueError("signal and control must have the same length and be finite.")
    t = np.arange(len(signal)) / fs
    window = fit_window_mask(
        len(signal), fs, b_settings["fit_skip_samples"], b_settings["fit_skip_end_s"]
    )
    masked = post_event_mask(time, mask_times, b_settings["motion_mask_s"])
    regression_mask = window & ~masked
    out = {}
    detrended = {}
    for name, raw in (("signal", signal), ("control", control)):
        x = median_despike(raw, fs, preprocessing["median_filter_s"])
        x = lowpass_zero_phase(x, fs, preprocessing["lowpass_hz"], preprocessing["lowpass_order"])
        fit, params, model, reason, rejected = fit_channel_b(
            t, x, b_settings, preprocessing, window
        )
        detrended[name] = x - fit
        out |= {
            f"fit_{name}": fit,
            f"params_{name}": params,
            f"model_{name}": model,
            f"fallback_reason_{name}": reason,
            f"rmse_fit_{name}": window_rmse(x, fit, window),
            f"rmse_rejected_exp2_{name}": np.nan
            if rejected is None
            else window_rmse(x, rejected, window),
            f"detrended_{name}": detrended[name],
        }
    corrected, _, stats = regress_out_control(
        detrended["signal"], detrended["control"], regression_mask
    )
    return out | {
        "corrected": corrected,
        "dff": delta_f_over_f(corrected, out["fit_signal"]),
        "motion_stats": stats,
        "fraction_masked": float(masked[window].mean()),
        "n_regression_samples": int(regression_mask.sum()),
    }

### Synthetic checks of pipeline B

**Whole trace (B1-B4).** The `exp2` fit must recover a bleaching curve of known form, with or without a constant; a failing fit
must fall back to the bounded model and say why; the fit window and the post-event mask must select exactly the intended
samples; leaving out the samples that carry a signal-only response must bring the motion slope closer to the truth; and the
whole of B1-B4 must recover the movement slope and the transients of the synthetic recording used for pipeline A.

**Event level (B5-B8).** Trials with a known offset and spread in their pre-event window: after B5 the baseline mean must be
zero; after B6 the baseline mean must be zero and the sample SD one; smoothing must give MATLAB's numbers; and a known
triangular response must give the right peak and area.

In [ ]:
def check_pipeline_b_whole_trace_on_synthetic_data(preprocessing: dict) -> pd.DataFrame:
    """Assert that B1-B4 and their helpers recover known answers; return the comparison."""
    rng, fs = np.random.default_rng(5), 130.0
    t = np.arange(int(900 * fs)) / fs
    window = fit_window_mask(len(t), fs, 99, 30.0)  # MATLAB's sample 100 is Python index 99
    assert (
        window.sum() == len(t) - 99 - int(30 * fs)
        and not window[:99].any()
        and window[99]
        and not window[-1]
    )
    rows = []

    exp2_settings = {**B_PREPROCESSING, "bleaching_model": "exp2"}
    two_exp = 1.5 * np.exp(-t / 2000) + 1.0 * np.exp(-t / 200)  # no constant: exactly the exp2 form
    with_constant = 1.2 + 0.8 * np.exp(-t / 400)  # a constant is the rate 0 of exp2
    for label, truth in (("exp2 form", two_exp), ("constant plus one exponential", with_constant)):
        y = truth + 0.002 * rng.standard_normal(len(t))
        curve, _, model, reason, _ = fit_channel_b(t, y, exp2_settings, preprocessing, window)
        error = np.abs(curve / truth - 1).max()
        assert model == "exp2" and reason == "" and error < 0.01, (label, model, reason, error)
        rows.append((f"exp2 fit, {label}: largest relative error", error, "< 0.01"))

    def failing_fit(*args):
        raise RuntimeError("forced failure")

    y = with_constant + 0.002 * rng.standard_normal(len(t))
    curve, _, model, reason, _ = fit_channel_b(
        t, y, exp2_settings, preprocessing, window, exp2_fit=failing_fit
    )
    assert model == "bounded" and "forced failure" in reason
    assert np.abs(curve / with_constant - 1).max() < 0.02
    rows.append(("exp2 failure falls back to the bounded model and reports why", 1.0, "= 1"))

    strict = {
        **exp2_settings,
        "exp2_max_amplitude_ratio": 1e-3,
    }  # every fit now breaks the amplitude rule
    curve, _, model, reason, rejected = fit_channel_b(t, y, strict, preprocessing, window)
    assert model == "bounded" and "amplitude-ratio rule" in reason and "no such rule" in reason, (
        reason
    )
    assert (
        rejected is not None and np.abs(rejected / with_constant - 1).max() < 0.02
    )  # the rejected curve is kept
    rows.append(
        ("amplitude-ratio rejection reports the rule and keeps the rejected curve", 1.0, "= 1")
    )

    time = np.arange(1000) / 10.0
    time[:50] = np.nan  # no time before the first sync pulse
    mask = post_event_mask(time, np.array([20.0, 60.0]), 5.0)
    assert mask.sum() == 100 and not mask[:50].any() and mask[200:250].all() and mask[600:650].all()
    assert not post_event_mask(time, np.array([20.0]), 0.0).any()
    rows.append(("post-event mask: samples masked (2 events x 5 s at 10 Hz)", mask.sum(), "= 100"))

    n = 20_000
    control = rng.standard_normal(n)
    masked = np.zeros(n, dtype=bool)
    masked[5000:8000] = True
    control[masked] += 2.0  # the masked stretch holds an evoked change that the control shares
    signal = 1.25 * control + 0.01 * rng.standard_normal(n)
    signal[masked] += 3.0  # ... plus a signal-only response, which biases the slope if it is fitted
    slope_all = regress_out_control(signal, control)[2]["slope"]
    slope_masked = regress_out_control(signal, control, ~masked)[2]["slope"]
    assert abs(slope_masked - 1.25) < 0.01 < abs(slope_all - 1.25)
    rows.append(("motion slope, mask on / off (true 1.25)", slope_masked, slope_all))

    fs_s = 130.0
    signal, control, truth = synthetic_recording(np.random.default_rng(1), fs_s)
    b_settings = {**B_PREPROCESSING, "motion_mask_s": 0.0}
    out = preprocess_channels_b(
        signal,
        control,
        np.arange(len(signal)) / fs_s,
        fs_s,
        preprocessing,
        b_settings,
        np.array([]),
    )
    slope_error = abs(out["motion_stats"]["slope"] / truth["slope"] - 1)
    expected = lowpass_zero_phase(
        truth["transients"], fs_s, preprocessing["lowpass_hz"], preprocessing["lowpass_order"]
    )
    transient_r = np.corrcoef(out["corrected"], expected)[0, 1]
    assert slope_error < 0.10 and transient_r > 0.95, (slope_error, transient_r)
    assert out["model_signal"] == "exp2" and out["model_control"] == "exp2", out[
        "fallback_reason_signal"
    ]
    rows.append(("B1-B4 on the synthetic recording: slope relative error", slope_error, "< 0.10"))
    rows.append(
        ("B1-B4 on the synthetic recording: corrected vs true transients r", transient_r, "> 0.95")
    )
    return pd.DataFrame(rows, columns=["check", "value", "limit"])


display(check_pipeline_b_whole_trace_on_synthetic_data(PREPROCESSING))


def check_pipeline_b_event_level_on_synthetic_data() -> pd.DataFrame:
    """Assert the event-level normalisation, smoothing and metrics against hand-computed answers."""
    rng = np.random.default_rng(2)
    rel = relative_time_grid((-5.0, 10.0), 130.0)
    offsets = rng.uniform(-3, 3, size=(40, 1))
    spreads = rng.uniform(0.5, 2.0, size=(40, 1))
    trials = offsets + spreads * rng.standard_normal((40, len(rel)))
    settings = ViewSettings((-5.0, 0.0), 1, 1, True)  # 1 point = no smoothing
    out, valid = normalise_trials_b(trials, trials, rel, settings)
    base = interval_mask(rel, (-5.0, 0.0))
    assert valid.all()
    np.testing.assert_allclose(out["dff"][:, base].mean(axis=1), 0, atol=1e-12)
    np.testing.assert_allclose(out["z"][:, base].mean(axis=1), 0, atol=1e-12)
    np.testing.assert_allclose(out["z"][:, base].std(axis=1, ddof=1), 1, atol=1e-12)
    # without the baseline subtraction (the code's default) the ΔF/F is returned as it came
    plain, _ = normalise_trials_b(trials, trials, rel, ViewSettings((-5.0, 0.0), 1, 1, False))
    np.testing.assert_allclose(plain["dff"], trials)
    # the z-score does not depend on the scale of the signal it is taken from
    scaled, _ = normalise_trials_b(trials, 7.0 * trials + 3.0, rel, settings)
    np.testing.assert_allclose(scaled["z"], out["z"], atol=1e-9)
    # a shorter pre-window gives that window's own baseline statistics
    short, _ = normalise_trials_b(trials, trials, rel, ViewSettings((-2.0, 0.0), 1, 1, True))
    np.testing.assert_allclose(
        short["z"][:, interval_mask(rel, (-2.0, 0.0))].mean(axis=1), 0, atol=1e-12
    )
    # MATLAB: movmean([1 2 3 4 5], 4) = [1.5 2 2.5 3.5 4]; movmean(1:5, 3) = [1.5 2 3 4 4.5]
    np.testing.assert_allclose(matlab_moving_mean(np.arange(1.0, 6), 4), [1.5, 2, 2.5, 3.5, 4])
    np.testing.assert_allclose(matlab_moving_mean(np.arange(1.0, 6), 3), [1.5, 2, 3, 4, 4.5])
    np.testing.assert_allclose(matlab_moving_mean(trials, 1), trials)
    # smoothing and averaging over trials commute
    np.testing.assert_allclose(
        matlab_moving_mean(trials, 80).mean(axis=0), matlab_moving_mean(trials.mean(axis=0), 80)
    )
    # a zero-variance baseline is reported invalid instead of dividing by zero
    flat = np.zeros((1, len(rel)))
    assert not trial_zscore(flat, rel, (-5.0, 0.0))[1][0]
    # a pre-window that is not before the outcome is refused
    try:
        ViewSettings((-5.0, 1.0), 1, 1, True)
    except ValueError:
        pass
    else:
        raise AssertionError("a pre-window reaching past the outcome must be refused")
    # windows are padded and then cropped, so the displayed edge samples are smoothed with a full window
    fs, crop = 130.0, (-2.0, 3.0)
    long_trace = rng.standard_normal(int(60 * fs))
    clock = np.arange(len(long_trace)) / fs
    rel_pad, padded, _ = extract_event_windows(
        clock, long_trace, np.array([30.0]), padded_window(crop, 80, fs), fs
    )
    shown = matlab_moving_mean(padded, 80)[:, crop_mask(rel_pad, crop, fs)]
    centre = round(30.0 * fs)
    reference = matlab_moving_mean(long_trace, 80)[
        centre + round(crop[0] * fs) : centre + round(crop[1] * fs) + 1
    ]
    np.testing.assert_allclose(shown[0], reference)
    bare = matlab_moving_mean(
        extract_event_windows(clock, long_trace, np.array([30.0]), crop, fs)[1], 80
    )
    assert not np.allclose(
        bare[0, :5], reference[:5]
    )  # without padding the first samples are smoothed with a shorter window
    # triangle: 0 at t=0 rising to 2 at t=1 and back to 0 at t=2: peak 2 at 1 s, area 2 z*s
    triangle = np.clip(1 - np.abs(rel - 1), 0, None) * 2
    metrics = peak_and_auc(rel, triangle, (0.0, 10.0))
    assert np.isclose(metrics["peak"], 2) and np.isclose(metrics["peak_time_s"], 1)
    assert np.isclose(metrics["auc"], 2, atol=0.02)
    return pd.DataFrame(
        {
            "check": [
                "B5/B6 baselines, scale-free z, short pre-window, MATLAB movmean, padded smoothing, commutation, invalid trials, peak and AUC"
            ],
            "result": ["passed"],
        }
    )


display(check_pipeline_b_event_level_on_synthetic_data())

### Pipeline B on every recording

Steps B1-B4 are cached per recording like pipeline A, keyed on the file, `PREPROCESSING` and `B_PREPROCESSING`. The motion
regression leaves out `motion_mask_s` after every outcome of the recording (all phases). Three tables follow: which model was
actually used per channel (**any exp2 failure is listed here**, with its cause; a failure caused by this notebook's amplitude-ratio rule is a rule MATLAB's `fit` does not have, and the RMSE of the rejected curve inside the fit window is shown against that of the bounded fit), how much of the session a post-outcome mask would remove and what
it would do to the motion slope, and how pipeline B's whole-trace result differs from pipeline A's.

In [ ]:
@memory.cache(ignore=["recording"])
def preprocess_recording_b_cached(
    key: tuple, preprocessing: dict, b_settings: dict, mask_times: np.ndarray, recording
) -> dict:
    """Pipeline B (steps B1-B4) for one recording; *key* and the settings identify it for the cache."""
    return preprocess_channels_b(
        recording.signal,
        recording.control,
        recording.time,
        recording.fs,
        preprocessing,
        b_settings,
        mask_times,
    )


def session_traces(
    processed_a: dict, processed_b: dict, z_from: str, channel: str = "signal"
) -> dict[str, np.ndarray]:
    """The whole-session traces that event windows are cut from, for the signal or the control channel.

    *processed_a* is ``PROCESSED[rid]["filtered"]``. ``B_z_source`` is the trace pipeline B's trial z-score is
    taken from: ``"corrected"`` or ``"dff"``. For the **signal**, "corrected" is the motion-corrected signal.
    The **control** has nothing to be regressed on, so it is processed in parallel without motion correction:
    its own bleaching fit is subtracted ("corrected" is then the detrended control), ΔF/F is relative to its own
    fit, and pipeline A's z-score is that of the detrended control over the session.
    """
    if z_from not in ("corrected", "dff"):
        raise ValueError(f"z_from must be 'corrected' or 'dff', not {z_from!r}.")
    if channel == "signal":
        a_dff, a_z = processed_a["dff"], processed_a["zscore"]
        b_dff, b_corrected = processed_b["dff"], processed_b["corrected"]
    elif channel == "control":
        a_dff, a_z = processed_a["control_dff"], session_zscore(processed_a["detrended_control"])
        b_corrected = processed_b["detrended_control"]
        b_dff = delta_f_over_f(b_corrected, processed_b["fit_control"])
    else:
        raise ValueError(f"channel must be 'signal' or 'control', not {channel!r}.")
    return {
        "A_dff": a_dff,
        "A_z": a_z,
        "B_dff": b_dff,
        "B_z_source": b_corrected if z_from == "corrected" else b_dff,
    }


def check_session_traces() -> None:
    """The control traces are the control's own detrended trace and ΔF/F, never the motion-corrected signal."""
    rng = np.random.default_rng(6)
    fit = 2.0 + np.zeros(50)
    detrended = rng.standard_normal(50)
    a = {
        "dff": rng.standard_normal(50),
        "zscore": rng.standard_normal(50),
        "control_dff": 100 * detrended / fit,
        "detrended_control": detrended,
    }
    b = {
        "dff": rng.standard_normal(50),
        "corrected": rng.standard_normal(50),
        "detrended_control": detrended,
        "fit_control": fit,
    }
    control = session_traces(a, b, "corrected", "control")
    np.testing.assert_allclose(control["B_dff"], a["control_dff"])
    np.testing.assert_allclose(control["B_z_source"], detrended)
    assert np.isclose(control["A_z"].mean(), 0) and np.isclose(control["A_z"].std(), 1)
    assert session_traces(a, b, "corrected", "signal")["B_z_source"] is b["corrected"]
    from_dff = session_traces(a, b, "dff", "control")
    np.testing.assert_allclose(from_dff["B_z_source"], from_dff["B_dff"])


check_session_traces()


_start = time.time()
PROCESSED_B = {
    rid: preprocess_recording_b_cached(
        file_key(rec.ppd_path),
        PREPROCESSING,
        B_PREPROCESSING,
        ALL_EVENTS[rid]["outcome_time"].to_numpy(),
        rec,
    )
    for rid, rec in RECORDINGS.items()
}
TRACES, TRACES_CONTROL = (
    {
        rid: session_traces(
            PROCESSED[rid]["filtered"], PROCESSED_B[rid], B_EVENT_LEVEL["z_from"], channel
        )
        for rid in RECORDINGS
    }
    for channel in ("signal", "control")
)
print(f"Pipeline B on {len(PROCESSED_B)} recordings: {time.time() - _start:.0f} s.")


def fit_model_table(recordings: dict, processed_b: dict) -> pd.DataFrame:
    """Per recording: bleaching model used for each channel, the reason for any fallback, and the motion fit."""
    return pd.DataFrame(
        [
            {
                "recording": rid,
                "mouse": rec.subject,
                "model_signal": processed_b[rid]["model_signal"],
                "model_control": processed_b[rid]["model_control"],
                "fallback_reason_signal": processed_b[rid]["fallback_reason_signal"],
                "fallback_reason_control": processed_b[rid]["fallback_reason_control"],
                "rmse_fit_used_signal": processed_b[rid]["rmse_fit_signal"],
                "rmse_rejected_exp2_signal": processed_b[rid]["rmse_rejected_exp2_signal"],
                "rmse_fit_used_control": processed_b[rid]["rmse_fit_control"],
                "rmse_rejected_exp2_control": processed_b[rid]["rmse_rejected_exp2_control"],
                "motion_slope": processed_b[rid]["motion_stats"]["slope"],
                "fraction_masked": processed_b[rid]["fraction_masked"],
            }
            for rid, rec in recordings.items()
        ]
    )


def describe_fallbacks(fit_models: pd.DataFrame) -> list[str]:
    """One plain sentence per channel whose exp2 fit was replaced, with the fit error inside the fit window (V).

    The rejected curve's RMSE is only available when the fit converged and the notebook then rejected it.
    """
    lines = []
    for _, row in fit_models.iterrows():
        for channel in ("signal", "control"):
            reason = row[f"fallback_reason_{channel}"]
            if not reason:
                continue
            rejected = row[f"rmse_rejected_exp2_{channel}"]
            used = row[f"rmse_fit_used_{channel}"]
            compare = (
                f"inside the fit window the rejected exp2 curve has RMSE {rejected:.2e} V against {used:.2e} V for the "
                f"bounded fit that replaced it ({'the rejected curve fits no worse' if rejected <= used else 'the rejected curve fits worse'})"
                if np.isfinite(rejected)
                else "there is no rejected curve (the refinement did not converge), so no comparison"
            )
            lines.append(f"{row['recording']} {channel}: {reason} {compare}.")
    return lines


fit_models = fit_model_table(RECORDINGS, PROCESSED_B)
show_table(fit_models, "pipeline_b_fit_models")
fell_back = fit_models[
    (fit_models["fallback_reason_signal"] != "") | (fit_models["fallback_reason_control"] != "")
]
if B_PREPROCESSING["bleaching_model"] == "exp2":
    n_channels = 2 * len(fit_models)
    n_fallback = int(
        (fit_models["fallback_reason_signal"] != "").sum()
        + (fit_models["fallback_reason_control"] != "").sum()
    )
    print(
        f"exp2 fell back to the bounded model for {n_fallback} of {n_channels} channel fits "
        f"({len(fell_back)} of {len(fit_models)} recordings)."
    )
    if len(fell_back):
        display(fell_back)
        print("\n".join(describe_fallbacks(fit_models)))

In [ ]:
def motion_mask_row(
    processed: dict,
    time: np.ndarray,
    fs: float,
    event_times: np.ndarray,
    b_settings: dict,
    durations,
) -> dict:
    """Fraction of the fit window a post-outcome mask would remove, and the motion slope without those samples."""
    window = fit_window_mask(
        len(time), fs, b_settings["fit_skip_samples"], b_settings["fit_skip_end_s"]
    )
    row = {}
    for duration in durations:
        masked = post_event_mask(time, event_times, duration)
        kept = window & ~masked
        row[f"masked_{duration:g}s"] = float(masked[window].mean())
        slope = regress_out_control(
            processed["detrended_signal"], processed["detrended_control"], kept
        )[2]
        row[f"slope_{duration:g}s"] = slope["slope"]
    return row


MASK_DURATIONS_S = (0.0, 2.0, 5.0, 10.0, 20.0)
motion_mask_table = pd.DataFrame(
    [
        {
            "recording": rid,
            "mouse": rec.subject,
            **motion_mask_row(
                PROCESSED_B[rid],
                rec.time,
                rec.fs,
                ALL_EVENTS[rid]["outcome_time"].to_numpy(),
                B_PREPROCESSING,
                MASK_DURATIONS_S,
            ),
        }
        for rid, rec in RECORDINGS.items()
    ]
)
show_table(motion_mask_table, "pipeline_b_motion_mask")
print(
    "Median fraction of the fit window removed by a mask after every outcome: "
    + ", ".join(
        f"{d:g} s: {motion_mask_table[f'masked_{d:g}s'].median():.2f}" for d in MASK_DURATIONS_S
    )
    + f". Active setting: motion_mask_s = {B_PREPROCESSING['motion_mask_s']:g}."
)


def pipeline_trace_comparison(
    recordings: dict, processed_a: dict, processed_b: dict, b_settings: dict
) -> pd.DataFrame:
    """Per recording: how pipeline B's whole-trace result differs from pipeline A's.

    Statistics are given over the whole recording and over the fit window of pipeline B (the stretch the exp2 fit
    was trained on; outside it the unconstrained curve is an extrapolation).
    """
    rows = []
    for rid, rec in recordings.items():
        a, b = processed_a[rid]["filtered"], processed_b[rid]
        window = fit_window_mask(
            len(rec.signal), rec.fs, b_settings["fit_skip_samples"], b_settings["fit_skip_end_s"]
        )
        fit_diff = np.abs(b["fit_signal"] / a["fit_signal"] - 1)
        rows.append(
            {
                "recording": rid,
                "mouse": rec.subject,
                "r_corrected_a_b": np.corrcoef(a["corrected"], b["corrected"])[0, 1],
                "r_in_fit_window": np.corrcoef(a["corrected"][window], b["corrected"][window])[
                    0, 1
                ],
                "sd_ratio_b_over_a": b["corrected"].std() / a["corrected"].std(),
                "sd_ratio_in_fit_window": b["corrected"][window].std()
                / a["corrected"][window].std(),
                "slope_a": a["motion_stats"]["slope"],
                "slope_b": b["motion_stats"]["slope"],
                "median_abs_fit_diff_pct": 100 * np.median(fit_diff[window]),
                "max_abs_fit_diff_in_window_pct": 100 * fit_diff[window].max(),
                "max_abs_fit_diff_outside_pct": 100 * fit_diff[~window].max(),
            }
        )
    return pd.DataFrame(rows)


trace_comparison = pipeline_trace_comparison(RECORDINGS, PROCESSED, PROCESSED_B, B_PREPROCESSING)
show_table(trace_comparison, "pipeline_comparison_traces")

In [ ]:
fig = stage_figure(
    EXAMPLE_TIME,
    [
        (
            "Signal (V)",
            [
                ("Low-passed", EXAMPLE_PROC["denoised_signal"], {"color": "0.6"}),
                ("A: bounded fit", EXAMPLE_PROC["fit_signal"], {"color": "k"}),
                (
                    "B: exp2 fit",
                    PROCESSED_B[EXAMPLE]["fit_signal"],
                    {"color": "tab:red", "ls": "--"},
                ),
            ],
        ),
        (
            "Corrected (V)",
            [
                ("A", EXAMPLE_PROC["corrected"], {"color": "k"}),
                ("B", PROCESSED_B[EXAMPLE]["corrected"], {"color": "tab:red", "alpha": 0.7}),
            ],
        ),
    ],
    EXAMPLE_ZOOM,
    f"B2-B3: Bleaching Fit and Motion-Corrected Signal, Pipelines A And B ({EXAMPLE})",
)
save_figure(fig, "example_B_whole_trace")
plt.show()

### Pipeline B event-level steps on the example recording

Steps B1-B4 are the traces just compared. The figure below shows the event-level steps B5-B8 on the example recording's
**rewarded** trials: the left column follows six trials chosen evenly through the session, the right column is the mean ± SEM
over all rewarded trials of the recording at the same step. B5 is shown with the pre-window mean subtracted although it is off by
default (`subtract_dff_baseline`), so its effect is visible. **Look for:** B5 puts every trial's pre-event (shaded) level at
zero; B6 puts it at zero *and* gives it unit spread, so noisy and quiet trials are on one scale; B7 smooths, and because the
moving mean is centred, the smoothed response begins rising about half the window *before* zero.

In [ ]:
def plot_trial_lines(ax, rel_time, trials, shade=None, ylabel="") -> None:
    """Thin lines for a few trials, with an optional shaded interval (the baseline)."""
    for row, colour in zip(trials, plt.cm.viridis(np.linspace(0.1, 0.9, len(trials))), strict=True):
        ax.plot(rel_time, row, color=colour, lw=0.8)
    if shade:
        ax.axvspan(*shade, color="0.5", alpha=0.15)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.set_ylabel(ylabel)


def plot_mean_sem(ax, rel_time, trials, colour="tab:green", shade=None, ylabel="") -> None:
    """Mean ± SEM over trials with the zero line."""
    mean, sem = mean_and_sem(trials)
    ax.plot(rel_time, mean, color=colour, lw=1.4, label=f"n = {len(trials)}")
    ax.fill_between(rel_time, mean - sem, mean + sem, color=colour, alpha=0.25, lw=0)
    if shade:
        ax.axvspan(*shade, color="0.5", alpha=0.15)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.set_ylabel(ylabel)
    ax.legend(loc="upper right", frameon=False)


_events = EVENTS[EXAMPLE]
rewarded_times_ex = _events.loc[_events["condition"] == "rewarded", "outcome_time"].to_numpy()
rel_pad, windows_ex, _ = trace_windows(
    EXAMPLE_REC,
    TRACES[EXAMPLE],
    rewarded_times_ex,
    padded_window(EXTRACTION_WINDOW, DEFAULT_SMOOTHING, EXAMPLE_REC.fs),
)
unsmoothed, valid_ex = normalise_trials_b(
    windows_ex["B_dff"],
    windows_ex["B_z_source"],
    rel_pad,
    ViewSettings(DEFAULT_PRE_WINDOW, 1, 1, True),
)
smoothed, _ = normalise_trials_b(
    windows_ex["B_dff"],
    windows_ex["B_z_source"],
    rel_pad,
    ViewSettings(DEFAULT_PRE_WINDOW, DEFAULT_SMOOTHING, DEFAULT_SMOOTHING, True),
)
shown = crop_mask(
    rel_pad, EXTRACTION_WINDOW, EXAMPLE_REC.fs
)  # the padding only serves the smoothing
rel_ex = rel_pad[shown]
dff_ex = windows_ex["B_dff"][valid_ex][:, shown]
unsmoothed = {name: trials[:, shown] for name, trials in unsmoothed.items()}
smoothed = {name: trials[:, shown] for name, trials in smoothed.items()}
pick = np.linspace(0, valid_ex.sum() - 1, 6).round().astype(int)
steps = [
    ("B4: ΔF/F (Exp2 Fit)", "ΔF/F (%)", dff_ex, DEFAULT_PRE_WINDOW),
    (
        "B5: Pre-Window Mean Subtracted (Optional)",
        "Change in ΔF/F (%)",
        unsmoothed["dff"],
        DEFAULT_PRE_WINDOW,
    ),
    (
        "B6: Trial Z-Score, Unsmoothed",
        "Z-Score (Pre-Event SD)",
        unsmoothed["z"],
        DEFAULT_PRE_WINDOW,
    ),
    (
        f"B7: Trial Z-Score, {DEFAULT_SMOOTHING}-Point Mean",
        "Z-Score (Pre-Event SD)",
        smoothed["z"],
        DEFAULT_PRE_WINDOW,
    ),
]
fig, axes = plt.subplots(len(steps), 2, figsize=(12, 10), sharex=True, constrained_layout=True)
for (label, ylabel, trials, shade), (ax_trials, ax_mean) in zip(steps, axes, strict=True):
    plot_trial_lines(ax_trials, rel_ex, trials[pick], shade, ylabel)
    plot_mean_sem(ax_mean, rel_ex, trials, "tab:green", shade, ylabel)
    ax_trials.set_title(f"{label}: Six Example Trials", loc="left", fontsize=9)
    ax_mean.set_title(f"{label}: Mean ± SEM, All Rewarded", loc="left", fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel("Time From Reward Onset (s)")
    ax.set_xlim(EVENT_WINDOW)
fig.suptitle(f"Pipeline B Event-Level Steps ({EXAMPLE})", fontsize=12)
save_figure(fig, "example_B_event_level_steps")
plt.show()

## 5. Written comparison of the two pipelines

**What they share.** Both low-pass the signal at 10 Hz with a zero-phase order-2 Butterworth filter (the same function), fit a
double exponential to each channel and subtract it, regress the control onto the signal and subtract the fit, and divide by the
bleaching fit to get ΔF/F (%). The primer recommends this order (filter, bleaching, movement, normalisation); the paper and the
authors' code do the same steps (section 4).

**Where they differ.** Section 4 lists the paper's and the code's choices; the table sets them against pipeline A.

| | Pipeline A (primer) | Pipeline B (paper and authors' code) |
|---|---|---|
| Bleaching fit | bounded double exponential with a constant, every sample | `exp2` (no constant, no bounds), MATLAB's sample 100 (Python index 99) to 30 s before the end; bounded fit only if exp2 fails (reported in section 4, including that the one failure comes from this notebook's own amplitude-ratio rule, which MATLAB lacks) |
| Motion regression | all samples | fit-window samples (optionally minus a post-outcome mask, off by default) |
| Normalisation reference | whole session | each trial's own pre-event window |
| ΔF/F offset | none | none by default (the code); the paper's tail-lift option subtracts the pre-window mean |
| z-score | session mean and **session SD** of the corrected signal (`session_zscore`) | trial's pre-event mean and **pre-event SD** of the motion-corrected signal (`trial_zscore`) |
| Smoothing | none beyond the 10 Hz low-pass (`DEFAULT_SMOOTHING_A`) | 80-point centred moving mean after normalisation (`DEFAULT_SMOOTHING`) |
| Summary | mean ± SEM of the trace | per mouse, then across mice (7d); peak and area under the curve |
| Needs events? | no: the whole trace is normalised once | yes: normalisation is part of the event analysis |

How different the whole-trace results are (correlation of the two corrected signals, ratio of their SDs, the two motion slopes, the
difference between the two bleaching fits) is the table `pipeline_comparison_traces` in section 4. The tables below measure the
event-level consequences.

**Consequences, each tied to the code here.**

- *Trial z-scores inflate when the pre-event SD is small.* `trial_zscore` divides by that trial's own pre-event SD, so a
  trial with a quiet baseline gets a large z for the same change; with a short window the SD estimate is itself noisy from
  trial to trial. The session z-score uses one SD for all trials. The table below measures how far apart the two denominators are
  in these recordings, for the default pre window.
- *Session z keeps differences between trials in their pre-event level.* Pipeline A does not subtract a baseline, so a
  trial that begins above the session mean stays above it; this slow variability adds to the spread of the mean across
  trials. Pipeline B's z removes it by construction (and with it any slow, real change that happens to cross the baseline
  window); its ΔF/F removes it only if `subtract_dff_baseline` is on. The table also gives the size of this spread in session-SD units.
- *ΔF/F denominators include autofluorescence.* `delta_f_over_f` divides by the fitted *total* fluorescence of the signal
  channel after LED-off subtraction, which contains the fibre's and tissue's autofluorescence as well as the
  indicator. The relative change of the indicator alone would be larger. This is true of both pipelines, so ΔF/F
  values are comparable between them but not directly with literature values from other set-ups.
- *The moving-mean smoothing spreads a response in time.* The moving mean is centred, so each smoothed sample averages
  points up to half the window ahead: a step at zero begins to rise before zero (computed below). It also lowers
  peaks and slows apparent onsets. Pipeline A's traces are not affected beyond the 10 Hz low-pass unless `DEFAULT_SMOOTHING_A` is raised.
- *Peak and area are defined on different scales.* Pipeline B's peak and AUC are in trial pre-event-SD units of the smoothed
  traces; pipeline A's are in session-SD units. They are not interchangeable.
- *A longer baseline is not automatically better here.* The paper's 15-s z baseline is meant for a cue followed by a long quiet
  stretch; our outcomes come about every 10 s, so a long window contains earlier outcomes and discards trials near the start of a
  recording (counted in 7c).

**Still not established:** the sampling rate behind the paper's 80 points (section 4), whether the 405-nm channel is isosbestic for
nLightG2 and the channel wiring (sections 1 and 7b), and the paper's exact treatment of trials for which the baseline window overlaps
an earlier event (the code takes the window as it comes).

In [ ]:
def baseline_comparison(recording, events, dff, window, baseline) -> dict:
    """Session SD against pre-event SD and between-trial baseline spread for one recording's ΔF/F."""
    _, trials, _ = extract_event_windows(
        recording.time, dff, events["outcome_time"].to_numpy(), window, recording.fs
    )
    rel_time = relative_time_grid(window, recording.fs)
    mask = interval_mask(rel_time, baseline)
    pre_sd = trials[:, mask].std(axis=1, ddof=1)
    pre_mean = trials[:, mask].mean(axis=1)
    session_sd = dff.std()
    return {
        "n_trials": len(trials),
        "session_sd_pct": session_sd,
        "median_pre_event_sd_pct": np.median(pre_sd),
        "median_session_sd_over_pre_sd": np.median(session_sd / pre_sd),
        "pre_sd_below_half_session_sd": np.mean(pre_sd < 0.5 * session_sd),
        "trial_to_trial_sd_of_pre_mean_over_session_sd": (
            pre_mean.std(ddof=1) / session_sd if len(pre_mean) > 1 else np.nan
        ),
    }


rows = []
for rid, rec in RECORDINGS.items():
    stats = baseline_comparison(
        rec, EVENTS[rid], PROCESSED[rid]["filtered"]["dff"], EXTRACTION_WINDOW, DEFAULT_PRE_WINDOW
    )
    rows.append({"recording": rid, **stats})
baseline_spread = pd.DataFrame(rows)
show_table(baseline_spread, "pipeline_comparison_baselines")

step = np.where(relative_time_grid((-5.0, 5.0), 130.0) >= 0, 1.0, 0.0)
smoothed_step = matlab_moving_mean(step, DEFAULT_SMOOTHING)
grid = relative_time_grid((-5.0, 5.0), 130.0)
onset = grid[np.argmax(smoothed_step > 0.01)]
print(
    f"A unit step at 0 s, smoothed with {DEFAULT_SMOOTHING} points at 130 Hz, first exceeds 1% of its height at "
    f"{onset:.2f} s and reaches half height at {grid[np.argmax(smoothed_step >= 0.5)]:.2f} s."
)

## 6. Quality control suite

Each QC function takes the recordings (or their processed traces) and returns a **tidy table with one row per
recording**. Every table is printed and saved as CSV, and carries a `flags` column listing which thresholds
(`QC_THRESHOLDS` in the configuration cell) the recording crosses. **Flags are for you to look at; nothing is
excluded.**

The tables follow the processing order:

| Table | Step | Question |
|---|---|---|
| 6.1 acquisition | A1 and sync | Was the recording healthy: light levels, clipping, dropouts, sync coverage? |
| 6.2 denoising | A3 | How much high-frequency noise was there and how much did the low-pass remove? |
| 6.3 bleaching | A4 | How much did each channel decay, and is the fit sensible? |
| 6.4 movement | A5 | How much of the signal was shared with the control, and are there connection jumps? |
| 6.5 SNR | final traces | How far do slow fluctuations sit above the noise floor, and is it specific to the sensor? |
| 8 (after section 7) | final traces | How large and how consistent is the evoked response (d′)? Per-mouse ranking. |

Tables 6.1-6.4 and the spectral SNR describe pipeline A's processing of the recording (pipeline B shares its low-pass and has its own
bleaching fit and motion regression; section 4 compares them). Evoked SNR depends on how events are normalised, so it is computed
for each pipeline's final traces; it needs the outcome-aligned windows of section 7 and is therefore finished in **section 8**,
which also holds the per-mouse ranking.

### How the SNR measures are defined

- **Spectral SNR (primary, event-independent).** The movement-corrected ΔF/F *before* the low-pass is split into the
  power spectral density (Welch, 30-s segments). SNR (dB) = 10 log10 of the mean PSD at 0.1-3 Hz (where the sensor
  fluctuations of interest live) over the mean PSD at 15-50 Hz (a noise floor, where nothing physiological is expected). It says how far the
  slow fluctuations sit above the noise floor. The same number for the **control** channel (its own ΔF/F) gives
  `specificity_db` = **uncorrected sensor SNR minus control SNR**, both ΔF/F of their own bleaching fit and *before* the movement regression
  (comparing the corrected sensor with the raw control would penalise the sensor for the correction). About **0 dB means the slow
  fluctuations are equally present in the control channel**, i.e. whatever drives the sensor's slow variance also drives the
  control (movement, haemodynamics, or a control that is not isosbestic); positive means the sensor carries slow power the control
  lacks, which is what a sensor responding to norepinephrine should show.
- **Robust noise floor** `sigma_noise_pct`: 1.4826 x the median absolute deviation of the >15 Hz component of that
  ΔF/F (percent). It is the noise outside the signal band, robust to the few large events. **Transient SNR:** transients are
  peaks of the 10 Hz low-passed ΔF/F whose prominence exceeds k = 5 robust SDs of that trace; we report them per minute and
  their median prominence divided by `sigma_noise_pct`.
- **Evoked SNR (section 8).** For each rewarded trial, response = mean z over 0-2 s minus mean z over −2-0 s. The
  mean response is summarised by d′ = mean / SD of the response across trials (an effect size that does not depend on the number of trials;
  **used for the ranking**).

In [ ]:
def run_lengths(mask: np.ndarray) -> np.ndarray:
    """Lengths of the runs of consecutive True values in *mask*."""
    padded = np.concatenate([[False], np.asarray(mask, dtype=bool), [False]])
    edges = np.flatnonzero(padded[1:] != padded[:-1])
    return edges[1::2] - edges[::2]


def robust_sd(x: np.ndarray) -> float:
    """1.4826 x median absolute deviation: the standard deviation of a normal distribution, robust to outliers."""
    return float(1.4826 * np.median(np.abs(x - np.median(x))))


def attach_flags(table: pd.DataFrame, rules: dict[str, pd.Series]) -> pd.DataFrame:
    """Add a ``flags`` column listing the names of the rules each row breaks (comparisons with NaN are False)."""
    broken = pd.DataFrame(rules, index=table.index).fillna(False).astype(bool)
    out = table.copy()
    out["flags"] = broken.apply(lambda row: ", ".join(row.index[row.to_numpy()]), axis=1)
    return out


def per_recording(
    recordings: dict, row_function: Callable, processed: dict | None = None, **settings
) -> pd.DataFrame:
    """One row per recording: ``row_function(rec, processed[rid], **settings)`` returns a dict of columns.

    *processed* (optional) maps recording id to that recording's processed traces; without it the
    row function is called as ``row_function(rec, **settings)``. Nothing is read from the notebook's globals.
    """
    rows = []
    for rid, rec in recordings.items():
        extra = () if processed is None else (processed[rid],)
        rows.append(
            {
                "recording": rid,
                "mouse": rec.subject,
                "session": rec.session,
                **row_function(rec, *extra, **settings),
            }
        )
    return pd.DataFrame(rows)

### 6.1 Acquisition (step A1 and sync)

Light levels (median and 1st/99th percentile of the baseline-subtracted voltage per channel), the LED-off baseline level,
the fraction of samples where the ADC clipped, runs of identical samples longer than 0.5 s (a frozen
recording), the sync match and the fraction of samples outside the sync coverage. The LED-off baseline can contain brief
upward spikes (counted as "baseline spikes per minute": runs above the baseline median plus `QC_BASELINE_SPIKE_SD` = 6 robust SDs), which subtract as
small downward blips from the LED-on signal. **Look for:** no clipping, no flat runs, baselines of a few tens of mV,
`fraction_matched` near 1, and little of the recording outside the sync coverage.

In [ ]:
def baseline_spikes_per_min(baseline: np.ndarray, fs: float, spike_sd: float) -> float:
    """Brief upward excursions of the LED-off baseline per minute (runs above median + *spike_sd* robust SD)."""
    above = baseline > np.median(baseline) + spike_sd * robust_sd(baseline)
    return len(run_lengths(above)) / (len(baseline) / fs / 60)


def acquisition_row(rec, flatline_s: float, spike_sd: float) -> dict:
    """The acquisition QC columns of one recording; runs of identical samples over *flatline_s* count as dropouts."""
    flat = [
        run_lengths(np.diff(x) == 0) + 1  # n identical consecutive diffs = n + 1 identical samples
        for x in (rec.signal_led_on, rec.control_led_on)
    ]
    flat_s = np.concatenate(flat) / rec.fs
    quality = rec.sync_quality
    row = {
        "duration_min": len(rec.signal) / rec.fs / 60,
        "led_current_ma_1": rec.led_current[0],
        "led_current_ma_2": rec.led_current[1],
    }
    for name, channel, baseline, clipping in (
        ("signal", rec.signal, rec.signal_baseline, rec.signal_clipping),
        ("control", rec.control, rec.control_baseline, rec.control_clipping),
    ):
        p01, median, p99 = np.percentile(channel, [1, 50, 99])
        row |= {
            f"{name}_median_v": median,
            f"{name}_p01_v": p01,
            f"{name}_p99_v": p99,
            f"{name}_baseline_v": np.median(baseline),
            f"{name}_baseline_spikes_per_min": baseline_spikes_per_min(baseline, rec.fs, spike_sd),
            f"{name}_clipping_fraction": clipping.mean(),
        }
    return row | {
        "flatline_runs": int((flat_s > flatline_s).sum()),
        "longest_flat_s": float(flat_s.max()),
        "sync_fraction_matched": quality["fraction_matched"],
        "sync_max_discrepancy_samples": quality["max_abs_interval_discrepancy_ms"] * rec.fs / 1000,
        "fraction_outside_sync": rec.n_samples_outside_sync / len(rec.signal),
    }


def acquisition_qc(
    recordings: dict, thresholds: dict, flatline_s: float, spike_sd: float
) -> pd.DataFrame:
    """Table 6.1 for every recording, with flags."""
    table = per_recording(recordings, acquisition_row, flatline_s=flatline_s, spike_sd=spike_sd)
    clipping = table[["signal_clipping_fraction", "control_clipping_fraction"]].max(axis=1)
    baseline = table[["signal_baseline_v", "control_baseline_v"]].max(axis=1)
    return attach_flags(
        table,
        {
            "short": table["duration_min"] < thresholds["min_duration_min"],
            "high_baseline": baseline > thresholds["max_baseline_v"],
            "clipping": clipping > thresholds["max_clipping_fraction"],
            "flatline": table["flatline_runs"] > thresholds["max_flatline_runs"],
            "sync_fraction": table["sync_fraction_matched"] < thresholds["min_sync_fraction"],
            "sync_discrepancy": table["sync_max_discrepancy_samples"]
            > thresholds["max_sync_discrepancy_samples"],
            "outside_sync": table["fraction_outside_sync"]
            > thresholds["max_outside_sync_fraction"],
        },
    )

### 6.2 Denoising (step A3)

High-frequency noise is the standard deviation of the >15 Hz component (a zero-phase high-pass at 15 Hz), measured before and
after the low-pass, in millivolts and as a percentage of the median signal level. **Look for:** `noise_after` far below
`noise_before` (the filter is doing its job); a `noise_before_pct` of a fraction of a percent. The noise is white-ish, so a recording with
much more noise than the others has a poorer light path or detector gain.

In [ ]:
def hf_noise_sd(x: np.ndarray, fs: float, cutoff_hz: float) -> float:
    """Standard deviation of the part of *x* above *cutoff_hz*."""
    return float(highpass_zero_phase(x, fs, cutoff_hz).std())


def denoising_row(rec, processed_traces: dict, cutoff_hz: float) -> dict:
    """Denoising QC columns of one recording; *processed_traces* is its ``PROCESSED`` entry."""
    processed = processed_traces["filtered"]
    row = {}
    for name, raw in (("signal", rec.signal), ("control", rec.control)):
        before = hf_noise_sd(raw, rec.fs, cutoff_hz)
        after = hf_noise_sd(processed[f"denoised_{name}"], rec.fs, cutoff_hz)
        row |= {
            f"{name}_noise_before_mv": 1000 * before,
            f"{name}_noise_after_mv": 1000 * after,
            f"{name}_noise_removed_pct": 100 * (1 - after / before),
            f"{name}_noise_before_pct": 100 * before / np.median(raw),
        }
    return row


def denoising_qc(
    recordings: dict, processed: dict, thresholds: dict, cutoff_hz: float
) -> pd.DataFrame:
    """Table 6.2 for every recording, with flags."""
    table = per_recording(recordings, denoising_row, processed, cutoff_hz=cutoff_hz)
    relative = table[["signal_noise_before_pct", "control_noise_before_pct"]].max(axis=1)
    return attach_flags(table, {"noisy": relative > thresholds["max_hf_noise_pct"]})

### 6.3 Bleaching (step A4)

The drop of each fitted curve from the first to the last sample (%), the root-mean-square error between the low-passed
channel and its fit (mV; this contains all real fluctuations, so it is a scale, not a goodness-of-fit test), and which of
the five fit parameters sit on a bound (within 0.1% of the range). A parameter on a bound is common in a
20-minute recording, where the decay is too slow to identify its time constants, and it does not make the curve wrong; many
parameters on bounds mean the *parameters* are not interpretable. **Look for:** drops of a few to some tens of percent, and a
control that decays at a similar rate to the signal.

In [ ]:
def params_at_bounds(params: np.ndarray, bounds: tuple, tolerance: float = 1e-3) -> list[str]:
    """Names of the fit parameters within *tolerance* of the range of a bound."""
    lower, upper = bounds
    span = upper - lower
    on_bound = (np.abs(params - lower) <= tolerance * span) | (
        np.abs(params - upper) <= tolerance * span
    )
    return [name for name, hit in zip(PARAMETER_NAMES, on_bound, strict=True) if hit]


def bleaching_row(rec, processed_traces: dict) -> dict:
    """Bleaching QC columns of one recording; *processed_traces* is its ``PROCESSED`` entry."""
    processed = processed_traces["filtered"]
    row = {}
    for name in ("signal", "control"):
        fit = processed[f"fit_{name}"]
        at_bounds = params_at_bounds(processed[f"params_{name}"], processed[f"bounds_{name}"])
        row |= {
            f"{name}_drop_pct": 100 * (fit[0] - fit[-1]) / fit[0],
            f"{name}_fit_rmse_mv": 1000
            * float(np.sqrt(np.mean((processed[f"denoised_{name}"] - fit) ** 2))),
            f"{name}_tau_slow_s": processed[f"params_{name}"][3],
            f"{name}_n_params_at_bound": len(at_bounds),
            f"{name}_params_at_bound": " ".join(at_bounds),
        }
    return row


def bleaching_qc(recordings: dict, processed: dict, thresholds: dict) -> pd.DataFrame:
    """Table 6.3 for every recording, with flags."""
    table = per_recording(recordings, bleaching_row, processed)
    drop = table[["signal_drop_pct", "control_drop_pct"]].abs().max(axis=1)
    at_bound = table[["signal_n_params_at_bound", "control_n_params_at_bound"]].max(axis=1)
    return attach_flags(
        table,
        {
            "large_drop": drop > thresholds["max_bleaching_drop_pct"],
            "many_params_at_bound": at_bound > thresholds["max_params_at_bound"],
        },
    )

### 6.4 Movement (step A5) and connection jumps

The Pearson correlation of the detrended signal with the detrended control, the regression slope, R², and the percentage
of the detrended signal's variance the subtraction removed (for a least-squares fit with an intercept this equals
100 x R², which is a useful consistency check). **Connection jumps**: sudden steps when a fibre connection shifts. They are found as peaks
of a **0.5-s mean-difference step detector** on the control (mean of the next 0.5 s minus mean of the previous 0.5 s) above
10 robust SDs (1.4826 x MAD of the step trace), and kept only if the *signal* shows a fractional change of **the same sign** at that moment (a
real light-path change moves both channels the same way). Because chance alone gives equal signs half of the time, the
rate is an upper bound on true jumps. We report jumps per minute and the median size (% of the fitted baseline).
**Look for:** a slope near 1, a high R² (most of the signal's variance is shared), and a jump rate near zero.

In [ ]:
def mean_difference_step(x: np.ndarray, width: int) -> tuple[np.ndarray, np.ndarray]:
    """Mean of the next *width* samples minus the mean of the previous *width*, at each boundary.

    Returns:
        tuple[np.ndarray, np.ndarray]: Index of the first sample after each boundary, and the step.
    """
    cumulative = np.concatenate([[0.0], np.cumsum(x)])
    boundary = np.arange(width, len(x) - width + 1)
    after = (cumulative[boundary + width] - cumulative[boundary]) / width
    before = (cumulative[boundary] - cumulative[boundary - width]) / width
    return boundary, after - before


def detect_jumps(
    control_detrended, signal_detrended, control_fit, signal_fit, fs, window_s, mad_threshold
) -> pd.DataFrame:
    """Candidate connection jumps on the control, confirmed when the signal changes the same way.

    Returns:
        pd.DataFrame: One row per candidate with ``sample``, ``control_pct`` and ``signal_pct`` (step as % of
        the fitted baseline in each channel) and ``confirmed`` (equal sign).
    """
    width = int(round(window_s * fs))
    boundary, step_control = mean_difference_step(control_detrended, width)
    _, step_signal = mean_difference_step(signal_detrended, width)
    threshold = mad_threshold * robust_sd(step_control)
    peaks, _ = sps.find_peaks(np.abs(step_control), height=threshold, distance=width)
    sample = boundary[peaks]
    control_pct = 100 * step_control[peaks] / control_fit[sample]
    signal_pct = 100 * step_signal[peaks] / signal_fit[sample]
    return pd.DataFrame(
        {
            "sample": sample,
            "control_pct": control_pct,
            "signal_pct": signal_pct,
            "confirmed": np.sign(control_pct) == np.sign(signal_pct),
        }
    )


def movement_row(rec, processed_traces: dict, window_s: float, jump_threshold: float) -> dict:
    """Movement-correction QC columns of one recording; *processed_traces* is its ``PROCESSED`` entry."""
    processed = processed_traces["filtered"]
    stats = processed["motion_stats"]
    removed = 100 * (1 - processed["corrected"].var() / processed["detrended_signal"].var())
    jumps = detect_jumps(
        processed["detrended_control"],
        processed["detrended_signal"],
        processed["fit_control"],
        processed["fit_signal"],
        rec.fs,
        window_s,
        jump_threshold,
    )
    confirmed = jumps[jumps["confirmed"]]
    return {
        "pearson_r": stats["r"],
        "slope": stats["slope"],
        "r2": stats["r2"],
        "variance_removed_pct": removed,
        "jumps_per_min": len(confirmed) / (len(rec.signal) / rec.fs / 60),
        "n_jumps": len(confirmed),
        "median_jump_size_pct": confirmed["control_pct"].abs().median()
        if len(confirmed)
        else np.nan,
    }


def movement_qc(
    recordings: dict, processed: dict, thresholds: dict, window_s: float, jump_threshold: float
) -> pd.DataFrame:
    """Table 6.4 for every recording, with flags."""
    table = per_recording(
        recordings, movement_row, processed, window_s=window_s, jump_threshold=jump_threshold
    )
    return attach_flags(
        table,
        {
            "non_positive_slope": table["slope"] <= thresholds["min_movement_slope"],
            "many_jumps": table["jumps_per_min"] > thresholds["max_jumps_per_min"],
        },
    )

### 6.5 Signal-to-noise of the final traces

Definitions are given at the start of this section. **Look for:** a positive `snr_db` (slow fluctuations above the noise
floor), a positive `specificity_db` (the *uncorrected* sensor beats the control), and
`transient_amp_over_noise` well above the detection threshold of k = 5.

**Reading `specificity_db`** (default flag threshold `min_specificity_db` = 1 dB): the sensor's *uncorrected* SNR minus the control's SNR. A value
near 0 dB means slow fluctuations are equally present in both channels; the movement regression can then only remove what is common, and little
sensor-specific slow signal is expected. `snr_db` (the movement-corrected sensor) remains the primary SNR; `uncorrected_snr_db` is the sensor before
step A5.

In [ ]:
def welch_band_means(x: np.ndarray, fs: float, bands: list[tuple[float, float]], segment_s: float):
    """Mean power spectral density (Welch) of *x* inside each band (Hz)."""
    freqs, psd = sps.welch(x, fs=fs, nperseg=min(int(segment_s * fs), len(x)), detrend="constant")
    return [float(psd[(freqs >= lo) & (freqs <= hi)].mean()) for lo, hi in bands]


def spectral_snr_db(x: np.ndarray, fs: float, signal_band, noise_band, segment_s: float) -> float:
    """10 log10 of mean PSD in *signal_band* over mean PSD in *noise_band*."""
    slow, noise = welch_band_means(x, fs, [signal_band, noise_band], segment_s)
    return 10 * np.log10(slow / noise)


def noise_floor_sigma(x: np.ndarray, fs: float, cutoff_hz: float) -> float:
    """Robust SD of the part of *x* above *cutoff_hz*."""
    return robust_sd(highpass_zero_phase(x, fs, cutoff_hz))


def transient_stats(lowpassed: np.ndarray, fs: float, sigma_noise: float, k: float) -> dict:
    """Transients of a low-passed trace: peaks with prominence above *k* robust SDs of the trace."""
    prominences = sps.find_peaks(lowpassed, prominence=k * robust_sd(lowpassed))[1]["prominences"]
    minutes = len(lowpassed) / fs / 60
    return {
        "transients_per_min": len(prominences) / minutes,
        "transient_amp_over_noise": float(np.median(prominences) / sigma_noise)
        if len(prominences)
        else np.nan,
    }


def snr_row(
    rec, processed_traces: dict, signal_band, noise_band, segment_s: float, prominence_sd: float
) -> dict:
    """Spectral and transient SNR columns of one recording; *processed_traces* is its ``PROCESSED`` entry."""
    unfiltered = processed_traces["unfiltered"]
    bands = (signal_band, noise_band, segment_s)
    sensor_db = spectral_snr_db(unfiltered["dff"], rec.fs, *bands)
    control_db = spectral_snr_db(unfiltered["control_dff"], rec.fs, *bands)
    sigma = noise_floor_sigma(unfiltered["dff"], rec.fs, noise_band[0])
    uncorrected_db = spectral_snr_db(unfiltered["uncorrected_dff"], rec.fs, *bands)
    return {
        "snr_db": sensor_db,
        "control_snr_db": control_db,
        "uncorrected_snr_db": uncorrected_db,
        "specificity_db": uncorrected_db - control_db,
        "sigma_noise_pct": sigma,
        **transient_stats(processed_traces["filtered"]["dff"], rec.fs, sigma, prominence_sd),
    }


def snr_qc(
    recordings: dict,
    processed: dict,
    thresholds: dict,
    signal_band,
    noise_band,
    segment_s: float,
    prominence_sd: float,
) -> pd.DataFrame:
    """Table 6.5 for every recording, with flags."""
    table = per_recording(
        recordings,
        snr_row,
        processed,
        signal_band=signal_band,
        noise_band=noise_band,
        segment_s=segment_s,
        prominence_sd=prominence_sd,
    )
    return attach_flags(
        table,
        {
            "low_snr": table["snr_db"] < thresholds["min_snr_db"],
            "low_specificity": table["specificity_db"] < thresholds["min_specificity_db"],
        },
    )

### Sanity check of the QC functions on synthetic data

Each measure is run on data with a **known** answer before it is used on real recordings. The spectral SNR must recover the
calculated value (a sum of equal-amplitude sinusoids at 0.1-3 Hz over white noise) to within 1 dB, and fall as the noise
rises; the noise floor must recover the white-noise level within the fraction that the >15 Hz band holds; transient
counts and sizes must match the injected ones; the jump detector must find injected steps and reject one whose sign
disagrees between channels; the evoked measures must recover a known response and d′.

In [ ]:
def synthetic_slow_plus_noise(
    rng, fs: float, minutes: float, noise_sd: float, n_tones: int = 50, tone_amp: float = 1.0
):
    """Equal-amplitude sinusoids at 0.1-3 Hz plus white noise, and the SNR (dB) the Welch ratio should give."""
    n = int(minutes * 60 * fs)
    t = np.arange(n) / fs
    freqs = np.linspace(0.15, 2.95, n_tones)
    slow = sum(tone_amp * np.sin(2 * np.pi * f * t + rng.uniform(0, 2 * np.pi)) for f in freqs)
    slow_psd = n_tones * tone_amp**2 / 2 / (QC_SIGNAL_BAND_HZ[1] - QC_SIGNAL_BAND_HZ[0])
    noise_psd = 2 * noise_sd**2 / fs  # one-sided PSD of white noise
    return slow + noise_sd * rng.standard_normal(n), 10 * np.log10(slow_psd / noise_psd)


def check_qc_functions_on_synthetic_data() -> pd.DataFrame:
    """Assert that each QC measure recovers a known answer; return the comparison."""
    fs, rng = 130.0, np.random.default_rng(3)
    rows = []

    snrs = []
    for noise_sd in (0.5, 1.0, 2.0):
        x, expected = synthetic_slow_plus_noise(rng, fs, 10.0, noise_sd)
        measured = spectral_snr_db(x, fs, QC_SIGNAL_BAND_HZ, QC_NOISE_BAND_HZ, QC_WELCH_SEGMENT_S)
        assert abs(measured - expected) < 1.0, (noise_sd, measured, expected)
        snrs.append(measured)
        rows.append((f"Spectral SNR, noise SD {noise_sd}", measured, expected))
    assert snrs[0] > snrs[1] > snrs[2], snrs

    sigma = noise_floor_sigma(0.5 * rng.standard_normal(int(120 * fs)), fs, QC_NOISE_BAND_HZ[0])
    share = 1 - QC_NOISE_BAND_HZ[0] / (fs / 2)  # fraction of white-noise variance above the cutoff
    assert abs(sigma / (0.5 * np.sqrt(share)) - 1) < 0.1, sigma
    rows.append(
        ("Noise floor (SD of >15 Hz part of white noise SD 0.5)", sigma, 0.5 * np.sqrt(share))
    )

    n_events, amplitude = 20, 5.0
    t = np.arange(int(300 * fs)) / fs
    onsets = np.linspace(10, 280, n_events)
    for noise_sd in (0.3, 0.6):
        trace = amplitude * sum(
            transient_shape(t, onset) for onset in onsets
        ) + noise_sd * rng.standard_normal(len(t))
        low = lowpass_zero_phase(trace, fs, 10.0)
        sigma_noise = noise_floor_sigma(trace, fs, QC_NOISE_BAND_HZ[0])
        stats = transient_stats(low, fs, sigma_noise, QC_TRANSIENT_PROMINENCE_SD)
        assert abs(stats["transients_per_min"] * 5 - n_events) <= 2, stats
        ratio = stats["transient_amp_over_noise"]
        assert abs(ratio / (amplitude / sigma_noise) - 1) < 0.25, (ratio, amplitude / sigma_noise)
        rows.append(
            (f"Transient amplitude / noise, noise SD {noise_sd}", ratio, amplitude / sigma_noise)
        )

    n = 130 * 60
    base_c, base_s = np.full(n, 2.0), np.full(n, 2.5)
    control = 0.004 * rng.standard_normal(n)
    signal = 0.004 * rng.standard_normal(n)
    control, signal = lowpass_zero_phase(control, fs, 10.0), lowpass_zero_phase(signal, fs, 10.0)
    for second, d_control, d_signal in ((20, 0.2, 0.25), (40, -0.1, -0.12), (50, 0.2, -0.25)):
        control[int(second * fs) :] += d_control
        signal[int(second * fs) :] += d_signal
    jumps = detect_jumps(
        control, signal, base_c, base_s, fs, QC_JUMP_WINDOW_S, QC_JUMP_MAD_THRESHOLD
    )
    assert len(jumps) == 3 and jumps["confirmed"].tolist() == [True, True, False], jumps
    np.testing.assert_allclose(jumps["sample"] / fs, [20, 40, 50], atol=0.02)
    np.testing.assert_allclose(jumps["control_pct"], [10, -5, 10], rtol=0.1)
    rows.append(("Jumps found / confirmed (injected 3 / 2)", len(jumps), 3))
    return pd.DataFrame(rows, columns=["check", "measured", "expected"])


display(check_qc_functions_on_synthetic_data())

### QC tables for every recording

Run in order. Each table is saved as CSV; the combined list of flags follows them.

In [ ]:
qc_acquisition = acquisition_qc(RECORDINGS, QC_THRESHOLDS, QC_FLATLINE_S, QC_BASELINE_SPIKE_SD)
show_table(qc_acquisition, "qc_1_acquisition")

In [ ]:
qc_denoising = denoising_qc(RECORDINGS, PROCESSED, QC_THRESHOLDS, QC_NOISE_BAND_HZ[0])
show_table(qc_denoising, "qc_2_denoising")

In [ ]:
qc_bleaching = bleaching_qc(RECORDINGS, PROCESSED, QC_THRESHOLDS)
show_table(qc_bleaching, "qc_3_bleaching")

In [ ]:
qc_movement = movement_qc(
    RECORDINGS, PROCESSED, QC_THRESHOLDS, QC_JUMP_WINDOW_S, QC_JUMP_MAD_THRESHOLD
)
show_table(qc_movement, "qc_4_movement")

In [ ]:
qc_snr = snr_qc(
    RECORDINGS,
    PROCESSED,
    QC_THRESHOLDS,
    QC_SIGNAL_BAND_HZ,
    QC_NOISE_BAND_HZ,
    QC_WELCH_SEGMENT_S,
    QC_TRANSIENT_PROMINENCE_SD,
)
show_table(qc_snr, "qc_5_snr")

In [ ]:
def combined_flags(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """One row per recording with the flags from each QC table."""
    out = next(iter(tables.values()))[["recording", "mouse", "session"]].copy()
    for name, table in tables.items():
        out[name] = table["flags"].to_numpy()
    return out


qc_flags = combined_flags(
    {
        "acquisition": qc_acquisition,
        "denoising": qc_denoising,
        "bleaching": qc_bleaching,
        "movement": qc_movement,
        "snr": qc_snr,
    }
)
show_table(qc_flags, "qc_flags")

## 7. Outcome-aligned responses, per mouse

**Events.** Rewarded trials are aligned to `outcome_time` with kind `reward_onset` (entry to the `choice` state, the onset
of reward pulse 1). Unrewarded trials are aligned to the `no_reward_cue_onset` (entry to `ITI`, the onset of the no-reward
cue). Both come from `load_outcome_events`; trials of the phases in `PHASES` are used.

**Windows.** −5 to +10 s around each event (`EVENT_WINDOW`), interpolated onto a common grid at the recording rate (130 Hz).
Only windows lying completely inside the sync coverage **and clear of any stretch of missing sync time** are kept (a window is never
padded, extrapolated, or interpolated across a gap); the exclusions are listed below, with the trials whose
pre-event baseline contains another outcome (kept) and any trial whose baseline SD is zero
(dropped from all four views so both pipelines use the same trials).

**Pooling.** Each recording is normalised on its own (pipeline A: its session ΔF/F and session z-score; pipeline B: each
trial's own baseline), and then the trials of a mouse's recordings are pooled. A mouse with several sessions
therefore contributes more trials, and **the shaded bands are the SEM across trials, not across mice** (the cohort is too
small and the trial counts too unequal for between-mouse statistics here; trials of one session are not independent either).

**Panels.** Rows are mice. Left column: ΔF/F (pipeline A: session ΔF/F, `DEFAULT_SMOOTHING_A` points; pipeline B: trial ΔF/F, baseline-subtracted only if
`B_EVENT_LEVEL["subtract_dff_baseline"]`, smoothed with `DEFAULT_SMOOTHING` points). Right column: z-score (A: session z-score; B: trial z-score over
`DEFAULT_PRE_WINDOW`, smoothed). Rewarded in green, unrewarded in red; `n` is the number
of trials. The dashed vertical line is the event. **What to look for:** a response that follows the event in the same direction
in many mice; rewarded and unrewarded traces that diverge after 0; a flat trace before 0.

In [ ]:
CONDITIONS = ("rewarded", "unrewarded")
CONDITION_COLORS = {"rewarded": "tab:green", "unrewarded": "tab:red"}
VIEWS = ("A_dff", "A_z", "B_dff", "B_z")
B_DFF_LABEL = (
    ("Baseline-Subtracted ΔF/F, Smoothed", "Change in ΔF/F (%)")
    if B_EVENT_LEVEL["subtract_dff_baseline"]
    else ("Trial ΔF/F, Smoothed", "ΔF/F (%)")
)
VIEW_LABELS = {
    "A_dff": ("Session ΔF/F", "ΔF/F (%)"),
    "A_z": ("Session Z-Score", "Z-Score (Session SD)"),
    "B_dff": B_DFF_LABEL,
    "B_z": ("Trial Z-Score, Smoothed", "Z-Score (Pre-Event SD)"),
    "signal_uncorrected": ("(a) Signal ΔF/F, Before Movement Correction", "ΔF/F (%)"),
    "control": ("(b) Control ΔF/F (Its Own Fit)", "ΔF/F (%)"),
    "control_contribution": ("(c) Fitted Control Contribution (Subtracted)", "ΔF/F (%)"),
    "corrected": ("(d) Corrected Signal ΔF/F", "ΔF/F (%)"),
}
CHANNEL_VIEWS = ("signal_uncorrected", "control", "control_contribution", "corrected")
MAIN_SETTINGS = ViewSettings(
    DEFAULT_PRE_WINDOW,
    DEFAULT_SMOOTHING_A,
    DEFAULT_SMOOTHING,
    B_EVENT_LEVEL["subtract_dff_baseline"],
)


def smoothing_text(points: int, fs: float = NOMINAL_FS_HZ) -> str:
    """``"no smoothing"`` or ``"80-point moving mean (0.62 s at 130 Hz)"``."""
    if points == 1:
        return "no smoothing"
    return f"{points}-point moving mean ({points / fs:.2f} s at {fs:g} Hz)"


def settings_text(settings: ViewSettings, pipeline: str) -> str:
    """The settings that shape pipeline *pipeline*'s ("A" or "B") views, for figure titles."""
    if pipeline == "A":
        return smoothing_text(settings.smoothing_a)
    pre = f"{-settings.pre_window[0]:g}-s pre-event baseline"
    return f"{pre}, {smoothing_text(settings.smoothing_b)}"


def channel_traces(p: dict) -> dict[str, np.ndarray]:
    """Whole-session traces (ΔF/F %) for the control-channel check: (a) = (c) + (d) exactly.

    *p* is a recording's ``PROCESSED[rid]["filtered"]``. The fitted control contribution is the estimated movement
    component (V) expressed as % of the signal's bleaching fit.
    """
    return {
        "signal_uncorrected": p["uncorrected_dff"],
        "control": p["control_dff"],
        "control_contribution": delta_f_over_f(p["movement"], p["fit_signal"]),
        "corrected": p["dff"],
    }


def compute_views(
    windows: dict[str, np.ndarray],
    rel_time: np.ndarray,
    settings: ViewSettings,
    shown_window: tuple[float, float],
    fs: float,
):
    """The four displayed versions of one set of event windows, restricted to the same trials.

    The windows are extracted over *shown_window* padded for the smoothing (:func:`padded_window`); the moving
    mean is applied to the padded windows and the views are then cropped to *shown_window*, so every returned
    sample was smoothed with a full window.

    Args:
        windows (dict): Output of :func:`trace_windows` for ``session_traces``, over the padded window.
        rel_time (np.ndarray): Time relative to the event (s) of the padded windows.
        settings (ViewSettings): Pre-window, smoothing of each pipeline, ΔF/F baseline switch.
        shown_window (tuple[float, float]): The interval kept, relative to the event (s).
        fs (float): Sampling rate (Hz).

    Returns:
        tuple[dict, np.ndarray]: Arrays by view name (``A_dff``, ``A_z``, ``B_dff``, ``B_z``) for the valid trials,
        cropped to *shown_window*, and the validity mask (False where the pipeline-B trial baseline had zero SD).
    """
    b_views, valid = normalise_trials_b(windows["B_dff"], windows["B_z_source"], rel_time, settings)
    views = {
        "A_dff": matlab_moving_mean(windows["A_dff"][valid], settings.smoothing_a),
        "A_z": matlab_moving_mean(windows["A_z"][valid], settings.smoothing_a),
        "B_dff": b_views["dff"],
        "B_z": b_views["z"],
    }
    shown = crop_mask(rel_time, shown_window, fs)
    return {name: trials[:, shown] for name, trials in views.items()}, valid


def count_baseline_overlaps(event_times: np.ndarray, all_outcomes: np.ndarray, baseline) -> int:
    """Events whose baseline window ``[t + start, t + stop)`` contains an outcome (the event itself is at ``t``)."""
    ordered = np.sort(all_outcomes)
    inside = np.searchsorted(ordered, event_times + baseline[1], side="left") - np.searchsorted(
        ordered, event_times + baseline[0], side="left"
    )
    return int((inside > 0).sum())


def check_count_baseline_overlaps() -> None:
    """The window is ``[t + start, t + stop)``: its stop is honoured, the event itself and later outcomes never count."""
    outcomes = np.array([0.0, 3.0, 4.5, 10.0, 12.0])
    one = np.array([12.0])
    assert count_baseline_overlaps(np.array([10.0]), outcomes, (-5.0, 0.0)) == 0  # [5, 10): nothing
    assert count_baseline_overlaps(one, outcomes, (-5.0, 0.0)) == 1  # [7, 12): the outcome at 10
    assert count_baseline_overlaps(one, outcomes, (-5.0, -1.0)) == 1  # [7, 11): the outcome at 10
    assert (
        count_baseline_overlaps(one, outcomes, (-5.0, -3.0)) == 0
    )  # [7, 9): the outcome at 10 is after the stop
    assert (
        count_baseline_overlaps(np.array([4.5]), outcomes, (-5.0, -1.0)) == 1
    )  # [-0.5, 3.5): outcomes at 0 and 3
    assert (
        count_baseline_overlaps(np.array([10.0, 12.0]), outcomes, (-5.0, 0.0)) == 1
    )  # counts events, not outcomes


check_count_baseline_overlaps()


def real_condition_windows(
    rec,
    processed: dict,
    traces: dict[str, np.ndarray],
    events: pd.DataFrame,
    condition: str,
    window: tuple[float, float],
    settings: ViewSettings,
    channel_views: bool = True,
) -> dict:
    """Event-aligned traces of one recording and one outcome, in all views, with the counts behind them.

    *processed* is the recording's ``PROCESSED[rid]["filtered"]``, *traces* its ``session_traces``, *events* its
    labelled outcome events and *window* the interval that is kept; the windows are cut wider, so that the
    smoothing never meets their edge (see :func:`compute_views`). ``pulses`` holds the reward-pulse count of
    each used trial. *channel_views* adds the four control-channel check views of section 7b (signal
    recordings only).
    """
    selected = events[events["condition"] == condition]
    times = selected["outcome_time"].to_numpy()
    padded = padded_window(window, max(settings.smoothing_a, settings.smoothing_b), rec.fs)
    rel_padded, windows, keep = trace_windows(rec, traces, times, padded)
    views, valid = compute_views(windows, rel_padded, settings, window, rec.fs)
    shown = crop_mask(rel_padded, window, rec.fs)
    if channel_views:
        for name, trace in channel_traces(processed).items():
            trials = extract_event_windows(rec.time, trace, times, padded, rec.fs)[1][valid]
            views[name] = trials[:, shown]
    return {
        "rel_time": rel_padded[shown],
        "views": views,
        "pulses": selected["num_rew_pulses"].to_numpy()[keep][valid],
        "n_events": len(times),
        "n_outside_sync": int((~keep).sum()),
        "n_zero_baseline_sd": int((~valid).sum()),
        "n_used": int(valid.sum()),
        "n_baseline_overlap": count_baseline_overlaps(
            times[keep][valid], events["outcome_time"].to_numpy(), settings.pre_window
        ),
    }


def build_real(
    recordings: dict,
    processed: dict,
    traces: dict,
    events: dict,
    window,
    settings: ViewSettings,
    channel_views: bool = True,
    conditions: tuple[str, ...] = CONDITIONS,
) -> dict:
    """:func:`real_condition_windows` for every recording and condition, keyed ``(recording id, condition)``."""
    return {
        (rid, cond): real_condition_windows(
            rec,
            processed[rid]["filtered"],
            traces[rid],
            events[rid],
            cond,
            window,
            settings,
            channel_views,
        )
        for rid, rec in recordings.items()
        for cond in conditions
    }


REAL = build_real(RECORDINGS, PROCESSED, TRACES, EVENTS, EXTRACTION_WINDOW, MAIN_SETTINGS)
REAL_CONTROL = build_real(
    RECORDINGS,
    PROCESSED,
    TRACES_CONTROL,
    EVENTS,
    EXTRACTION_WINDOW,
    MAIN_SETTINGS,
    channel_views=False,
)
REL_TIME = next(iter(REAL.values()))["rel_time"]

exclusions = pd.DataFrame(
    [
        {
            "recording": rid,
            "mouse": RECORDINGS[rid].subject,
            "condition": cond,
            **{k: v for k, v in r.items() if k.startswith("n_")},
        }
        for (rid, cond), r in REAL.items()
    ]
)
show_table(exclusions, "window_exclusions")
print(
    f"Trials: {exclusions.groupby('condition')['n_events'].sum().to_dict()} found, "
    f"{exclusions.groupby('condition')['n_used'].sum().to_dict()} used; "
    f"{exclusions['n_outside_sync'].sum()} excluded for incomplete windows, "
    f"{exclusions['n_zero_baseline_sd'].sum()} for a zero baseline SD; "
    f"{exclusions['n_baseline_overlap'].sum()} used trials have another outcome inside the baseline window."
)

In [ ]:
def pool_real(real: dict, recordings: dict, mouse: str, condition: str, view: str) -> np.ndarray:
    """All trials of *mouse* for one outcome and view, over its recordings (rows are trials).

    *real* maps ``(recording id, condition)`` to :func:`real_condition_windows` output; *recordings* maps id to recording.
    """
    parts = [
        real[(rid, condition)]["views"][view]
        for rid, rec in recordings.items()
        if rec.subject == mouse
    ]
    return np.concatenate(parts, axis=0)


def plot_outcome_responses(
    ax, rel_time, trials_by_condition: dict[str, np.ndarray], ylabel: str, legend: bool = True
) -> None:
    """Mean ± SEM across trials for each outcome, with n in the legend (if *legend*) and the event marked."""
    for condition, trials in trials_by_condition.items():
        label = f"{condition.capitalize()} (n = {len(trials)})"
        if len(trials) == 0:
            ax.plot([], [], color=CONDITION_COLORS[condition], label=label)
            continue
        mean, sem = mean_and_sem(trials)
        ax.plot(rel_time, mean, color=CONDITION_COLORS[condition], lw=1.3, label=label)
        ax.fill_between(
            rel_time, mean - sem, mean + sem, color=CONDITION_COLORS[condition], alpha=0.25, lw=0
        )
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.axhline(0, color="0.8", lw=0.6)
    ax.set_ylabel(ylabel)
    if legend:
        ax.legend(loc="upper right", frameon=False)


def response_grid(
    views: tuple[str, ...],
    title: str,
    plot_cell: Callable,
    mice: list[str],
    event_window: tuple[float, float],
    legend_handles=None,
    axes=None,
):
    """One row per mouse, one column per view; *plot_cell* draws a mouse's cell into an axis.

    *legend_handles* (optional) puts one shared legend below the figure instead of one per panel. *axes* is an
    optional ``(len(mice), len(views))`` array of axes to draw on; if None a new figure is created. Returns the figure.
    """
    if axes is None:
        _, axes = plt.subplots(
            len(mice),
            len(views),
            figsize=(max(11, 3.9 * len(views)), 2.3 * len(mice) + (0.6 if legend_handles else 0)),
            sharex=True,
            constrained_layout=True,
            squeeze=False,
        )
    axes = np.asarray(axes).reshape(len(mice), len(views))
    fig = axes[0, 0].figure
    for row, mouse in zip(axes, mice, strict=True):
        for ax, view in zip(row, views, strict=True):
            plot_cell(ax, mouse, view, VIEW_LABELS[view][1])
        row[0].annotate(
            mouse,
            xy=(0.02, 0.96),
            xycoords="axes fraction",
            va="top",
            fontsize=9,
            fontweight="bold",
        )
    for ax, view in zip(axes[0], views, strict=True):
        ax.set_title(VIEW_LABELS[view][0])
    for ax in axes[-1]:
        ax.set_xlabel("Time From Outcome (s)")
        ax.set_xlim(event_window)
    if legend_handles:
        fig.legend(
            handles=legend_handles,
            loc="outside lower center",
            ncol=len(legend_handles),
            frameon=False,
        )
    fig.suptitle(title, fontsize=12)
    return fig


def plot_pooled_cell(ax, mouse, view, ylabel, *, real, recordings, rel_time):
    """Section-7 cell: rewarded and unrewarded responses of *mouse* in *view*."""
    pooled = {c: pool_real(real, recordings, mouse, c, view) for c in CONDITIONS}
    plot_outcome_responses(ax, rel_time, pooled, ylabel)


plot_pooled = partial(plot_pooled_cell, real=REAL, recordings=RECORDINGS, rel_time=REL_TIME)


def panel_ylims(fig: plt.Figure, n_rows: int, n_cols: int) -> list[list[tuple[float, float]]]:
    """y-limits of the panels of a figure made by ``plt.subplots(n_rows, n_cols)`` (row-major)."""
    axes = np.array(fig.axes[: n_rows * n_cols]).reshape(n_rows, n_cols)
    return [[ax.get_ylim() for ax in row] for row in axes]


def apply_ylims(fig: plt.Figure, ylims: list[list[tuple[float, float]]]) -> None:
    """Set the y-limits of a figure's panels (row-major) to *ylims*, so two figures compare directly."""
    n_rows, n_cols = len(ylims), len(ylims[0])
    axes = np.array(fig.axes[: n_rows * n_cols]).reshape(n_rows, n_cols)
    for row_axes, row_limits in zip(axes, ylims, strict=True):
        for ax, limits in zip(row_axes, row_limits, strict=True):
            ax.set_ylim(limits)

### Pipeline A: session ΔF/F and session z-score

In [ ]:
fig = response_grid(
    ("A_dff", "A_z"),
    f"Pipeline A: Outcome-Aligned Responses (Mean ± SEM Across Trials; {settings_text(MAIN_SETTINGS, 'A')})",
    plot_pooled,
    MICE,
    EVENT_WINDOW,
)
YLIMS_A = panel_ylims(fig, len(MICE), 2)
save_figure(fig, "responses_pipeline_A")
plt.show()

### Pipeline B: trial ΔF/F and trial z-score, smoothed

In [ ]:
fig = response_grid(
    ("B_dff", "B_z"),
    f"Pipeline B: Outcome-Aligned Responses (Mean ± SEM Across Trials; {settings_text(MAIN_SETTINGS, 'B')})",
    plot_pooled,
    MICE,
    EVENT_WINDOW,
)
YLIMS_B = panel_ylims(fig, len(MICE), 2)
save_figure(fig, "responses_pipeline_B")
plt.show()

### The 405-nm control channel on the same footing as the signal

The control is processed **in parallel with the signal and without the motion correction** (there is nothing to regress it on):
the same low-pass, its own bleaching fit (pipeline A: the bounded double exponential; pipeline B: the same model choice as the
signal, `exp2` unless it fell back), ΔF/F relative to that fit, pipeline A's z-score over the detrended control across the
session, and pipeline B's trial normalisation exactly as for the signal (the same pre-event window, z-score from the detrended
control, same smoothing). Layout and settings are those of the signal figures above, and **each panel has the same y-limits as the
matching signal panel**, so a control response of the same size as the signal's looks the same size. The control's ΔF/F is much larger than
the corrected signal's (it has no motion correction), so in many panels the control leaves the signal's y-range; the same figures are
therefore repeated with each panel on its own y-limits, to show the shape of the control response.

**What to look for.** An outcome response of the same shape and size in the control as in the (uncorrected) signal points to a
source the two channels share, such as movement or haemodynamics, or to the control channel itself responding to norepinephrine
(405 nm being not exactly isosbestic). A response in the signal and a flat control is what a sensor response would look like.
Note that pipeline A and B's *signal* panels are corrected for movement, the control panels are not, so the comparison with the
uncorrected signal is in section 7b.

In [ ]:
plot_pooled_control = partial(
    plot_pooled_cell, real=REAL_CONTROL, recordings=RECORDINGS, rel_time=REL_TIME
)
for matched in (True, False):
    for pipeline, views, signal_ylims in (
        ("A", ("A_dff", "A_z"), YLIMS_A),
        ("B", ("B_dff", "B_z"), YLIMS_B),
    ):
        scale_note = (
            "Y-Limits Per Panel As In The Signal Figure Above (Clipped Where The Control Is Larger)"
            if matched
            else "Own Y-Limits Per Panel, Not Comparable In Size With The Signal Figure"
        )
        fig = response_grid(
            views,
            f"Pipeline {pipeline}, 405-nm Control Channel, Not Motion-Corrected (Mean ± SEM Across Trials; "
            f"{settings_text(MAIN_SETTINGS, pipeline)})\n{scale_note}",
            plot_pooled_control,
            MICE,
            EVENT_WINDOW,
        )
        if matched:
            apply_ylims(fig, signal_ylims)
        save_figure(
            fig, f"responses_control_pipeline_{pipeline}" + ("" if matched else "_own_scale")
        )
        plt.show()

### Peak and area under the curve

Descriptive summaries of each mouse's **mean z-score trace** over the post-event window (0 to 10 s): the largest value, its
time, and the signed area under the curve (trapezoid rule). Pipeline A's are in session SD units, pipeline B's in pre-event SD
units of the (smoothed) trace, so the two are not comparable in size.

In [ ]:
def response_metrics_table(
    real: dict, recordings: dict, mice: list[str], rel_time: np.ndarray, window
) -> pd.DataFrame:
    """Peak, peak time and AUC (over *window*) of each mouse's mean z-score trace, by outcome and pipeline."""
    rows = []
    for mouse in mice:
        for condition in CONDITIONS:
            for view, pipeline in (("A_z", "A"), ("B_z", "B")):
                trials = pool_real(real, recordings, mouse, condition, view)
                if len(trials) == 0:
                    continue
                mean, _ = mean_and_sem(trials)
                rows.append(
                    {
                        "mouse": mouse,
                        "condition": condition,
                        "pipeline": pipeline,
                        "n_trials": len(trials),
                    }
                    | peak_and_auc(rel_time, mean, window)
                )
    return pd.DataFrame(rows)


response_metrics = response_metrics_table(REAL, RECORDINGS, MICE, REL_TIME, METRIC_WINDOW)
show_table(response_metrics, "response_peak_auc")

### 7b. Control-channel responses to the outcomes

The primer recommends looking at the control channel aligned to events: a control that is truly isosbestic for the sensor should not
respond to neural activity, and its response (if any) shows what the movement regression will subtract. Columns, all ΔF/F (%) of the
**session** (no baseline subtraction, same window, mean ± SEM across trials):

- (a) the **signal** channel ΔF/F *before* movement correction;
- (b) the **control** channel ΔF/F relative to its own bleaching fit;
- (c) the **fitted control contribution**: the part of the signal that the regression attributes to the control and subtracts, as % of the signal's
  fit (its shape is the control's, scaled by the regression slope);
- (d) the **corrected** ΔF/F, which is exactly (a) − (c).

**How to read the patterns.**

- A response in (a) and (d) but **not** in (b): a response that exists only in the signal channel, consistent with a sensor signal.
- A **same-sign** response in (a) and (b) that is mostly gone in (d): movement or haemodynamic artefact common to both channels, which the regression
  removes. The same picture would arise if the **control itself responds to norepinephrine**; then the regression subtracts real signal, and (c) shows how much.
- An **opposite-sign** response in (b) relative to (a): the 405 nm channel is probably **not isosbestic for nLightG2** (its fluorescence changes
  with the sensor's state in the opposite direction), and the control regression cannot be trusted as a movement correction.

**Not verified:** the isosbestic point of nLightG2 has not been checked here; Meg should confirm that 405 nm is isosbestic for this sensor
(and that the channels are wired as assumed).

The table gives, per mouse and outcome, the **peak absolute response in 0-2 s** of the session-ΔF/F mean trace relative to its −2 to 0 s mean, for the signal before correction
and for the control, their ratio (signal / control), and whether the two peaks have the same sign.

In [ ]:
def peak_abs_response(rel_time: np.ndarray, mean_trace: np.ndarray, pre, post) -> float:
    """Signed value of the largest |mean trace minus its *pre*-interval mean| within the *post* interval."""
    response = mean_trace - mean_trace[interval_mask(rel_time, pre)].mean()
    window = response[interval_mask(rel_time, post)]
    return float(window[np.argmax(np.abs(window))])


def control_channel_table(
    real: dict, recordings: dict, mice: list[str], rel_time: np.ndarray, pre, post
) -> pd.DataFrame:
    """Per mouse and outcome: peak response (0-2 s) of the uncorrected signal and of the control, and their ratio."""
    rows = []
    for mouse in mice:
        for condition in CONDITIONS:
            signal = pool_real(real, recordings, mouse, condition, "signal_uncorrected")
            control = pool_real(real, recordings, mouse, condition, "control")
            if len(signal) == 0:
                continue
            peak_signal = peak_abs_response(rel_time, signal.mean(axis=0), pre, post)
            peak_control = peak_abs_response(rel_time, control.mean(axis=0), pre, post)
            rows.append(
                {
                    "mouse": mouse,
                    "condition": condition,
                    "n_trials": len(signal),
                    "peak_signal_pct": peak_signal,
                    "peak_control_pct": peak_control,
                    "ratio_signal_over_control": abs(peak_signal) / abs(peak_control),
                    "same_sign": bool(np.sign(peak_signal) == np.sign(peak_control)),
                }
            )
    return pd.DataFrame(rows)


fig = response_grid(
    CHANNEL_VIEWS,
    "Control-Channel Check: Outcome-Aligned ΔF/F Before and After Movement Correction",
    plot_pooled,
    MICE,
    EVENT_WINDOW,
)
save_figure(fig, "responses_control_channel")
plt.show()

control_channel = control_channel_table(REAL, RECORDINGS, MICE, REL_TIME, EVOKED_PRE, EVOKED_POST)
show_table(control_channel, "qc_7_control_channel_peaks")

### 7c. Sensitivity to the pre-outcome window and to smoothing

For pipeline B's trial-normalised ΔF/F and z-score, a grid: **rows are the pre-outcome windows** (`PRE_WINDOWS_S`: 2, 5, 10 and
15 s; 15 s is the paper's z baseline), **columns are the moving-mean lengths** (`SMOOTHING_POINTS`, with seconds at 130 Hz in the
column titles). Each panel shows the **group-level** response of all mice: each mouse's trial mean first (trials of its recordings pooled),
then the mean ± SEM **across mice**, rewarded in green and unrewarded in red. The y-axis is shared along a row.

- The ΔF/F grid has the pre-window mean **subtracted** (the paper's tail-lift option). With the code's default (no subtraction) the
  ΔF/F does not depend on the window at all, except through which trials are complete, so the subtracted version is the one that
  shows the window's effect.
- Smoothing never meets the edge of a window: each trial window is cut wider by the moving-mean half-width, smoothed, and then cropped
  to the displayed interval (section 4). The same trials are used at every length of a given pre window.
- A window that reaches further back needs more of the recording before the outcome, so **longer windows exclude more trials**
  (an outcome in the first seconds of a recording, or after a sync gap, cannot have a complete window). The table gives the trial
  counts, the mice that keep at least one trial, and how many kept trials have another outcome inside the window.
- Pipeline A (session-normalised) has no pre window; its smoothing is varied the same way in the last figure of this section.
  In `sensitivity_peak_responses` its `A_z` rows repeat across pre windows except for the trials kept (column `pre_window_affects`).
  The ΔF/F rows there are named `B_dff_minus_pre_mean`: they are the baseline-subtracted ΔF/F of this grid, not the main figures'
  `B_dff` (which has no subtraction by default).

In [ ]:
SENSITIVITY_VIEWS = ("A_dff", "A_z", "B_dff", "B_z")


def sensitivity_sums(
    recordings: dict,
    traces: dict,
    events: dict,
    event_window: tuple[float, float],
    other_windows: tuple,
    pre_windows: list,
    smoothings: list,
    subtract_dff_baseline: bool,
) -> dict:
    """Per pre window and smoothing: the sum over trials and trial count of each recording, outcome and view.

    Sums and counts, not trials, are kept, so the whole grid stays small; a mouse's mean trace is the total sum
    over the total count. The windows are cut once per pre window, padded for the longest smoothing so the same
    trials are used at every length, then smoothed for each length and cropped.

    Returns:
        dict: ``"sums"``: ``{(pre, smoothing): {(rid, cond, view): (sum_trace, n)}}``; ``"rel_time"``:
        ``{pre: grid}``; ``"counts"``: ``{(pre, rid, cond): (n_used, n_overlap)}`` with the trials kept and
        how many of them have another outcome inside the pre window.
    """
    sums = {(pre, s): {} for pre in pre_windows for s in smoothings}
    rel_times, counts = {}, {}
    for pre in pre_windows:
        window = widest_window(event_window, pre, *other_windows)
        for rid, rec in recordings.items():
            padded = padded_window(window, max(smoothings), rec.fs)
            all_outcomes = events[rid]["outcome_time"].to_numpy()
            for cond in CONDITIONS:
                times = events[rid].loc[events[rid]["condition"] == cond, "outcome_time"].to_numpy()
                rel_padded, windows, keep = trace_windows(rec, traces[rid], times, padded)
                rel_times[pre] = rel_padded[crop_mask(rel_padded, window, rec.fs)]
                for s in smoothings:
                    settings = ViewSettings(pre, s, s, subtract_dff_baseline)
                    views, valid = compute_views(windows, rel_padded, settings, window, rec.fs)
                    for view in SENSITIVITY_VIEWS:
                        sums[(pre, s)][(rid, cond, view)] = (
                            views[view].sum(axis=0),
                            len(views[view]),
                        )
                counts[(pre, rid, cond)] = (
                    int(valid.sum()),
                    count_baseline_overlaps(times[keep][valid], all_outcomes, pre),
                )
    return {"sums": sums, "rel_time": rel_times, "counts": counts}


def mouse_means_from_sums(
    sums: dict, recordings: dict, mice: list, condition: str, view: str
) -> dict:
    """Each mouse's mean trace (total sum over total trials of its recordings); mice without a trial are left out."""
    means = {}
    for mouse in mice:
        parts = [
            sums[(rid, condition, view)] for rid, rec in recordings.items() if rec.subject == mouse
        ]
        n = sum(count for _, count in parts)
        if n:
            means[mouse] = sum(total for total, _ in parts) / n
    return means


def across_mice(mouse_means: dict) -> tuple[np.ndarray, np.ndarray, int]:
    """Mean and SEM across mice of the mouse-level traces, and the number of mice."""
    mean, sem = mean_and_sem(np.stack(list(mouse_means.values())))
    return mean, sem, len(mouse_means)


def sensitivity_counts_table(
    counts: dict, recordings: dict, mice: list, pre_windows: list
) -> pd.DataFrame:
    """Per pre window and outcome: trials kept, mice with at least one trial, fewest trials of a mouse, overlaps."""
    rows = []
    for pre in pre_windows:
        for cond in CONDITIONS:
            per_mouse = {
                m: sum(
                    counts[(pre, rid, cond)][0]
                    for rid, rec in recordings.items()
                    if rec.subject == m
                )
                for m in mice
            }
            rows.append(
                {
                    "pre_window_s": -pre[0],
                    "condition": cond,
                    "trials": sum(per_mouse.values()),
                    "mice_with_trials": sum(n > 0 for n in per_mouse.values()),
                    "fewest_trials_of_a_mouse": min(per_mouse.values()),
                    "trials_with_other_outcome_in_window": sum(
                        counts[(pre, rid, cond)][1] for rid in recordings
                    ),
                }
            )
    return pd.DataFrame(rows)


def plot_sensitivity_grid(
    axes: np.ndarray,
    rows: list[tuple[str, tuple, str]],
    sensitivity: dict,
    recordings: dict,
    mice: list,
    smoothings: list,
    x_limits: tuple[float, float],
) -> None:
    """Fill ``axes[row, column]``: rewarded and unrewarded group mean ± SEM across mice.

    *rows* is ``[(row label, pre window, view)]``; columns are *smoothings*. Rows share their y-axis; the time
    axis spans *x_limits*.
    """
    for r, (label, pre, view) in enumerate(rows):
        for c, s in enumerate(smoothings):
            ax = axes[r, c]
            for cond in CONDITIONS:
                means = mouse_means_from_sums(
                    sensitivity["sums"][(pre, s)], recordings, mice, cond, view
                )
                mean, sem, _ = across_mice(means)
                rel_time = sensitivity["rel_time"][pre]
                ax.plot(rel_time, mean, color=CONDITION_COLORS[cond], lw=1.2)
                ax.fill_between(
                    rel_time, mean - sem, mean + sem, color=CONDITION_COLORS[cond], alpha=0.25, lw=0
                )
            ax.axvline(0, color="k", ls="--", lw=0.8)
            ax.axhline(0, color="0.8", lw=0.6)
            if r == 0:
                ax.set_title(
                    smoothing_text(s).replace(" moving mean", "").replace(" (", "\n("), fontsize=9
                )
            if c == 0:
                ax.set_ylabel(label, fontsize=8)
    for ax in axes[-1]:
        ax.set_xlabel("Time From Outcome (s)")
        ax.set_xlim(x_limits)


def sensitivity_figure(sensitivity, recordings, mice, smoothings, rows, title, x_limits):
    """A grid figure (rows x smoothings) of :func:`plot_sensitivity_grid` with a shared legend below."""
    fig, axes = plt.subplots(
        len(rows),
        len(smoothings),
        figsize=(3.2 * len(smoothings) + 1.2, 2.3 * len(rows) + 1.0),
        sharex=True,
        sharey="row",
        constrained_layout=True,
        squeeze=False,
    )
    plot_sensitivity_grid(axes, rows, sensitivity, recordings, mice, smoothings, x_limits)
    fig.legend(
        handles=[
            plt.Line2D(
                [], [], color=CONDITION_COLORS[c], label=f"{c.capitalize()}: mean ± SEM across mice"
            )
            for c in CONDITIONS
        ],
        loc="outside lower center",
        ncol=2,
        frameon=False,
    )
    fig.suptitle(title, fontsize=12)
    return fig


def row_label(pre: tuple, view_text: str, counts: pd.DataFrame) -> str:
    """Row label: the view, the pre window and the rewarded/unrewarded trial counts for that window."""
    c = counts[counts["pre_window_s"] == -pre[0]].set_index("condition")["trials"]
    return f"{view_text}\nPre {-pre[0]:g} s\n{c['rewarded']} rew. / {c['unrewarded']} unrew."


_start = time.time()
SENSITIVITY = sensitivity_sums(
    RECORDINGS,
    TRACES,
    EVENTS,
    EVENT_WINDOW,
    (EVOKED_PRE, EVOKED_POST, METRIC_WINDOW),
    PRE_WINDOWS_S,
    SMOOTHING_POINTS,
    subtract_dff_baseline=True,
)
print(
    f"Sensitivity grid: {len(PRE_WINDOWS_S)} windows x {len(SMOOTHING_POINTS)} lengths in {time.time() - _start:.0f} s."
)
sensitivity_counts = sensitivity_counts_table(
    SENSITIVITY["counts"], RECORDINGS, MICE, PRE_WINDOWS_S
)
show_table(sensitivity_counts, "sensitivity_trial_counts")

for view, text, name in (
    ("B_z", "Trial z", "sensitivity_B_z"),
    ("B_dff", "ΔF/F minus pre-window mean (%)", "sensitivity_B_dff"),
):
    fig = sensitivity_figure(
        SENSITIVITY,
        RECORDINGS,
        MICE,
        SMOOTHING_POINTS,
        [(row_label(pre, text, sensitivity_counts), pre, view) for pre in PRE_WINDOWS_S],
        f"Pipeline B Sensitivity: {text} Against Pre-Outcome Window (Rows) And Moving-Mean Length (Columns)",
        EVENT_WINDOW,
    )
    save_figure(fig, name)
    plt.show()


def sensitivity_response_table(
    sensitivity: dict,
    recordings: dict,
    mice: list,
    pre_windows: list,
    smoothings: list,
    views: tuple,
    pre,
    post,
    dff_baseline_subtracted: bool,
) -> pd.DataFrame:
    """Peak response (0-2 s against the -2-0 s mean) of the group mean trace for every pre window, length and view.

    The sensitivity grid is computed with the pre-window mean subtracted from pipeline B's ΔF/F
    (*dff_baseline_subtracted*), which is not the main figures' ``B_dff`` when that switch is off, so the row is
    named ``B_dff_minus_pre_mean``. Pipeline A's session z-score has no pre window: its rows repeat across pre
    windows except for the trials kept, which ``pre_window_affects`` says.
    """
    rows = []
    for view in views:
        shown_name = "B_dff_minus_pre_mean" if view == "B_dff" and dff_baseline_subtracted else view
        affects = (
            "trial set only (session z has no pre window)"
            if view.startswith("A_")
            else "baseline and trial set"
        )
        for pre_window in pre_windows:
            for s in smoothings:
                for cond in CONDITIONS:
                    means = mouse_means_from_sums(
                        sensitivity["sums"][(pre_window, s)], recordings, mice, cond, view
                    )
                    mean, sem, n_mice = across_mice(means)
                    rows.append(
                        {
                            "view": shown_name,
                            "pre_window_affects": affects,
                            "pre_window_s": -pre_window[0],
                            "smoothing_points": s,
                            "condition": cond,
                            "n_mice": n_mice,
                            "peak_response_0_2s": peak_abs_response(
                                sensitivity["rel_time"][pre_window], mean, pre, post
                            ),
                            "median_sem_across_mice": float(np.median(sem)),
                        }
                    )
    return pd.DataFrame(rows)


sensitivity_responses = sensitivity_response_table(
    SENSITIVITY,
    RECORDINGS,
    MICE,
    PRE_WINDOWS_S,
    SMOOTHING_POINTS,
    ("B_z", "B_dff", "A_z"),
    EVOKED_PRE,
    EVOKED_POST,
    dff_baseline_subtracted=True,
)
show_table(sensitivity_responses, "sensitivity_peak_responses")

fig = sensitivity_figure(
    SENSITIVITY,
    RECORDINGS,
    MICE,
    SMOOTHING_POINTS,
    [
        ("Session ΔF/F (%)", DEFAULT_PRE_WINDOW, "A_dff"),
        ("Session z-score", DEFAULT_PRE_WINDOW, "A_z"),
    ],
    f"Pipeline A Sensitivity To Smoothing (Trials As For The {-DEFAULT_PRE_WINDOW[0]:g}-s Pre Window)",
    EVENT_WINDOW,
)
save_figure(fig, "sensitivity_A")
plt.show()

### 7d. Comparing two groups of mice

Groups are defined in the configuration by subject-name prefix (`GROUP_PREFIXES`): **Group 1** = `RT_176_*` and `RT_64_*`,
**Group 2** = `RT_177_*` and `RT_65_*`. Membership is printed below. As the paper does, each mouse's trial mean (the trials of
all its recordings pooled) is computed first, and a group is the **mean ± SEM across its mice**. Each column title gives the n of every line in that column, with its unit (mice or trials; it differs between
rewarded and unrewarded columns), and the legend says what each band is.
Rows are pipeline A and pipeline B; columns are rewarded ΔF/F, rewarded z, unrewarded ΔF/F, unrewarded z. Both pipelines use
the same smoothing here (`DEFAULT_SMOOTHING`) so the rows differ only in the normalisation; pipeline B's pre-event baseline is
`DEFAULT_PRE_WINDOW`. Both are in each figure's title.

The **second figure** compares **`RT_176_L` alone** with Group 2. `RT_176_L` is singled out because it is the only mouse whose
signal-channel outcome response is about twice that of its control channel (section 7b), and whose specificity is clearly
positive (+3.6 dB; section 6.5). Its line is the **mean ± SEM across its trials**, Group 2's line is the **mean ± SEM across mice**,
so the two bands are not the same kind of uncertainty; the legend and title say so.

Each figure is repeated for the **405-nm control channel** (processed as in section 7, without motion correction). Each figure has its own
y-limits, because the control's ΔF/F is several times the corrected signal's (it has no motion correction) and one common scale would flatten the
signal; the sizes are compared per mouse in the matched-scale figures of section 7. A group difference that is also present in the control is not a
sensor difference.

In [ ]:
GROUP_COLORS = {
    "Group 1": "#0072B2",
    "Group 2": "#D55E00",
    SINGLE_MOUSE: "#CC79A7",
}  # colour-blind-safe set


def assign_groups(mice: list[str], prefixes: dict[str, tuple[str, ...]]) -> dict[str, list[str]]:
    """Mice of each group by subject-name prefix, in the order of *mice*.

    Raises:
        ValueError: If a mouse matches more than one group, or a group has no mouse.
    """
    groups = {name: [m for m in mice if m.startswith(tuple(p))] for name, p in prefixes.items()}
    for mouse in mice:
        matching = [name for name, members in groups.items() if mouse in members]
        if len(matching) > 1:
            raise ValueError(f"{mouse} matches several groups: {matching}.")
    empty = [name for name, members in groups.items() if not members]
    if empty:
        raise ValueError(f"no mouse in group(s) {empty}; mice are {mice}.")
    return groups


GROUPS = assign_groups(MICE, GROUP_PREFIXES)
for _name, _members in GROUPS.items():
    print(
        f"{_name} ({len(_members)} mice; prefixes {', '.join(GROUP_PREFIXES[_name])}): {', '.join(_members)}"
    )
_unassigned = [m for m in MICE if not any(m in members for members in GROUPS.values())]
print(f"In neither group: {', '.join(_unassigned) if _unassigned else 'none'}.")
if SINGLE_MOUSE not in MICE or SINGLE_MOUSE_VS not in GROUPS:
    raise ValueError(f"{SINGLE_MOUSE!r} or {SINGLE_MOUSE_VS!r} not found among the mice / groups.")


def mouse_mean_traces(
    real: dict, recordings: dict, mice: list, condition: str, view: str
) -> list[np.ndarray]:
    """Each mouse's mean trace over all its trials (recordings pooled); mice without a trial are left out."""
    traces = []
    for mouse in mice:
        trials = pool_real(real, recordings, mouse, condition, view)
        if len(trials):
            traces.append(trials.mean(axis=0))
    return traces


def group_series(real, recordings, mice, condition, view):
    """Mean, SEM across mice and number of mice of a group."""
    traces = mouse_mean_traces(real, recordings, mice, condition, view)
    mean, sem = mean_and_sem(np.stack(traces))
    return mean, sem, len(traces)


def trial_series(real, recordings, mouse, condition, view):
    """Mean, SEM across trials and number of trials of one mouse."""
    trials = pool_real(real, recordings, mouse, condition, view)
    mean, sem = mean_and_sem(trials)
    return mean, sem, len(trials)


COMPARISON_COLUMNS = (
    ("rewarded", "dff"),
    ("rewarded", "z"),
    ("unrewarded", "dff"),
    ("unrewarded", "z"),
)


def plot_comparison_panel(ax, rel_time, series: list, condition: str, view: str) -> list[int]:
    """Lines of mean ± SEM for each ``(name, colour, unit, compute)`` in *series*; returns their n."""
    counts = []
    for _, colour, _, compute in series:
        mean, sem, n = compute(condition, view)
        ax.plot(rel_time, mean, color=colour, lw=1.3)
        ax.fill_between(rel_time, mean - sem, mean + sem, color=colour, alpha=0.25, lw=0)
        counts.append(n)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.axhline(0, color="0.8", lw=0.6)
    ax.set_ylabel(VIEW_LABELS[view][1], fontsize=8)
    return counts


def comparison_figure(rel_time, series, title, x_limits):
    """Two rows (pipeline A, B) by four columns (rewarded/unrewarded x ΔF/F, z) of group lines.

    One legend below the figure names each line and what its band is; each column title gives the n of
    every line in that column, with units (a column's n depends on its outcome). Rewarded and unrewarded
    panels of the same row and quantity share y-limits. Returns the figure.
    """
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, constrained_layout=True)
    counts = {}
    for r, pipeline in enumerate(("A", "B")):
        for c, (condition, kind) in enumerate(COMPARISON_COLUMNS):
            counts[(r, c)] = plot_comparison_panel(
                axes[r, c], rel_time, series, condition, f"{pipeline}_{kind}"
            )
        axes[r, 0].set_ylabel(f"Pipeline {pipeline}\n{axes[r, 0].get_ylabel()}", fontsize=8)
    for c, (condition, kind) in enumerate(COMPARISON_COLUMNS):
        if counts[(0, c)] != counts[(1, c)]:
            raise ValueError(
                f"pipelines A and B use different trials in column {c}: {counts[(0, c)]} / {counts[(1, c)]}"
            )
        n_text = "; ".join(
            f"{name.split(',')[0]}: {n} {unit}"
            for (name, _, unit, _), n in zip(series, counts[(0, c)], strict=True)
        )
        axes[0, c].set_title(
            f"{condition.capitalize()}: {'ΔF/F' if kind == 'dff' else 'Z-Score'}\n{n_text}",
            fontsize=9,
        )
        axes[-1, c].set_xlabel("Time From Outcome (s)")
        axes[-1, c].set_xlim(x_limits)
    for r in range(2):
        for kind_columns in ((0, 2), (1, 3)):
            low = min(axes[r, c].get_ylim()[0] for c in kind_columns)
            high = max(axes[r, c].get_ylim()[1] for c in kind_columns)
            for c in kind_columns:
                axes[r, c].set_ylim(low, high)
    fig.legend(
        handles=[plt.Line2D([], [], color=colour, label=name) for name, colour, _, _ in series],
        loc="outside lower center",
        ncol=len(series),
        frameon=False,
    )
    fig.suptitle(title, fontsize=12)
    return fig


GROUP_SETTINGS = ViewSettings(
    DEFAULT_PRE_WINDOW, DEFAULT_SMOOTHING, DEFAULT_SMOOTHING, B_EVENT_LEVEL["subtract_dff_baseline"]
)
GROUP_REAL = {
    channel: build_real(
        RECORDINGS,
        PROCESSED,
        traces,
        EVENTS,
        EXTRACTION_WINDOW,
        GROUP_SETTINGS,
        channel_views=False,
    )
    for channel, traces in (("signal", TRACES), ("control", TRACES_CONTROL))
}
GROUP_SETTINGS_TEXT = f"pipeline B baseline {-GROUP_SETTINGS.pre_window[0]:g} s; both pipelines {smoothing_text(DEFAULT_SMOOTHING)}"


def group_series_list(
    real: dict, names: tuple, recordings: dict, groups: dict, single_mouse: str, colours: dict
) -> list:
    """Series for :func:`comparison_figure`: the groups named in *names* (members in *groups*), or *single_mouse* alone."""
    series = []
    for name in names:
        if name == single_mouse:
            compute = partial(trial_series, real, recordings, single_mouse)
            series.append(
                (f"{single_mouse}, mean ± SEM across trials", colours[name], "trials", compute)
            )
        else:
            compute = partial(group_series, real, recordings, groups[name])
            series.append((f"{name}, mean ± SEM across mice", colours[name], "mice", compute))
    return series


def check_group_functions() -> None:
    """Mouse-level averaging weights each mouse once; groups are found by prefix and may not overlap."""
    assert assign_groups(["A_1_x", "A_2_x", "B_1_x"], {"g1": ("A_1_",), "g2": ("B_",)}) == {
        "g1": ["A_1_x"],
        "g2": ["B_1_x"],
    }
    for bad in ({"g1": ("A",), "g2": ("A_1",)}, {"g1": ("Z",)}):
        try:
            assign_groups(["A_1_x"], bad)
        except ValueError:
            continue
        raise AssertionError(f"{bad} must be refused")

    class _Rec:  # the two attributes pool_real reads
        def __init__(self, subject):
            self.subject = subject

    recordings = {"r1": _Rec("m1"), "r2": _Rec("m2")}
    n_time = 3
    real = {
        ("r1", "rewarded"): {
            "views": {"v": np.full((9, n_time), 1.0)}
        },  # mouse m1: nine trials, mean 1
        ("r2", "rewarded"): {
            "views": {"v": np.full((1, n_time), 5.0)}
        },  # mouse m2: one trial, mean 5
    }
    mean, sem, n = group_series(real, recordings, ["m1", "m2"], "rewarded", "v")
    np.testing.assert_allclose(
        mean, 3.0
    )  # mice weighted equally; pooling the trials would give 1.4
    np.testing.assert_allclose(sem, np.std([1.0, 5.0], ddof=1) / np.sqrt(2))
    assert n == 2
    mean, sem, n = trial_series(real, recordings, "m1", "rewarded", "v")
    np.testing.assert_allclose(mean, 1.0)
    np.testing.assert_allclose(sem, 0.0)
    assert n == 9


check_group_functions()


def comparison_metrics(series: list, rel_time: np.ndarray, pre, post) -> pd.DataFrame:
    """Peak response (0-2 s against the -2-0 s mean) of each line of a comparison figure, panel by panel."""
    rows = []
    for name, _, unit, compute in series:
        for pipeline in ("A", "B"):
            for condition, kind in COMPARISON_COLUMNS:
                mean, _, n = compute(condition, f"{pipeline}_{kind}")
                rows.append(
                    {
                        "line": name.split(",")[0],
                        "n": f"{n} {unit}",
                        "pipeline": pipeline,
                        "outcome": condition,
                        "quantity": kind,
                        "peak_response_0_2s": peak_abs_response(rel_time, mean, pre, post),
                    }
                )
    return pd.DataFrame(rows)


metrics = []
# Each figure keeps its own y-limits (shared by its rewarded and unrewarded panels): the control's ΔF/F is
# several times the corrected signal's, and one common scale would flatten the signal.
for name, (group_names, title) in {
    "group_comparison": (("Group 1", "Group 2"), "Group 1 Against Group 2"),
    "single_mouse_comparison": (
        (SINGLE_MOUSE, SINGLE_MOUSE_VS),
        f"{SINGLE_MOUSE} Alone Against {SINGLE_MOUSE_VS}",
    ),
}.items():
    for channel, label in (
        ("signal", "Signal (Motion-Corrected)"),
        ("control", "405-nm Control, Not Motion-Corrected"),
    ):
        series = group_series_list(
            GROUP_REAL[channel], group_names, RECORDINGS, GROUPS, SINGLE_MOUSE, GROUP_COLORS
        )
        fig = comparison_figure(
            REL_TIME,
            series,
            f"{label}: {title} ({GROUP_SETTINGS_TEXT})\n"
            "Own Y-Limits, Not Comparable In Size Between The Signal And Control Figures",
            EVENT_WINDOW,
        )
        save_figure(fig, f"{name}_{channel}")
        plt.show()
        metrics.append(
            comparison_metrics(series, REL_TIME, EVOKED_PRE, EVOKED_POST).assign(
                comparison=name, channel=channel
            )
        )
show_table(pd.concat(metrics, ignore_index=True), "group_comparison_metrics")

### 7e. Responses by reward size (number of reward pulses)

Sections 7 and 7d split trials only into rewarded and unrewarded. Here the rewarded trials are split further by the
number of reward pulses delivered (`num_rew_pulses` of the outcome print: 1 to 4), and the unrewarded trials (0 pulses)
are kept as the reference. The categories are the pulse counts that occur in the data. Blocked-task sessions use 0, 1 and
4 pulses; the two mice that also ran drifting sessions (`RT_177_N`, `RT_177_LR`) add 2 and 3 pulses, so **the 2- and 3-pulse
lines come from those two mice only**.

**Configuration.** `PULSE_PHASES = ("task",)`: the 2-pulse trials of the `port_finding` phase are left out by default,
because that phase delivers 2 pulses at every arm, so those trials say nothing about reward size and they would
add a second, unrelated source of 2-pulse trials; set `("task", "port_finding")` to include them. `MIN_TRIALS_PER_PULSE = 5`:
a mouse's pulse category is drawn, enters group means and enters the correlation only with at least that many usable
trials (after the window exclusions of section 7); what was dropped is listed below.

**Alignment.** As in section 7: 0 pulses at the no-reward cue onset, 1 to 4 pulses at the onset of the first reward
pulse. **These are two different events, so time zero is not the same thing for 0 pulses and for 1 or more.** Comparisons
among the rewarded categories share one zero point.

**Normalisation.** Each recording is normalised exactly as in section 7 (pipeline A: session ΔF/F and session z-score;
pipeline B: each trial's own pre-event baseline, `DEFAULT_PRE_WINDOW`, and `DEFAULT_SMOOTHING`), and only afterwards are
the trials split by pulse count (`split_by_pulses`, `src/photometry_analysis/pulse_groups.py`, tested). The split therefore
cannot change any trial's trace: a category is a selection of the rows already shown in section 7. The per-mouse
figures use the section-7 settings; the group figures use the section-7d settings (both pipelines smoothed by
`DEFAULT_SMOOTHING`).

In [ ]:
def trial_responses(
    trials: np.ndarray, rel_time: np.ndarray, post: tuple[float, float], pre: tuple[float, float]
) -> np.ndarray:
    """Per-trial evoked response: mean over the *post* interval minus mean over the *pre* interval."""
    return trials[:, interval_mask(rel_time, post)].mean(axis=1) - trials[
        :, interval_mask(rel_time, pre)
    ].mean(axis=1)

In [ ]:
ALL_TRIALS = (
    "trial"  # one condition for every trial, so the pulse split happens after the windows are cut
)
PULSE_VIEWS = ("A_dff", "A_z", "B_dff", "B_z")


def label_all_trials(events: pd.DataFrame) -> pd.DataFrame:
    """*events* with every trial in the single condition :data:`ALL_TRIALS` (the pulse count stays a column)."""
    return events.assign(condition=ALL_TRIALS)


PULSE_EVENTS = {  # from the table section 7 uses (EVENTS), so every trial here is also a section-7 trial
    rid: label_all_trials(ev[ev["phase"].isin(PULSE_PHASES)].reset_index(drop=True))
    for rid, ev in EVENTS.items()
}
PULSE_CATEGORIES = pulse_categories(
    np.concatenate([ev["num_rew_pulses"].to_numpy() for ev in PULSE_EVENTS.values()])
)
print(f"Trials from phases {PULSE_PHASES}; pulse categories found: {PULSE_CATEGORIES}.")

# (channel, settings) of each data set: per-mouse figures use the section-7 settings, group figures the 7d ones.
PULSE_DATASETS = {
    ("signal", "main"): (TRACES, MAIN_SETTINGS),
    ("control", "main"): (TRACES_CONTROL, MAIN_SETTINGS),
    ("signal", "group"): (TRACES, GROUP_SETTINGS),
}
PULSE_REAL = {
    key: build_real(
        RECORDINGS,
        PROCESSED,
        traces,
        PULSE_EVENTS,
        EXTRACTION_WINDOW,
        settings,
        channel_views=False,
        conditions=(ALL_TRIALS,),
    )
    for key, (traces, settings) in PULSE_DATASETS.items()
}


def pool_pulse_windows(
    real: dict,
    recordings: dict,
    mouse: str,
    categories: list[int],
    condition: str,
    views: tuple[str, ...],
) -> dict[int, dict[str, np.ndarray]]:
    """The usable trials of *mouse* (its recordings pooled) split by pulse count: ``{pulses: {view: windows}}``.

    *real* maps ``(recording id, condition)`` to :func:`real_condition_windows` output; *condition* is the
    label that holds every trial and *views* the views to keep.
    """
    parts = [real[(rid, condition)] for rid, rec in recordings.items() if rec.subject == mouse]
    pulses = np.concatenate([part["pulses"] for part in parts])
    pooled = {v: np.concatenate([part["views"][v] for part in parts]) for v in views}
    return split_by_pulses(pooled, pulses, categories)


PULSE_SPLIT = {
    key: {
        mouse: pool_pulse_windows(
            real, RECORDINGS, mouse, PULSE_CATEGORIES, ALL_TRIALS, PULSE_VIEWS
        )
        for mouse in MICE
    }
    for key, real in PULSE_REAL.items()
}


def n_trials(windows: dict[str, np.ndarray]) -> int:
    """Number of trials in one category's ``{view: windows}`` (every view has the same trials)."""
    return len(next(iter(windows.values())))


def drawn_categories(split_mouse: dict, min_trials: int) -> list[int]:
    """Categories of one mouse with at least *min_trials* usable trials."""
    return categories_with_min_trials({c: n_trials(w) for c, w in split_mouse.items()}, min_trials)


def pulse_count_table(split: dict, min_trials: int) -> pd.DataFrame:
    """Usable trials per mouse and pulse category (categories a mouse never had are omitted), with their status."""
    rows = [
        {
            "mouse": mouse,
            "pulses": c,
            "n_trials": n_trials(w),
            "status": "drawn" if n_trials(w) >= min_trials else f"dropped (< {min_trials} trials)",
        }
        for mouse, by_category in split.items()
        for c, w in by_category.items()
        if n_trials(w) > 0
    ]
    return pd.DataFrame(rows)


def pulse_label(category: int) -> str:
    """``"0 pulses (unrewarded)"``, ``"1 pulse"``, ``"4 pulses"``."""
    if category == 0:
        return "0 pulses (unrewarded)"
    return f"{category} pulse" + ("" if category == 1 else "s")


def dropped_categories(counts: pd.DataFrame) -> set[tuple[str, int]]:
    """``(mouse, pulses)`` of the categories below the minimum trial count."""
    below = counts[counts["status"] != "drawn"]
    return set(zip(below["mouse"], below["pulses"], strict=True))


def describe_drops(counts: pd.DataFrame, min_trials: int) -> str:
    """One sentence naming the dropped categories with their trial counts."""
    below = counts[counts["status"] != "drawn"]
    named = "; ".join(
        f"{r.mouse} {pulse_label(r.pulses)} (n = {r.n_trials})" for r in below.itertuples()
    )
    return f"Dropped for fewer than {min_trials} usable trials: {named or 'none'}."


PULSE_COUNTS = {
    key: pulse_count_table(split, MIN_TRIALS_PER_PULSE) for key, split in PULSE_SPLIT.items()
}
for key, counts in PULSE_COUNTS.items():
    print(f"{key[0]} channel, {key[1]} settings. {describe_drops(counts, MIN_TRIALS_PER_PULSE)}")
drops = {key: dropped_categories(counts) for key, counts in PULSE_COUNTS.items()}
print(
    "The same categories are dropped in every data set."
    if len({frozenset(d) for d in drops.values()}) == 1
    else "The data sets drop different categories (see above); the figures show each its own."
)
pulse_counts = PULSE_COUNTS[("signal", "main")]
show_table(
    pulse_counts.pivot(index="mouse", columns="pulses", values="n_trials")
    .fillna(0)
    .astype(int)
    .reset_index(),
    "pulse_trial_counts",
)

#### Per mouse

Rows are mice; ΔF/F and z-score columns; one line per pulse category (mean ± SEM across trials, n trials in the legend to the right of the z-score column). Pipeline A, then pipeline B, for the signal; then the same two figures for the 405-nm control channel, each panel on its own y-limits.

In [ ]:
UNREWARDED_COLOR = CONDITION_COLORS["unrewarded"]


def pulse_colours(categories: list[int]) -> dict[int, str]:
    """0 pulses in the unrewarded red; 1 to 4 pulses from teal to dark blue (yellow-green-blue ramp), spaced by pulse count.

    The shade depends only on the count, so a category keeps its colour in every figure; the ramp turns
    from green-teal to blue, so neighbouring counts (3 and 4 pulses) differ in hue as well as lightness.
    """
    rewarded = [c for c in categories if c > 0]
    low, high = (min(rewarded), max(rewarded)) if rewarded else (0, 1)
    ramp = plt.get_cmap("YlGnBu")
    shades = {c: ramp(0.5 + 0.5 * (c - low) / max(high - low, 1)) for c in rewarded}
    return {c: UNREWARDED_COLOR if c == 0 else shades[c] for c in categories}


PULSE_COLOURS = pulse_colours(PULSE_CATEGORIES)


# A line to draw: (pulse category, mean, SEM, legend label, line style).
PulseLine = tuple[int, np.ndarray, np.ndarray, str, str]


def mouse_lines(split_mouse: dict, view: str, min_trials: int) -> list[PulseLine]:
    """One mouse's lines: mean ± SEM across trials for each drawn category, n trials in the label."""
    lines = []
    for c in drawn_categories(split_mouse, min_trials):
        trials = split_mouse[c][view]
        mean, sem = mean_and_sem(trials)
        lines.append((c, mean, sem, f"{pulse_label(c)}, n = {len(trials)}", "-"))
    return lines


def short_name(mouse: str) -> str:
    """``RT_177_N`` -> ``177_N`` (legend space is tight)."""
    return mouse.removeprefix("RT_")


def group_lines(
    split: dict, mice: list[str], view: str, min_trials: int, categories: list[int]
) -> list[PulseLine]:
    """A group's lines: mean ± SEM across mice of each mouse's trial mean, for each category.

    Only mice with at least *min_trials* trials in a category enter it. A category present in fewer than all the
    mice is dashed and names its mice, so the reader sees which lines average different animals.
    """
    lines = []
    for c in categories:
        members = [m for m in mice if c in drawn_categories(split[m], min_trials)]
        if not members:
            continue
        mean, sem = mean_and_sem(np.stack([split[m][c][view].mean(axis=0) for m in members]))
        partial_group = len(members) < len(mice)
        names = f": {', '.join(short_name(m) for m in members)}" if partial_group else ""
        label = f"{pulse_label(c)}, n = {len(members)}/{len(mice)} mice{names}"
        lines.append((c, mean, sem, label, "--" if partial_group else "-"))
    return lines


def draw_pulse_lines(
    ax,
    rel_time,
    lines: list[PulseLine],
    colours: dict,
    ylabel: str,
    legend: bool = True,
    outside: bool = False,
) -> None:
    """Draw the lines (mean with an SEM band) into *ax*, mark the event, and add a legend if *legend*.

    *outside* puts the legend to the right of the axis, where it cannot hide the traces."""
    for c, mean, sem, label, style in lines:
        ax.plot(rel_time, mean, color=colours[c], ls=style, lw=1.2, label=label)
        ax.fill_between(rel_time, mean - sem, mean + sem, color=colours[c], alpha=0.18, lw=0)
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.axhline(0, color="0.8", lw=0.6)
    ax.set_ylabel(ylabel)
    if legend and lines:
        place = (
            {"loc": "center left", "bbox_to_anchor": (1.02, 0.5)}
            if outside
            else {"loc": "upper right"}
        )
        ax.legend(frameon=False, fontsize=7, **place)


def plot_pulse_cell(ax, mouse, view, ylabel, *, split, rel_time, min_trials, colours):
    """Per-mouse cell: one line per drawn pulse category; the legend sits right of the z-score column only."""
    lines = mouse_lines(split[mouse], view, min_trials)
    draw_pulse_lines(ax, rel_time, lines, colours, ylabel, legend=view.endswith("_z"), outside=True)


def solid_ylim(
    lines_by_entity: list[list[PulseLine]], label: str, margin: float = 0.08
) -> tuple[float, float]:
    """Common y-limits from the solid lines (mean ± SEM) of every entity, widened by *margin* of their range.

    Dashed lines (categories present in only some mice) are left out so a few noisy animals do not set the scale;
    they may extend past the axes. A missing SEM (a single member) counts as zero, so it cannot make the limits NaN.

    Raises:
        ValueError: If no entity has a solid line; *label* names the panel.
    """
    solid = [line for lines in lines_by_entity for line in lines if line[4] == "-"]
    if not solid:
        raise ValueError(f"no solid line to set the y-limits of {label}.")
    spreads = [np.nan_to_num(sem) for _, _, sem, _, _ in solid]
    low = min(np.nanmin(mean - s) for (_, mean, _, _, _), s in zip(solid, spreads, strict=True))
    high = max(np.nanmax(mean + s) for (_, mean, _, _, _), s in zip(solid, spreads, strict=True))
    pad = margin * (high - low)
    return low - pad, high + pad


def check_solid_ylim() -> None:
    """Dashed lines do not set the limits; a NaN SEM does not poison them; no solid line is refused."""
    t = np.linspace(0, 1, 5)
    solid = (1, t, np.full(5, 0.5), "a", "-")
    dashed = (2, 100 * t, np.full(5, 50.0), "b", "--")
    lo, hi = solid_ylim([[solid, dashed]], "x", margin=0.0)
    assert (lo, hi) == (-0.5, 1.5), (lo, hi)
    single = (1, t, np.full(5, np.nan), "c", "-")
    assert solid_ylim([[single]], "x", margin=0.0) == (0.0, 1.0)
    try:
        solid_ylim([[dashed]], "the panel")
    except ValueError as error:
        assert "the panel" in str(error)
    else:
        raise AssertionError("a panel without a solid line must be refused")


check_solid_ylim()


def pulse_comparison_figure(rel_time, entities: list, title: str, x_limits, colours: dict):
    """Two rows (pipeline A, B) by (ΔF/F, z) columns for each entity, one line per pulse category.

    *entities* is a list of ``(name, lines_for)`` where ``lines_for(view)`` returns the lines of that entity in
    that view. Panels of the same row and quantity share y-limits (from the solid lines only), so entities compare directly. The legend
    (with n) is right of the pipeline-A z-score panel of each entity; pipeline B uses the same trials. Returns the figure.
    """
    n_cols = 2 * len(entities)
    fig, axes = plt.subplots(
        2,
        n_cols,
        figsize=(4.4 * n_cols + 3 * len(entities), 7),
        sharex=True,
        constrained_layout=True,
    )
    for e, (name, lines_for) in enumerate(entities):
        for r, pipeline in enumerate(("A", "B")):
            for k, kind in enumerate(("dff", "z")):
                ax = axes[r, 2 * e + k]
                view = f"{pipeline}_{kind}"
                draw_pulse_lines(
                    ax,
                    rel_time,
                    lines_for(view),
                    colours,
                    VIEW_LABELS[view][1],
                    legend=(r == 0 and k == 1),
                    outside=True,
                )
                if r == 0:
                    ax.set_title(f"{name}\n{'ΔF/F' if kind == 'dff' else 'Z-Score'}", fontsize=9)
        for ax in axes[-1, 2 * e : 2 * e + 2]:
            ax.set_xlabel("Time From Outcome (s)")
            ax.set_xlim(x_limits)
    for r in range(2):
        axes[r, 0].set_ylabel(f"Pipeline {'AB'[r]}\n{axes[r, 0].get_ylabel()}", fontsize=8)
        for k in range(2):
            view = f"{'AB'[r]}_{('dff', 'z')[k]}"
            names = " and ".join(name for name, _ in entities)
            limits = solid_ylim(
                [lines_for(view) for _, lines_for in entities], f"{view} of {names}"
            )
            for e in range(len(entities)):
                axes[r, 2 * e + k].set_ylim(limits)
    fig.suptitle(title, fontsize=12)
    return fig


PULSE_NOTE = (
    "0 pulses aligned to the no-reward cue, 1-4 pulses to reward onset; solid = at least "
    f"{MIN_TRIALS_PER_PULSE} usable trials in every mouse of the group, dashed = fewer mice (named in the legend);\n"
    "y-limits from the solid lines, dashed lines may extend past the axes"
)
for channel, scale_note in (
    ("signal", ""),
    ("control", "405-nm Control, Not Motion-Corrected, Own Y-Limits\n"),
):
    for pipeline, views in (("A", ("A_dff", "A_z")), ("B", ("B_dff", "B_z"))):
        fig = response_grid(
            views,
            f"{scale_note}Pipeline {pipeline}: Responses By Number Of Reward Pulses "
            f"(Mean ± SEM Across Trials; {settings_text(MAIN_SETTINGS, pipeline)})",
            partial(
                plot_pulse_cell,
                split=PULSE_SPLIT[(channel, "main")],
                rel_time=REL_TIME,
                min_trials=MIN_TRIALS_PER_PULSE,
                colours=PULSE_COLOURS,
            ),
            MICE,
            EVENT_WINDOW,
        )
        suffix = "" if channel == "signal" else "_control_own_scale"
        save_figure(fig, f"pulses_per_mouse_pipeline_{pipeline}{suffix}")
        plt.show()

#### Groups

Group 1 against Group 2, and `RT_176_L` against Group 2 (signal channel, section-7d settings).

In [ ]:
GROUP_PULSE_SPLIT = PULSE_SPLIT[("signal", "group")]


def group_entity(
    name: str, split: dict, groups: dict, min_trials: int, categories: list[int]
) -> tuple:
    """``(legend title, lines_for)`` for a group of mice: mean ± SEM across mice."""
    lines_for = partial(
        group_lines, split, groups[name], min_trials=min_trials, categories=categories
    )
    return f"{name} (mean ± SEM across mice)", lines_for


def single_mouse_entity(mouse: str, split: dict, min_trials: int) -> tuple:
    """``(legend title, lines_for)`` for one mouse: mean ± SEM across its trials."""
    return f"{mouse} (mean ± SEM across trials)", partial(
        mouse_lines, split[mouse], min_trials=min_trials
    )


def group_of(name: str) -> tuple:
    """:func:`group_entity` with this notebook's split, groups, minimum and categories."""
    return group_entity(name, GROUP_PULSE_SPLIT, GROUPS, MIN_TRIALS_PER_PULSE, PULSE_CATEGORIES)


for name, entities, title in (
    (
        "pulses_group_comparison",
        [group_of("Group 1"), group_of("Group 2")],
        "Group 1 Against Group 2",
    ),
    (
        "pulses_single_mouse_comparison",
        [
            single_mouse_entity(SINGLE_MOUSE, GROUP_PULSE_SPLIT, MIN_TRIALS_PER_PULSE),
            group_of(SINGLE_MOUSE_VS),
        ],
        f"{SINGLE_MOUSE} Alone Against {SINGLE_MOUSE_VS}",
    ),
):
    fig = pulse_comparison_figure(
        REL_TIME,
        entities,
        f"Signal (Motion-Corrected), By Number Of Reward Pulses: {title}\n{GROUP_SETTINGS_TEXT}\n{PULSE_NOTE}",
        EVENT_WINDOW,
        PULSE_COLOURS,
    )
    save_figure(fig, name)
    plt.show()

#### Response size per mouse and category

Mean z-score response (0 to 2 s minus -2 to 0 s) per pulse category and the Spearman correlation with pulse count over the rewarded trials.

In [ ]:
def pulse_response_row(
    split_mouse: dict, view: str, rel_time: np.ndarray, post, pre, min_trials: int
) -> dict:
    """One mouse and pipeline: mean z response (*post* minus *pre*) and n trials per category, and the correlation.

    Categories below *min_trials* keep their n but get a NaN mean and stay out of the correlation, which uses the
    drawn rewarded categories only.
    """
    drawn = drawn_categories(split_mouse, min_trials)
    responses = {
        c: trial_responses(w[view], rel_time, post, pre)
        for c, w in split_mouse.items()
        if n_trials(w)
    }
    row = {}
    for c, w in split_mouse.items():
        row[f"z_{c}"] = responses[c].mean() if c in drawn else np.nan
        row[f"n_{c}"] = n_trials(w)
    pulses = np.concatenate([np.full(len(responses[c]), c) for c in drawn])
    all_responses = np.concatenate([responses[c] for c in drawn])
    corr = pulse_response_correlation(all_responses, pulses)
    return row | {
        "rho": corr["rho"],
        "rho_n_trials": corr["n_trials"],
        "rho_n_categories": corr["n_categories"],
    }


def pulse_response_table(
    split: dict, mice: list[str], rel_time, post, pre, min_trials
) -> pd.DataFrame:
    """:func:`pulse_response_row` for every mouse and both pipelines' z-scores."""
    return pd.DataFrame(
        [
            {"mouse": mouse, "pipeline": pipeline}
            | pulse_response_row(split[mouse], f"{pipeline}_z", rel_time, post, pre, min_trials)
            for mouse in mice
            for pipeline in ("A", "B")
        ]
    )


pulse_responses = pulse_response_table(
    PULSE_SPLIT[("signal", "main")], MICE, REL_TIME, EVOKED_POST, EVOKED_PRE, MIN_TRIALS_PER_PULSE
)
show_table(pulse_responses, "pulse_responses")

**What each figure shows.**

- `pulses_per_mouse_pipeline_A` / `_B`: one row per mouse, ΔF/F and z-score columns, one line per pulse category (0 pulses in
  red; 1 to 4 pulses from teal to dark blue), mean ± SEM across that mouse's trials, with n trials in the legend. A category
  below `MIN_TRIALS_PER_PULSE` trials is not drawn.
- `pulses_per_mouse_pipeline_A_control_own_scale` / `pulses_per_mouse_pipeline_B_control_own_scale`: the same split of the 405-nm control channel (processed in parallel
  without motion correction, as in section 7), each panel on its own y-limits. A reward-size effect that is as large in the control
  is not a sensor response.
- `pulses_group_comparison`: Group 1 and Group 2 side by side (rows: pipeline A and B; columns: ΔF/F and z-score of each group), mean
  ± SEM across mice of each mouse's trial mean, n mice per category in the legend. Panels of the same row and quantity share
  y-limits, set by the solid lines only (dashed lines may extend past the axes). **Solid** means at least
  `MIN_TRIALS_PER_PULSE` usable trials in every mouse of the group; a line from fewer mice is **dashed** and names its mice. In
  Group 2 three lines are dashed: 1 pulse (`RT_65_LR` has too few 1-pulse trials, so 4 of 5 mice) and 2 and 3 pulses (`RT_177_N`
  and `RT_177_LR` only). Dashed lines therefore average different animals from the solid ones and from Group 1.
- `pulses_single_mouse_comparison`: `RT_176_L` (mean ± SEM across its trials) beside Group 2 (mean ± SEM across mice), same layout. The two bands are
  different kinds of uncertainty.
- Table `pulse_responses` (CSV in the table folder): per mouse and pipeline, the mean z-score response (0 to 2 s minus -2 to 0 s, the
  evoked-response definition of section 8) per pulse category, `n_<pulses>` trials, and `rho`, the Spearman correlation
  of the trial response with the pulse count over the rewarded trials of the drawn categories (`rho_n_trials`, `rho_n_categories`).
  No p-value is given: trials of a session are not independent, so the trial count is not an effective sample size.
  `rho` is NaN where a mouse has fewer than two drawn rewarded categories.

**Caveats.**

- **Two different zero points.** The 0-pulse line is aligned to the no-reward cue, the others to reward onset. A difference
  between 0 pulses and the rest mixes reward size with the event type (cue sound and light against pump and port). Reward
  size proper is the comparison among 1 to 4 pulses.
- **Reward size is confounded with arm and value.** Pulse count follows the arm's value, so a larger response to more
  pulses may reflect value, the arm chosen, or the preceding choice history, not the amount alone. In the blocked sessions
  only 1 and 4 pulses occur, so they are a two-point comparison.
- **Time at the port.** More pulses take longer to deliver (4 pulses take three more 300-ms intervals, about 0.9 s, than 1 pulse),
  so the mouse stays at the port longer, with more movement and possibly more movement artefact in the window after
  reward onset. The control-channel figures show whether the split appears in the 405-nm channel too; the motion-corrected
  signal is only as good as that correction.
- **Different mice in different lines.** Categories 2 and 3 exist for two mice only, and the 1-pulse line of Group 2 lacks `RT_65_LR` (below the minimum); the mouse-level means of the lines of a
  group figure are therefore not over the same animals. Per-mouse figures and the table do not have this problem.
- **Dependence.** Trials of a session are correlated; the SEM across trials and the trial counts overstate the independent
  evidence.

## 8. Quality control, continued: evoked SNR and the per-mouse ranking

**Evoked response** of a trial = mean over 0-2 s minus mean over −2-0 s. It is computed from the **rewarded** trials of each recording, in session-z units
for pipeline A and in smoothed trial-z units for pipeline B (with the settings of the main figures). One number describes a recording:

- **`d_prime` (the measure used for ranking).** The mean response divided by its SD across trials. It is a standardised effect size: it does **not** grow
  with the number of trials, so a recording is not favoured for having run more trials. Its sign says whether the response is a rise (+) or a dip (−).

Recordings with fewer than `EVOKED_MIN_TRIALS` rewarded trials get no evoked SNR (flag `few_trials`).

**Per mouse.** Each recording's *absolute* d′ is computed first, and the mouse's value is the **median of |d′| over its recordings**, so a rise in one session and a dip
in another do **not** cancel (a signed median of two recordings with d′ of −2 and +2 would be 0). The recordings' signs are shown next to it (for example "−/+" for a mouse with
one dip and one rise), because a mouse whose sessions disagree in sign is not responding consistently, whatever its |d′|.
The primary ranking is by the event-independent spectral SNR; the second ranking is by median |d′| (pipeline A). They answer different
questions (how clean is the trace vs. how strongly does it follow the outcome), so they need not agree.

In [ ]:
def evoked_snr(responses: np.ndarray) -> dict:
    """Mean response and d′ (mean over SD across trials, the effect size)."""
    return {"mean_response": responses.mean(), "d_prime": responses.mean() / responses.std(ddof=1)}


def check_evoked_snr_on_synthetic_data() -> pd.DataFrame:
    """A known response (mean m, SD 0.5) must give d′ near m/0.5 at any trial count."""
    rng = np.random.default_rng(4)
    rows = []
    for n_trials in (50, 200):
        for mean_response in (0.5, 1.0, 2.0):
            out = evoked_snr(rng.normal(mean_response, 0.5, n_trials))
            expected_d = mean_response / 0.5
            assert abs(out["d_prime"] / expected_d - 1) < 0.25, out
            rows.append((n_trials, mean_response, out["d_prime"], expected_d))
    table = pd.DataFrame(rows, columns=["n_trials", "true_mean", "d_prime", "expected_d"])
    by_n = table.groupby("n_trials")["d_prime"].mean()
    assert abs(by_n.loc[200] / by_n.loc[50] - 1) < 0.15, by_n  # d′ does not depend on n
    return table


display(check_evoked_snr_on_synthetic_data())


def evoked_row(rec, inputs: dict, rel_time: np.ndarray, post, pre, min_trials: int) -> dict:
    """Evoked SNR columns of one recording for both pipelines.

    *inputs* holds the recording's rewarded data: ``n_used`` trials and the ``views`` of the real trials.
    """
    row = {"n_rewarded_trials": inputs["n_used"]}
    for view, name in (("A_z", "a"), ("B_z", "b")):
        keys = ("mean_response", "d_prime")
        if inputs["n_used"] < min_trials:
            row |= {f"{name}_{k}": np.nan for k in keys}
            continue
        out = evoked_snr(trial_responses(inputs["views"][view], rel_time, post, pre))
        row |= {f"{name}_{k}": out[k] for k in keys}
    return row


def evoked_qc(recordings: dict, real: dict, thresholds: dict, rel_time, post, pre) -> pd.DataFrame:
    """Evoked-SNR table for every recording, with flags."""
    inputs = {
        rid: {
            "n_used": real[(rid, "rewarded")]["n_used"],
            "views": real[(rid, "rewarded")]["views"],
        }
        for rid in recordings
    }
    table = per_recording(
        recordings,
        evoked_row,
        inputs,
        rel_time=rel_time,
        post=post,
        pre=pre,
        min_trials=thresholds["min_evoked_trials"],
    )
    return attach_flags(
        table, {"few_trials": table["n_rewarded_trials"] < thresholds["min_evoked_trials"]}
    )


qc_evoked = evoked_qc(RECORDINGS, REAL, QC_THRESHOLDS, REL_TIME, EVOKED_POST, EVOKED_PRE)
show_table(qc_evoked, "qc_8_evoked")

In [ ]:
def split_flags(flags: pd.Series) -> list[str]:
    """Flag names from a column of comma-separated flag strings."""
    return [f for cell in flags.fillna("") for f in cell.split(", ") if f]


def sign_string(values: pd.Series) -> str:
    """The signs of the finite *values* in order, e.g. ``"−/+"`` (empty if none is finite)."""
    return "/".join("−" if v < 0 else "+" for v in values if np.isfinite(v))


def evoked_by_mouse(evoked: pd.DataFrame) -> pd.DataFrame:
    """Per mouse: median over recordings of |d′| (the ranking basis), and the recordings' signs.

    The median is taken of the absolute values, so recordings with opposite-sign responses do not cancel.
    """
    rows = []
    for mouse, group in evoked.groupby("mouse", sort=False):
        rows.append(
            {
                "mouse": mouse,
                "abs_d_prime_a": group["a_d_prime"].abs().median(),
                "signs_a": sign_string(group["a_d_prime"]),
                "abs_d_prime_b": group["b_d_prime"].abs().median(),
                "signs_b": sign_string(group["b_d_prime"]),
            }
        )
    return pd.DataFrame(rows)


def check_evoked_by_mouse_on_synthetic_data() -> pd.DataFrame:
    """Opposite-sign recordings must not cancel, and mice are ranked by |d′|."""
    evoked = pd.DataFrame(
        {
            "mouse": ["M1", "M1", "M2", "M2", "M3"],
            "a_d_prime": [-2.0, 2.0, 0.5, 0.5, 1.0],
            "b_d_prime": [-1.0, 1.0, 0.2, 0.2, 0.4],
        }
    )
    out = evoked_by_mouse(evoked).set_index("mouse")
    assert out.loc["M1", "abs_d_prime_a"] == 2.0 and out.loc["M1", "signs_a"] == "−/+", out
    assert out.loc["M2", "signs_a"] == "+/+" and out.loc["M3", "signs_a"] == "+", out
    ranks = out["abs_d_prime_a"].rank(ascending=False, method="min")
    assert ranks.to_dict() == {"M1": 1.0, "M2": 3.0, "M3": 2.0}, ranks  # ranked by |d′|
    return out.reset_index()


display(check_evoked_by_mouse_on_synthetic_data())


def mouse_summary(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Per mouse: median SNR measures over its recordings, recording count, flag counts and both ranks."""
    snr, evoked = tables["snr"], tables["evoked"]
    rows = []
    for mouse, group in snr.groupby("mouse", sort=False):
        flags = pd.Series(
            split_flags(pd.concat([t.loc[t["mouse"] == mouse, "flags"] for t in tables.values()]))
        )
        rows.append(
            {
                "mouse": mouse,
                "n_recordings": len(group),
                "snr_db": group["snr_db"].median(),
                "specificity_db": group["specificity_db"].median(),
                "flags": ", ".join(f"{k} x{v}" for k, v in flags.value_counts().items()),
            }
        )
    out = pd.DataFrame(rows).merge(evoked_by_mouse(evoked), on="mouse")
    out["rank_snr"] = out["snr_db"].rank(ascending=False, method="min").astype(int)
    out["rank_evoked"] = (
        out["abs_d_prime_a"].rank(ascending=False, method="min", na_option="bottom").astype(int)
    )
    return out


MOUSE_SUMMARY = mouse_summary(
    {
        "acquisition": qc_acquisition,
        "denoising": qc_denoising,
        "bleaching": qc_bleaching,
        "movement": qc_movement,
        "snr": qc_snr,
        "evoked": qc_evoked,
    }
)
_rewarded_peaks = control_channel.query("condition == 'rewarded'").set_index("mouse")[
    ["peak_signal_pct", "peak_control_pct", "ratio_signal_over_control"]
]
MOUSE_SUMMARY = MOUSE_SUMMARY.merge(
    _rewarded_peaks.add_suffix("_rewarded"), left_on="mouse", right_index=True, how="left"
)
save_table(MOUSE_SUMMARY, "qc_mouse_summary")
RANKING_COLUMNS = [
    "mouse",
    "n_recordings",
    "snr_db",
    "specificity_db",
    "abs_d_prime_a",
    "signs_a",
    "flags",
]

print(
    "Mice ranked by primary SNR (spectral, event-independent); median over each mouse's recordings:\n"
)
by_snr = MOUSE_SUMMARY.sort_values("rank_snr")[["rank_snr", *RANKING_COLUMNS]].rename(
    columns={"rank_snr": "rank"}
)
print(by_snr.to_string(index=False))

print(
    "\nPeak |response| 0-2 s of the uncorrected signal and of the control (session ΔF/F %), rewarded trials:\n"
)
print(
    MOUSE_SUMMARY[
        [
            "mouse",
            "peak_signal_pct_rewarded",
            "peak_control_pct_rewarded",
            "ratio_signal_over_control_rewarded",
        ]
    ].to_string(index=False)
)
print(
    "\nMice ranked by evoked d′ (median over recordings of |d′| of the rewarded response, pipeline A):\n"
)
by_evoked = MOUSE_SUMMARY.sort_values("rank_evoked")[
    [
        "rank_evoked",
        "mouse",
        "n_recordings",
        "abs_d_prime_a",
        "signs_a",
        "abs_d_prime_b",
        "signs_b",
        "snr_db",
    ]
]
print(
    "(Ranked by |d′|, which does not depend on the trial count. signs_a: sign of each recording's d′, '−/+' = one dip and one rise.)"
)
print(by_evoked.rename(columns={"rank_evoked": "rank"}).to_string(index=False))

gap = (MOUSE_SUMMARY["rank_snr"] - MOUSE_SUMMARY["rank_evoked"]).abs()
disagree = MOUSE_SUMMARY.loc[gap >= MIN_RANK_DISAGREEMENT, ["mouse", "rank_snr", "rank_evoked"]]
print(f"\nMice whose two ranks differ by {MIN_RANK_DISAGREEMENT} or more:")
print(disagree.to_string(index=False) if len(disagree) else "  none")
print(
    f"Spearman correlation between the two rankings: {MOUSE_SUMMARY['rank_snr'].corr(MOUSE_SUMMARY['rank_evoked'], method='spearman'):.2f}"
)
mixed = MOUSE_SUMMARY.loc[
    MOUSE_SUMMARY["signs_a"].str.contains("−") & MOUSE_SUMMARY["signs_a"].str.contains(r"\+"),
    ["mouse", "signs_a"],
]
print(
    "Mice whose recordings disagree in the sign of the evoked response:",
    mixed.to_string(index=False) if len(mixed) else "none",
)

In [ ]:
def plot_mouse_ranking(axes, summary: pd.DataFrame) -> None:
    """Horizontal bars of primary SNR, specificity and evoked |d′| per mouse, best primary SNR at the top.

    The evoked panel writes each mouse's per-recording signs beside its bar.
    """
    ordered = summary.sort_values("rank_snr", ascending=False)
    for ax, column, label in zip(
        axes,
        ("snr_db", "specificity_db", "abs_d_prime_a"),
        (
            "Spectral SNR (dB)",
            "Specificity: Uncorrected Sensor Minus Control (dB)",
            "Evoked |d′| (Median Over Recordings; Signs Shown)",
        ),
        strict=True,
    ):
        ax.barh(ordered["mouse"], ordered[column], color="tab:blue")
        ax.axvline(0, color="k", lw=0.6)
        ax.set(xlabel=label, title=label.split(" (")[0])
    for y, (value, signs) in enumerate(
        zip(ordered["abs_d_prime_a"], ordered["signs_a"], strict=True)
    ):
        axes[2].annotate(
            signs, xy=(value, y), xytext=(3, 0), textcoords="offset points", va="center", fontsize=8
        )
    axes[2].margins(x=0.2)
    axes[0].set_ylabel("Mouse (Best Primary SNR at Top)")


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True, constrained_layout=True)
plot_mouse_ranking(axes, MOUSE_SUMMARY)
fig.suptitle("Per-Mouse SNR (Median Over Recordings)", fontsize=12)
save_figure(fig, "qc_mouse_ranking")
plt.show()

## 9. Summary

**What was run.** Cohort-04 recordings were put on pyControl time (photometry and video, section 2), preprocessed by pipeline A (primer
order) and pipeline B (the nLightG2 paper and the authors' MATLAB code, section 4), checked with the QC suite (sections 6 and 8), aligned
to rewarded and unrewarded outcomes for the signal and the 405-nm control (section 7), and compared across pre-outcome windows,
smoothing lengths and two groups of mice (7c, 7d). The counts and locations are printed below.

**Observations from this run** (see the tables and figures for the numbers):

- Sync is good for all recordings (matched fraction near 1, interval disagreement within one photometry sample); video lies on the same clock.
- The detrended signal and control are strongly correlated in most recordings (R² in `qc_4_movement`), so most of the signal's
  variance is shared with the control and is removed by step A5.
- Uncorrected specificity (uncorrected sensor SNR minus control SNR, `specificity_db` in `qc_5_snr`) has per-mouse medians from **−4.7 to +3.6 dB**:
  within ±1 dB for 5 of 10 mice (RT_64_N −0.25, RT_177_N +0.68, RT_176_N −0.90, RT_177_LR −0.01, RT_64_R −0.07), **below −1 dB for 4** (RT_177_R −3.18,
  RT_177_L −2.02, RT_176_R −4.65, RT_65_LR −1.93: the control carries *more* slow power than the sensor) and above +1 dB for only 1 (RT_176_L +3.59).
  13 of 17 recordings fall below the 1-dB flag. So slow fluctuations are about as strong in the control as in the signal channel, or stronger.
- The control-channel check (section 7b, `qc_7_control_channel_peaks`) shows the outcome-locked response in the *control* too, with the **same sign** as in
  the uncorrected signal and a size comparable to the signal's (signal/control peak ratio 0.4-2, mostly 0.7-1). The regression removes most of it, leaving responses of
  order 0.05-0.1 % ΔF/F. A same-sign response in both channels fits movement/haemodynamic artefact, or a control that responds to the sensor's target;
  an opposite sign would have indicated a non-isosbestic control (not seen).
- Units were verified on the example recording (SD of the corrected signal 6 mV on a 2.6 V baseline = 0.23 %); the movement regression removes 15-94 % of the SD of the
  session ΔF/F (`qc_correction_size`), so the small corrected ΔF/F reflects how much of the signal is shared with the control.
- The LED-off baseline contains brief upward spikes in some sessions (`*_baseline_spikes_per_min` in `qc_1_acquisition`); in the example
  recording they are not at the rsync pulses. Their origin is not known.
- **Pipeline B as the paper and code define it (section 4).** The `exp2` fit (no constant, MATLAB sample 100 (Python index 99) to end − 30 s) was replaced by the bounded fit for 1 of 34 channel fits: the 405-nm control of RT_64_N, session 45. That rejection comes from **this notebook's own
  amplitude-ratio rule** (the two exponentials cancel each other), **which MATLAB's `fit` does not have**, so the authors' code would have used the exp2 curve. Inside the fit window the
  rejected exp2 curve has RMSE 24.3 mV against 26.7 mV for the bounded fit that replaced it, i.e. the rejected curve was not the worse fit there (`pipeline_b_fit_models`, printed below the table).
  The rule is marginal for this recording: its signal channel passes it, and the outcome depends on details of the fit such as the first sample used. A post-outcome mask of the motion fit would remove a median
  of 16 % (2 s), 38 % (5 s), 63 % (10 s) and 74 % (20 s, the code's value) of the fit window (`pipeline_b_motion_mask`), and the motion slope changes by up to 0.18 (RT_177_R 0.56 → 0.38 at 20 s), so it is off by default.
  Pipeline B's corrected signal agrees with A's only partly over the whole trace (`pipeline_comparison_traces`: within the fit window r = 0.37-1.00, median 0.86; 4 of 17 recordings below 0.6): the two bleaching fits differ by a median of 0.16 % or less of the fitted level
  in 15 of 17 recordings (0.22 % in RT_64_N session 45, 0.95 % in RT_64_R session 44), but the corrected signal is only 0.05-0.3 % of that level, so slow components differ. The unconstrained `exp2` curve also bends away from the data in the 30 s that are left out of the fit (the example figure, RT_177_N session 43). At the outcomes the group-mean ΔF/F responses of the two pipelines agree to two decimals (rewarded Group 1 −0.020 % and −0.029 %); the z responses differ more
  (rewarded Group 1 +0.14 in A, −0.34 in B), because B's z is relative to the pre-event window and its 0-2 s extreme is a dip (table `group_comparison_metrics`, 7d).
- **Pre-outcome window and smoothing (7c).** Rewarded trials kept: 1362, 1362, 1361 and 1359 of 1379 for 2, 5, 10 and 15 s windows (unrewarded 509, 509, 509 and 508 of 515). The windows are not the main cost: **254, 983 and 1157 of the kept rewarded trials (19 %, 72 %, 85 %)** have another outcome
  inside the 5-, 10- and 15-s windows (none in 2 s). On pipeline B's trial z, the group-mean rewarded response (0-2 s against −2-0 s, `sensitivity_peak_responses`) is −0.23 (2 s window, no smoothing), −0.24 (5 s), −0.21 (10 s), −0.20 (15 s): the window changes it by under 20 %.
  Smoothing changes it more: with the 5-s window −0.24, −0.23, −0.18, −0.15 for 1, 10, 40, 80 points (0.08, 0.31, 0.62 s at 130 Hz); the unrewarded response stays at +0.14 to +0.17. Pipeline A's session z: +0.16 → +0.09 from no smoothing to 80 points.
- **Groups (7d).** Group 1 (RT_176_*, RT_64_*; 5 mice) and Group 2 (RT_177_*, RT_65_*; 5 mice): the signal-channel mean ± SEM across mice overlap for most of the window in every panel. Rewarded z, 0-2 s response: pipeline A +0.14 (Group 1) and +0.08 (Group 2); pipeline B −0.34 and +0.11 (Group 1's extreme in that window is the dip after a pre-zero peak).
  Unrewarded z: +0.10 / +0.10 (A) and +0.14 / +0.17 (B). In the 405-nm control the outcome response is larger than the signal's: rewarded ΔF/F −0.35 % (Group 1) and +0.25 % (Group 2) against −0.02 % and +0.02 % (A) in the signal; rewarded z −0.27 and −0.25 (A), −0.71 and −0.64 (B).
- **RT_176_L against Group 2.** Rewarded z, 0-2 s response: RT_176_L +0.66 (A) and +0.84 (B), against +0.08 and +0.11 for Group 2 and +0.33 and +0.40 for RT_176_L's own control; unrewarded +0.31 and +0.39 against +0.13 and +0.22 in its control. RT_176_L's response is about twice its control's
  (section 7b ratio 2.0), and its ΔF/F is +0.035 % (Group 2: +0.019 %). It rests on 75 rewarded and 25 unrewarded trials of one session.
- **Evoked ranking.** Mice are ranked by the median over their recordings of |d′| (mean rewarded response / SD across trials; it does not grow with the trial count),
  with each recording's sign shown. Three mice have recordings that disagree in sign (RT_64_N "+/−/−", RT_64_R "−/+", RT_177_N "+/+/−"); the median of absolute values keeps them from cancelling.
  The two rankings (primary spectral SNR, median |d′|) are only weakly related in this cohort (Spearman 0.09); they measure different things, and
  with 1-4 recordings per mouse neither ranking is stable.

**Open questions.**

1. **Channel wiring.** `analog_1` = nLightG2 (470 nm) and `analog_2` = isosbestic control (405 nm) is assumed from the earlier notebooks and has not been
   confirmed against the physical LED connections.
2. **Remaining uncertainty in pipeline B.** The sampling rate behind the paper's 80 points is not stated (section 4); the code's low-pass is 40 Hz and causal, the paper's 10 Hz (this notebook follows the paper, zero-phase);
   the 20-s post-outcome mask and the 15-s baseline do not fit a task with outcomes every 10 s, and the choice of defaults here (mask off, 5-s baseline) is ours.
3. **The 6-ms cue offset.** The no-reward cue timer fires 494 ms after cue onset (logged 500 ms), below one 7.7-ms photometry sample. Should
   unrewarded events be shifted? (Its cause is not established.)
4. **Sharp peak or dip at t = 0 in both outcomes.** Several mice show a sharp deflection at the outcome for rewarded *and* unrewarded trials,
   present in the control channel as well (sections 7 and 7b), and larger there than in the corrected signal. Is it physiological, or task-locked (light, sound, pump, a reflection when the animal moves at the port)?
5. **Uncorrected specificity at or below 0 dB.** For 9 of 10 mice slow fluctuations are as strong in the control as in the signal, or stronger (median −4.7 to +0.7 dB). Is 405 nm isosbestic
   for nLightG2 (not verified; please check), and is the signal channel dominated by movement/haemodynamics in these recordings?
6. **Small ΔF/F.** Corrected responses are about 0.05-0.1 % ΔF/F, small for an NE sensor. The arithmetic is verified (section 3); the open question
   is whether sensor expression/fibre placement, the large autofluorescence-containing denominator (about 2.6 V), or the regression removing real signal explains it.
7. **Thresholds.** The QC thresholds are starting points (`low_specificity` below 1 dB; `few_trials` below 5 rewarded trials).
8. **Recordings of concern.** The 1.9-minute recording of RT_177_N (ses-40, one trial) is flagged `short` and `outside_sync` and contributes almost
   nothing; several mice have only one session, so their medians rest on one recording.

In [ ]:
n_used = exclusions.groupby("condition")["n_used"].sum()
print(f"Mice: {len(MICE)}; recordings: {len(RECORDINGS)}; skipped .ppd files: {len(skipped)}.")
print(f"Trials used: {int(n_used.sum())} ({n_used.to_dict()}); phases: {PHASES}.")
print(
    f"Recordings flagged low_specificity (< {QC_THRESHOLDS['min_specificity_db']} dB): "
    f"{int((qc_snr['specificity_db'] < QC_THRESHOLDS['min_specificity_db']).sum())} of {len(qc_snr)}."
)
print(f"Figures: {FIGURE_DIR.relative_to(REPO_ROOT)}")
print(f"Tables:  {TABLE_DIR.relative_to(REPO_ROOT)}")
print(f"Cache:   {memory.location}")